# CBOT Corn Futures Strategy Notebook

This notebook is consolidated from several research notebooks. It takes raw local input files, cleans them into a daily corn state panel, builds forecast modules, reconciles the balance sheet, constructs trading factors, and exports a strategy ready dataset.

The notebook assumes that official source files are stored inside the project folder. It does not fetch data from external services.


## Setup

The notebook uses only relative folders. Put raw source files under `data/raw`, keep processed tables under `data/processed`, and write final outputs under `outputs`.


In [ ]:
from pathlib import Path
import glob
import json
import re

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from IPython.display import display
from pandas.core.dtypes.common import is_numeric_dtype

pd.set_option("display.max_columns", 160)
pd.set_option("display.width", 220)
pd.set_option("display.max_colwidth", 140)

PROJECT_ROOT = Path.cwd()
RAW_DATA_DIR = PROJECT_ROOT / "data" / "raw"
PROCESSED_DATA_DIR = PROJECT_ROOT / "data" / "processed"
OUTPUT_DATA_DIR = PROJECT_ROOT / "outputs"

for directory in [RAW_DATA_DIR, PROCESSED_DATA_DIR, OUTPUT_DATA_DIR]:
    directory.mkdir(parents=True, exist_ok=True)

state_forecast_output_directory = OUTPUT_DATA_DIR


## Raw local input inventory

This section scans the local raw data folder and records the files that will be normalized. The folder names are only conventions, so the code also searches nested folders.


In [ ]:
RAW_INPUT_PATTERNS = {
    "nass": ["nass/*.csv", "nass/*.xlsx", "nass/*.xls", "nass/*.parquet"],
    "wasde": ["wasde/*.csv", "wasde/*.xlsx", "wasde/*.xls", "wasde/*.parquet"],
    "exports": ["exports/*.csv", "export_sales/*.csv", "exports/*.xlsx", "export_sales/*.xlsx"],
    "ethanol": ["ethanol/*.csv", "ethanol/*.xlsx", "eia/*.csv", "eia/*.xlsx"],
    "positioning": ["cftc/*.csv", "cftc/*.xlsx", "positioning/*.csv", "positioning/*.xlsx"],
    "weather": ["weather/*.csv", "weather/*.xlsx", "noaa/*.csv", "noaa/*.xlsx"],
    "market": ["cme/*.csv", "cme/*.xlsx", "market/*.csv", "market/*.xlsx", "prices/*.csv", "prices/*.xlsx"],
    "other": ["*.csv", "*.xlsx", "*.xls", "*.parquet"],
}

raw_input_rows = []
for source_group, patterns in RAW_INPUT_PATTERNS.items():
    for pattern in patterns:
        for file_path in sorted(RAW_DATA_DIR.glob(pattern)):
            raw_input_rows.append({"source_group": source_group, "path": file_path})

raw_input_inventory = (
    pd.DataFrame(raw_input_rows)
    .drop_duplicates(subset=["path"])
    .sort_values(["source_group", "path"])
    .reset_index(drop=True)
)

raw_file_paths = raw_input_inventory["path"].tolist()
display(raw_input_inventory)


## Canonical helpers for raw files

These helpers convert mixed official drops into a common long format with one value per source, metric, geography, and date. The goal is to preserve raw information while making later daily modeling reproducible.


In [ ]:
CANONICAL_SOURCE_COLUMNS = [
    "source_group",
    "source_file",
    "series_name",
    "metric",
    "value",
    "unit",
    "frequency",
    "geography",
    "observation_date",
    "report_date",
    "asof_date",
    "source_row_id",
    "extra_json",
]

DATE_CANDIDATES = [
    "asof_date",
    "date",
    "observation_date",
    "report_date",
    "release_date",
    "week_ending",
    "load_time",
    "timestamp",
    "trade_date",
    "business_date",
    "settle_date",
]

GEOGRAPHY_CANDIDATES = ["state_alpha", "state", "state_name", "country", "country_name", "geography", "market"]
UNIT_CANDIDATES = ["unit", "unit_desc", "units", "measurement_unit"]
FREQUENCY_CANDIDATES = ["freq_desc", "frequency", "period", "week"]


def clean_column_name(name):
    cleaned = re.sub(r"[^0-9a-zA-Z]+", "_", str(name).strip().lower())
    return cleaned.strip("_")


def normalize_number(value):
    if pd.isna(value):
        return np.nan
    text = str(value).strip()
    missing_values = {"", "na", "n_a", "none", "nan", "d", "na_", "_d_", "_na_"}
    normalized_text = text.lower().replace("(", "_").replace(")", "_").strip("_")
    if normalized_text in missing_values:
        return np.nan
    if normalized_text == "z":
        return 0.0
    numeric_text = text.replace(",", "").replace("$", "").replace("%", "")
    return pd.to_numeric(numeric_text, errors="coerce")


def find_first_column(columns, candidates):
    available = set(columns)
    matches = [column for column in candidates if column in available]
    return matches[0] if matches else None


def load_local_table(file_path):
    suffix = file_path.suffix.lower()
    if suffix == ".csv":
        return pd.read_csv(file_path, low_memory=False)
    if suffix in {".xlsx", ".xls"}:
        return pd.read_excel(file_path)
    if suffix == ".parquet":
        return pd.read_parquet(file_path)
    raise ValueError(f"Unsupported input file type: {file_path}")


def source_group_for_path(file_path):
    path_text = str(file_path.relative_to(RAW_DATA_DIR)).lower()
    matched_groups = [group for group in RAW_INPUT_PATTERNS if group in path_text]
    return matched_groups[0] if matched_groups else "other"


## Normalize raw files to long series

Each raw table is cleaned, numeric fields are melted into long form, and metadata columns are kept inside `extra_json`. This keeps the raw evidence available without forcing every source into a custom schema.


In [ ]:
def local_table_to_canonical(file_path):
    raw_table = load_local_table(file_path)
    raw_table = raw_table.rename(columns={column: clean_column_name(column) for column in raw_table.columns})

    date_column = find_first_column(raw_table.columns, DATE_CANDIDATES)
    geography_column = find_first_column(raw_table.columns, GEOGRAPHY_CANDIDATES)
    unit_column = find_first_column(raw_table.columns, UNIT_CANDIDATES)
    frequency_column = find_first_column(raw_table.columns, FREQUENCY_CANDIDATES)

    working_table = raw_table.copy()
    working_table["source_row_id"] = np.arange(len(working_table))
    working_table["asof_date"] = pd.to_datetime(working_table[date_column], errors="coerce")

    numeric_columns = []
    for column in working_table.columns:
        if column in {"source_row_id", "asof_date"}:
            continue
        if is_numeric_dtype(working_table[column]):
            numeric_columns.append(column)
        else:
            converted = working_table[column].map(normalize_number)
            if converted.notna().sum() > 0:
                working_table[column] = converted
                numeric_columns.append(column)

    id_columns = [column for column in working_table.columns if column not in numeric_columns]
    long_table = working_table.melt(
        id_vars=id_columns,
        value_vars=numeric_columns,
        var_name="metric",
        value_name="value",
    )

    long_table = long_table.dropna(subset=["asof_date", "value"]).copy()
    long_table["source_group"] = source_group_for_path(file_path)
    long_table["source_file"] = file_path.name
    long_table["series_name"] = long_table["source_group"] + "_" + long_table["metric"].astype(str)
    long_table["unit"] = long_table[unit_column].astype(str) if unit_column else "unknown"
    long_table["frequency"] = long_table[frequency_column].astype(str) if frequency_column else "unknown"
    long_table["geography"] = long_table[geography_column].astype(str) if geography_column else "all"
    long_table["observation_date"] = long_table["asof_date"]
    long_table["report_date"] = long_table["asof_date"]

    metadata_columns = [column for column in id_columns if column not in {"asof_date"}]
    long_table["extra_json"] = [
        json.dumps(record, default=str)
        for record in long_table[metadata_columns].to_dict(orient="records")
    ]

    return long_table[CANONICAL_SOURCE_COLUMNS]

canonical_raw_series = pd.concat(
    [local_table_to_canonical(file_path) for file_path in raw_file_paths],
    ignore_index=True,
)

canonical_raw_series = canonical_raw_series.sort_values(["asof_date", "source_group", "series_name"]).reset_index(drop=True)
canonical_raw_series.to_csv(PROCESSED_DATA_DIR / "corn_canonical_raw_series.csv", index=False)

display(canonical_raw_series.head(20))


## Build the daily input panel

The long raw series are pivoted into one daily table. Repeated source updates on the same date are resolved by taking the latest available value after sorting by file and row order.


In [ ]:
canonical_raw_series["series_key"] = (
    canonical_raw_series["source_group"].astype(str)
    + "__"
    + canonical_raw_series["geography"].astype(str).str.lower().str.replace(r"[^0-9a-z]+", "_", regex=True)
    + "__"
    + canonical_raw_series["metric"].astype(str)
)

canonical_raw_series = canonical_raw_series.sort_values(["asof_date", "source_file", "source_row_id"])

daily_raw_panel = (
    canonical_raw_series
    .pivot_table(index="asof_date", columns="series_key", values="value", aggfunc="last")
    .reset_index()
    .sort_values("asof_date")
    .reset_index(drop=True)
)

daily_raw_panel.columns.name = None
daily_raw_panel.to_csv(PROCESSED_DATA_DIR / "corn_daily_input_panel.csv", index=False)

print(f"Daily raw panel rows: {daily_raw_panel.shape[0]:,}")
print(f"Daily raw panel columns: {daily_raw_panel.shape[1]:,}")
display(daily_raw_panel.head())


## Repair the daily panel

This section removes invalid dates, collapses duplicated daily rows, coerces numeric fields, and prepares the table used by the forecasting and factor sections.


In [ ]:
def last_observed_value(series):
    observed = series.dropna()
    return observed.iloc[-1] if len(observed) else np.nan

source_panel = daily_raw_panel.copy()
source_panel["asof_date"] = pd.to_datetime(source_panel["asof_date"], errors="coerce")
source_panel = source_panel.dropna(subset=["asof_date"]).copy()
source_panel = source_panel[source_panel["asof_date"] >= pd.Timestamp("1971-01-01")].copy()
source_panel = source_panel.sort_values("asof_date")

value_columns = [column for column in source_panel.columns if column != "asof_date"]
raw_corn_panel = (
    source_panel
    .groupby("asof_date", as_index=False)[value_columns]
    .agg(last_observed_value)
    .sort_values("asof_date")
    .reset_index(drop=True)
)

for column in value_columns:
    raw_corn_panel[column] = pd.to_numeric(raw_corn_panel[column], errors="ignore")

print(f"Clean daily panel rows: {raw_corn_panel.shape[0]:,}")
print(f"Clean daily panel columns: {raw_corn_panel.shape[1]:,}")


## Market price repair

The market price column is selected from common settlement and close fields. Prices are standardized to dollars per bushel, then short horizon return and volatility fields are created.


In [ ]:
def series_has_signal(series, minimum_unique=5):
    numeric_series = pd.to_numeric(series, errors="coerce")
    return numeric_series.notna().sum() >= minimum_unique and numeric_series.nunique(dropna=True) >= minimum_unique


def select_price_column(data):
    exact_candidates = [
        "front_settle",
        "settle",
        "settlement",
        "adj_close",
        "close",
        "last",
        "price",
        "futures_price",
    ]
    exact_matches = [column for column in exact_candidates if column in data.columns]
    pattern_matches = [
        column for column in data.columns
        if any(token in column.lower() for token in ["settle", "close", "price", "last"])
    ]
    ordered_candidates = list(dict.fromkeys(exact_matches + pattern_matches))
    signal_candidates = [column for column in ordered_candidates if series_has_signal(data[column])]
    return signal_candidates[0]

price_column = select_price_column(raw_corn_panel)
raw_corn_panel["front_price_raw"] = pd.to_numeric(raw_corn_panel[price_column], errors="coerce")
price_median = raw_corn_panel["front_price_raw"].dropna().median()
price_scale = 100.0 if price_median > 50 else 1.0

raw_corn_panel["front_settle"] = raw_corn_panel["front_price_raw"] / price_scale
raw_corn_panel["front_return_1d"] = raw_corn_panel["front_settle"].pct_change(1)
raw_corn_panel["front_return_5d"] = raw_corn_panel["front_settle"].pct_change(5)
raw_corn_panel["front_volatility_20d"] = raw_corn_panel["front_return_1d"].rolling(20, min_periods=10).std() * np.sqrt(252)

print("Selected price column:", price_column)
display(raw_corn_panel[["asof_date", "front_settle", "front_return_1d", "front_volatility_20d"]].head())


## Clean panel quality audit

The audit confirms that the final daily input has one row per date, no invalid timestamps, and a monotonic daily spine before modeling begins.


In [ ]:
clean_panel_audit = pd.DataFrame({
    "check": [
        "date column has no missing values",
        "dates are monotonic increasing",
        "dates are unique",
        "all dates are after 1971",
        "front settlement has observations",
    ],
    "passed": [
        raw_corn_panel["asof_date"].notna().all(),
        raw_corn_panel["asof_date"].is_monotonic_increasing,
        raw_corn_panel["asof_date"].is_unique,
        raw_corn_panel["asof_date"].ge(pd.Timestamp("1971-01-01")).all(),
        raw_corn_panel["front_settle"].notna().sum() > 0,
    ],
})

clean_panel_audit["passed"] = clean_panel_audit["passed"].astype(bool)
display(clean_panel_audit)

raw_corn_panel.to_csv(PROCESSED_DATA_DIR / "corn_clean_daily_panel.csv", index=False)


### Preview the raw panel

Show the first and last rows to confirm the daily panel loaded correctly.


In [ ]:
display(raw_corn_panel.head())
display(raw_corn_panel.tail())


### Summarize raw panel coverage

Report the row count, date range, duplicated rows, and duplicated columns.


In [ ]:
date_min = raw_corn_panel["asof_date"].min()
date_max = raw_corn_panel["asof_date"].max()

summary = pd.DataFrame({
    "item": [
        "rows",
        "columns",
        "start_date",
        "end_date",
        "duplicate_rows",
        "duplicate_columns",
    ],
    "value": [
        f"{raw_corn_panel.shape[0]:,}",
        f"{raw_corn_panel.shape[1]:,}",
        date_min,
        date_max,
        f"{raw_corn_panel.duplicated().sum():,}",
        raw_corn_panel.columns[raw_corn_panel.columns.duplicated()].tolist(),
    ],
})

display(summary)


### Plot raw panel timeline

Visualize the monthly row count to check the daily time spine.


In [ ]:
timeline = (
    raw_corn_panel
    .assign(asof_date=pd.to_datetime(raw_corn_panel["asof_date"], errors="coerce"))
    .dropna(subset=["asof_date"])
    .set_index("asof_date")
    .resample("M")
    .size()
)

ax = timeline.plot(figsize=(11, 4))
ax.set_title("Rows by month")
ax.set_xlabel("")
ax.set_ylabel("Rows")
ax.grid(alpha=0.25)
plt.tight_layout()
plt.show()


## Source coverage and schema audit

Audit source columns, missingness, data types, and module level coverage.


### Build the schema audit

audits data types, missing values, and column level coverage.


In [ ]:
schema = pd.DataFrame({
    "column": raw_corn_panel.columns,
    "dtype": [str(raw_corn_panel[col].dtype) for col in raw_corn_panel.columns],
    "non_null": [raw_corn_panel[col].notna().sum() for col in raw_corn_panel.columns],
    "missing": [raw_corn_panel[col].isna().sum() for col in raw_corn_panel.columns],
    "missing_pct": [raw_corn_panel[col].isna().mean() * 100 for col in raw_corn_panel.columns],
    "unique_values": [raw_corn_panel[col].nunique(dropna=True) for col in raw_corn_panel.columns],
})

schema = schema.sort_values(["missing_pct", "column"], ascending=[False, True])

display(schema)


### Build the schema audit

audits data types, missing values, and column level coverage.


In [ ]:
plot_missingness_table(
    schema,
    column_col="column",
    missing_col="missing_pct",
    top_n=30,
    title="Highest missingness columns",
)


### Review output

Preview the current table before continuing.


In [ ]:
date_tokens = ["date", "time", "week", "month", "year", "report", "vintage"]

date_cols = [
    col for col in raw_corn_panel.columns
    if any(token in col.lower() for token in date_tokens)
]

display(date_cols)


### Map source columns to modules

groups source columns by the forecast module that will consume them.


In [ ]:
module_keywords = {
    "acreage": ["acre", "planted", "harvested"],
    "yield": ["yield", "condition", "gdd", "drought", "heat", "weather"],
    "production": ["production", "prod"],
    "exports": ["export", "sales", "inspection", "shipment"],
    "ethanol": ["ethanol", "eia"],
    "feed_residual": ["feed", "residual"],
    "stocks": ["stock", "ending", "beginning", "carryout", "stocks_to_use", "stu"],
    "futures_spreads": ["future", "settle", "price", "spread", "basis", "curve", "nearby"],
    "positioning": ["cftc", "managed", "money", "position", "cot"],
    "regime": ["regime", "season", ""],
}

module_cols = {}

for module, keywords in module_keywords.items():
    module_cols[module] = [
        col for col in raw_corn_panel.columns
        if any(keyword in col.lower() for keyword in keywords)
    ]

module_summary = pd.DataFrame({
    "module": module_cols.keys(),
    "column_count": [len(columns) for columns in module_cols.values()],
    "columns": module_cols.values(),
})

display(module_summary)


### Map source columns to modules

groups source columns by the forecast module that will consume them.


In [ ]:
module_coverage = pd.DataFrame([
    {
        "module": module_name,
        "available_columns": len([column for column in columns if column in raw_corn_panel.columns]),
        "requested_columns": len(columns),
    }
    for module_name, columns in module_cols.items()
])

module_coverage["coverage_percent"] = (
    module_coverage["available_columns"]
    / module_coverage["requested_columns"]
    * 100
)

ax = module_coverage.plot(
    kind="bar",
    x="module",
    y="coverage_percent",
    legend=False,
    figsize=(10, 4),
)
ax.set_title("Forecast module source column coverage")
ax.set_xlabel("")
ax.set_ylabel("Coverage percent")
ax.grid(axis="y", alpha=0.25)
plt.xticks(rotation=45, ha="right")
plt.tight_layout()
plt.show()

display(module_coverage)


### Review output

Preview the current table before continuing.


In [ ]:
numeric_cols = raw_corn_panel.select_dtypes(include=[np.number]).columns.tolist()

print(f"Numeric columns: {len(numeric_cols):,}")

display(raw_corn_panel[numeric_cols].describe().T)


### Summarize numeric fields

summarizes numeric columns before model features are built.


In [ ]:
numeric_columns_to_plot = [
    column
    for column in numeric_summary["column"].head(8).tolist()
    if column in raw_corn_panel.columns
]

plot_numeric_histograms(
    raw_corn_panel,
    numeric_columns_to_plot,
    title_prefix="Raw numeric distribution",
)


### Summarize categorical fields

summarizes categorical columns and their dominant values.


In [ ]:
categorical_cols = raw_corn_panel.select_dtypes(include=["object", "category"]).columns.tolist()

categorical_summary = pd.DataFrame({
    "column": categorical_cols,
    "unique_values": [raw_corn_panel[col].nunique(dropna=True) for col in categorical_cols],
    "sample_values": [
        raw_corn_panel[col].dropna().astype(str).unique()[:5].tolist()
        for col in categorical_cols
    ],
})

print(f"Categorical columns: {len(categorical_cols):,}")

display(categorical_summary)


## Modeling panel and physical variables

Create the working panel and derive the physical corn balance sheet variables.


### Clean physical state variables

prepares clean physical variables for the trading model.


In [ ]:
model_source_panel = raw_corn_panel.copy()

date_cols = [
    col for col in model_source_panel.columns
    if col in [
        "asof_date",
        "asof_timestamp",
        "max_feature_knowledge_timestamp",
        "max_all_knowledge_timestamp_state_forecast",
        "cftc_report_date",
        "cftc_observation_date",
        "cftc_clean_release_timestamp",
    ]
    or col.endswith("__knowledge_timestamp")
]

for col in date_cols:
    model_source_panel[col] = pd.to_datetime(model_source_panel[col], errors="coerce")

model_source_panel = model_source_panel.sort_values("asof_date").reset_index(drop=True)

print(f"Parsed date columns: {len(date_cols)}")
print(f"Rows: {model_source_panel.shape[0]:,}")
print(f"Columns: {model_source_panel.shape[1]:,}")


### Review output

Preview the current table before continuing.


In [ ]:
input_audit = pd.DataFrame({
    "item": [
        "rows",
        "columns",
        "start_date",
        "end_date",
        "first_marketing_year",
        "last_marketing_year",
        "duplicate_asof_dates",
    ],
    "value": [
        f"{model_source_panel.shape[0]:,}",
        f"{model_source_panel.shape[1]:,}",
        model_source_panel["asof_date"].min().date(),
        model_source_panel["asof_date"].max().date(),
        model_source_panel["marketing_year"].min(),
        model_source_panel["marketing_year"].max(),
        model_source_panel["asof_date"].duplicated().sum(),
    ],
})

display(input_audit)


### Review output

Preview the current table before continuing.


In [ ]:
checks = {}

if "state_forecast_state_estimation_ready" in model_source_panel.columns:
    checks["state_forecast_state_estimation_ready_share"] = model_source_panel["state_forecast_state_estimation_ready"].mean()

if "strategy_input_spread_pricing_ready" in model_source_panel.columns:
    checks["strategy_input_spread_pricing_ready_share"] = model_source_panel["strategy_input_spread_pricing_ready"].mean()

if "knowledge_leakage_flag" in model_source_panel.columns:
    checks["knowledge_leakage_flag_count"] = model_source_panel["knowledge_leakage_flag"].sum()

if "state_forecast_knowledge_leakage_flag" in model_source_panel.columns:
    checks["state_forecast_knowledge_leakage_flag_count"] = model_source_panel["state_forecast_knowledge_leakage_flag"].sum()

if {"max_all_knowledge_timestamp_state_forecast", "asof_timestamp"}.issubset(model_source_panel.columns):
    checks["state_forecast_knowledge_after_asof_count"] = (
        model_source_panel["max_all_knowledge_timestamp_state_forecast"] > model_source_panel["asof_timestamp"]
    ).sum()

if {"max_feature_knowledge_timestamp", "asof_timestamp"}.issubset(model_source_panel.columns):
    checks["feature_knowledge_after_asof_count"] = (
        model_source_panel["max_feature_knowledge_timestamp"] > model_source_panel["asof_timestamp"]
    ).sum()

leakage_check = pd.DataFrame(checks.items(), columns=["check", "value"])

display(leakage_check)


### Review output

Preview the current table before continuing.


In [ ]:
coverage = pd.DataFrame({
    "column": model_source_panel.columns,
    "dtype": [str(model_source_panel[col].dtype) for col in model_source_panel.columns],
    "non_null": [model_source_panel[col].notna().sum() for col in model_source_panel.columns],
    "coverage_pct": [model_source_panel[col].notna().mean() * 100 for col in model_source_panel.columns],
    "unique_values": [model_source_panel[col].nunique(dropna=True) for col in model_source_panel.columns],
})

coverage = coverage.sort_values(["coverage_pct", "column"], ascending=[True, True])

display(coverage)


### Review output

Preview the current table before continuing.


In [ ]:
empty_cols = coverage.loc[coverage["coverage_pct"] == 0, "column"].tolist()

sparse_cols = coverage.loc[
    (coverage["coverage_pct"] > 0)
    & (coverage["coverage_pct"] < 5),
    "column"
].tolist()

print(f"Empty columns: {len(empty_cols)}")
display(empty_cols)

print(f"Sparse columns below 5% coverage: {len(sparse_cols)}")
display(sparse_cols)


### Review output

Preview the current table before continuing.


In [ ]:
regime_counts = (
    model_source_panel["seasonal_regime"]
    .value_counts(dropna=False)
    .rename_axis("seasonal_regime")
    .reset_index(name="rows")
)

display(regime_counts)


### Plot diagnostic chart

produces the diagnostic plot for the preceding transformation.


In [ ]:
seasonal_counts = model_source_panel["seasonal_regime"].value_counts().sort_index()

ax = seasonal_counts.plot(kind="bar", figsize=(9, 4))
ax.set_title("Seasonal regime row balance")
ax.set_xlabel("Seasonal regime")
ax.set_ylabel("Rows")
ax.grid(axis="y", alpha=0.25)
plt.xticks(rotation=45, ha="right")
plt.tight_layout()
plt.show()


### Map source columns to modules

groups source columns by the forecast module that will consume them.


In [ ]:
module_cols = {
    "acreage": [
        "planted_acres",
        "harvested_acres",
    ],
    "yield": [
        "yield_bu_per_acre",
        "crop_condition",
        "crop_progress",
        "gdd_corn_f",
        "drought_d0_or_worse_pct_area",
        "drought_d1_or_worse_pct_area",
        "drought_d2_or_worse_pct_area",
        "drought_d3_or_worse_pct_area",
    ],
    "production": [
        "production_bu",
        "harvested_acres",
        "yield_bu_per_acre",
    ],
    "exports": [
        "exports",
        "ty_exports",
        "weekly_exports",
        "accumulated_exports",
        "export_pace_4w",
        "export_pace_4w_clean",
    ],
    "ethanol": [
        "ethanol",
        "ethanol_pace_4w",
        "ethanol_pace_4w_clean",
        "ethanol_pace_change_4w_clean",
    ],
    "feed_residual": [
        "feed_and_residual",
        "feed_dom_consumption",
    ],
    "stocks": [
        "beginning_stocks",
        "ending_stocks",
        "total_use",
        "stocks_to_use",
    ],
    "weather": [
        "prcp",
        "tmax",
        "tmin",
        "gdd_corn_f",
        "drought_d0_or_worse_pct_area",
        "drought_d1_or_worse_pct_area",
        "drought_d2_or_worse_pct_area",
        "drought_d3_or_worse_pct_area",
    ],
    "positioning": [
        "managed_money_net",
        "managed_money_net_pct_oi",
        "managed_money_net_z_1y_clean",
        "managed_money_net_change_clean",
        "producer_merchant_net",
    ],
    "basis": [
        "basis_max",
        "basis_min",
        "basis_max_change",
        "basis_min_change",
    ],
    "front_futures": [
        "front_settle",
        "front_close",
        "front_volume",
        "front_open_interest",
        "front_days_to_last_trade",
    ],
}


### Map source columns to modules

groups source columns by the forecast module that will consume them.


In [ ]:
rows = []

for module, columns in module_cols.items():
    existing_cols = [col for col in columns if col in model_source_panel.columns]
    missing_cols = [col for col in columns if col not in model_source_panel.columns]

    if existing_cols:
        coverages = [model_source_panel[col].notna().mean() * 100 for col in existing_cols]
        average_coverage = np.mean(coverages)
        minimum_coverage = np.min(coverages)
    else:
        average_coverage = np.nan
        minimum_coverage = np.nan

    rows.append({
        "module": module,
        "existing_cols": existing_cols,
        "missing_cols": missing_cols,
        "average_coverage_pct": average_coverage,
        "minimum_coverage_pct": minimum_coverage,
    })

module_audit = pd.DataFrame(rows)
module_audit = module_audit.sort_values("average_coverage_pct", ascending=False)

display(module_audit)


### Transform data

Prepare the next modeling table used by the workflow.


In [ ]:
physical_checks = []

if {"planted_acres", "harvested_acres"}.issubset(model_source_panel.columns):
    raw_harvest_ratio = model_source_panel["harvested_acres"] / model_source_panel["planted_acres"]
    raw_harvest_ratio = raw_harvest_ratio.replace([np.inf, -np.inf], np.nan)

    plausible_ratio = raw_harvest_ratio.between(0.50, 1.05)

    physical_checks.append({
        "check": "harvested_acres / planted_acres between 0.50 and 1.05",
        "non_null": raw_harvest_ratio.notna().sum(),
        "pass_count": plausible_ratio.sum(),
        "pass_share_pct": plausible_ratio.mean() * 100,
        "median_value": raw_harvest_ratio.median(),
        "min_value": raw_harvest_ratio.min(),
        "max_value": raw_harvest_ratio.max(),
    })

if {"harvested_acres", "yield_bu_per_acre", "production_bu"}.issubset(model_source_panel.columns):
    production_calc = model_source_panel["harvested_acres"] * model_source_panel["yield_bu_per_acre"]

    production_error = (
        (production_calc - model_source_panel["production_bu"]).abs()
        / model_source_panel["production_bu"].abs().replace(0, np.nan)
    )

    production_error = production_error.replace([np.inf, -np.inf], np.nan)

    physical_checks.append({
        "check": "production_bu vs harvested_acres * yield_bu_per_acre within 1%",
        "non_null": production_error.notna().sum(),
        "pass_count": (production_error <= 0.01).sum(),
        "pass_share_pct": (production_error <= 0.01).mean() * 100,
        "median_value": production_error.median(),
        "min_value": production_error.min(),
        "max_value": production_error.max(),
    })


### Review output

Preview the current table before continuing.


In [ ]:
if {"ending_stocks", "total_use", "stocks_to_use"}.issubset(model_source_panel.columns):
    stocks_to_use_calc = model_source_panel["ending_stocks"] / model_source_panel["total_use"]

    stocks_to_use_error = (
        (stocks_to_use_calc - model_source_panel["stocks_to_use"]).abs()
        / model_source_panel["stocks_to_use"].abs().replace(0, np.nan)
    )

    stocks_to_use_error = stocks_to_use_error.replace([np.inf, -np.inf], np.nan)

    physical_checks.append({
        "check": "stocks_to_use = ending_stocks / total_use",
        "non_null": stocks_to_use_error.notna().sum(),
        "pass_count": (stocks_to_use_error <= 1e-8).sum(),
        "pass_share_pct": (stocks_to_use_error <= 1e-8).mean() * 100,
        "median_value": stocks_to_use_error.median(),
        "min_value": stocks_to_use_error.min(),
        "max_value": stocks_to_use_error.max(),
    })

physical_audit = pd.DataFrame(physical_checks)

display(physical_audit)


### Create the modelling panel

creates the working copy used by all forecast modules.


In [ ]:
modeling_panel = model_source_panel.drop(columns=empty_cols).copy()

print("modeling_panel created")
print(f"Rows: {modeling_panel.shape[0]:,}")
print(f"Columns: {modeling_panel.shape[1]:,}")
print("Note: asof_date is kept as a normal column, not as the index.")


### Transform data

Prepare the next modeling table used by the workflow.


In [ ]:
modeling_panel = modeling_panel.copy()

def safe_divide(numerator, denominator):
    denominator = denominator.replace(0, np.nan)
    return (numerator / denominator).replace([np.inf, -np.inf], np.nan)

def relative_error(estimate, actual):
    actual = actual.abs().replace(0, np.nan)
    return ((estimate - actual).abs() / actual).replace([np.inf, -np.inf], np.nan)

def coverage_pct(series):
    return series.notna().mean() * 100


### Transform data

Prepare the next modeling table used by the workflow.


In [ ]:
raw_physical_cols = [
    "planted_acres",
    "harvested_acres",
    "area_harvested",
    "yield",
    "yield_bu_per_acre",
    "production",
    "production_bu",
]

for col in raw_physical_cols:
    if col in modeling_panel.columns:
        raw_col = f"{col}_raw"
        if raw_col not in modeling_panel.columns:
            modeling_panel[raw_col] = modeling_panel[col]


### Transform data

Prepare the next modeling table used by the workflow.


In [ ]:
modeling_panel["harvest_ratio_raw"] = safe_divide(
    modeling_panel["harvested_acres"],
    modeling_panel["planted_acres"],
)

modeling_panel["harvest_ratio_raw_plausible"] = modeling_panel["harvest_ratio_raw"].between(
    0.50,
    1.05,
)

modeling_panel["production_bu_calc_raw"] = (
    modeling_panel["harvested_acres"] * modeling_panel["yield_bu_per_acre"]
)

modeling_panel["production_bu_raw_error"] = relative_error(
    modeling_panel["production_bu_calc_raw"],
    modeling_panel["production_bu"],
)

modeling_panel["production_bu_raw_pass"] = modeling_panel["production_bu_raw_error"] <= 0.01

modeling_panel["production_calc_area_yield_raw"] = (
    modeling_panel["area_harvested"] * modeling_panel["yield"]
)

modeling_panel["production_area_yield_error"] = relative_error(
    modeling_panel["production_calc_area_yield_raw"],
    modeling_panel["production"],
)

modeling_panel["production_area_yield_pass"] = modeling_panel["production_area_yield_error"] <= 0.01


### Transform data

Prepare the next modeling table used by the workflow.


In [ ]:
modeling_panel["harvested_acres_implied"] = safe_divide(
    modeling_panel["production_bu"],
    modeling_panel["yield_bu_per_acre"],
)

modeling_panel["harvested_acres_implied_mil"] = (
    modeling_panel["harvested_acres_implied"] / 1_000_000
)

corn_rules = {
    "production_min": 8_000_000_000,
    "production_max": 18_000_000_000,
    "yield_min": 80,
    "yield_max": 230,
    "harvested_acres_min": 50_000_000,
    "harvested_acres_max": 105_000_000,
}

modeling_panel["production_national_like"] = modeling_panel["production_bu"].between(
    corn_rules["production_min"],
    corn_rules["production_max"],
)

modeling_panel["yield_national_like"] = modeling_panel["yield_bu_per_acre"].between(
    corn_rules["yield_min"],
    corn_rules["yield_max"],
)

modeling_panel["harvested_acres_implied_national_like"] = modeling_panel["harvested_acres_implied"].between(
    corn_rules["harvested_acres_min"],
    corn_rules["harvested_acres_max"],
)

modeling_panel["physical_national_like"] = (
    modeling_panel["production_national_like"]
    & modeling_panel["yield_national_like"]
    & modeling_panel["harvested_acres_implied_national_like"]
)


### Create rule based fields

creates rule based fields used by the forecasting workflow.


In [ ]:
modeling_panel["planted_acres_model"] = np.where(
    modeling_panel["planted_acres"].between(60_000_000, 105_000_000),
    modeling_panel["planted_acres"],
    np.nan,
)

modeling_panel["harvested_acres_model"] = np.where(
    modeling_panel["physical_national_like"],
    modeling_panel["harvested_acres_implied"],
    np.nan,
)

modeling_panel["yield_bu_per_acre_model"] = np.where(
    modeling_panel["physical_national_like"],
    modeling_panel["yield_bu_per_acre"],
    np.nan,
)

modeling_panel["production_bu_model"] = np.where(
    modeling_panel["physical_national_like"],
    modeling_panel["production_bu"],
    np.nan,
)

modeling_panel["harvest_ratio_model"] = safe_divide(
    modeling_panel["harvested_acres_model"],
    modeling_panel["planted_acres_model"],
)

modeling_panel["harvest_ratio_model_plausible"] = modeling_panel["harvest_ratio_model"].between(
    0.50,
    1.05,
)

modeling_panel["physical_crop_quality"] = np.select(
    [
        modeling_panel["physical_national_like"],
        modeling_panel["production_area_yield_pass"],
        modeling_panel["harvest_ratio_raw_plausible"],
    ],
    [
        "national_like_from_production_yield",
        "area_yield_identity_only",
        "raw_harvest_ratio_only",
    ],
    default="quarantined",
)


### Review output

Preview the current table before continuing.


In [ ]:
physical_coverage = pd.DataFrame({
    "field": [
        "planted_acres",
        "harvested_acres",
        "yield_bu_per_acre",
        "production_bu",
        "planted_acres_model",
        "harvested_acres_model",
        "yield_bu_per_acre_model",
        "production_bu_model",
    ],
    "coverage_pct": [
        coverage_pct(modeling_panel["planted_acres"]),
        coverage_pct(modeling_panel["harvested_acres"]),
        coverage_pct(modeling_panel["yield_bu_per_acre"]),
        coverage_pct(modeling_panel["production_bu"]),
        coverage_pct(modeling_panel["planted_acres_model"]),
        coverage_pct(modeling_panel["harvested_acres_model"]),
        coverage_pct(modeling_panel["yield_bu_per_acre_model"]),
        coverage_pct(modeling_panel["production_bu_model"]),
    ],
    "non_null": [
        modeling_panel["planted_acres"].notna().sum(),
        modeling_panel["harvested_acres"].notna().sum(),
        modeling_panel["yield_bu_per_acre"].notna().sum(),
        modeling_panel["production_bu"].notna().sum(),
        modeling_panel["planted_acres_model"].notna().sum(),
        modeling_panel["harvested_acres_model"].notna().sum(),
        modeling_panel["yield_bu_per_acre_model"].notna().sum(),
        modeling_panel["production_bu_model"].notna().sum(),
    ],
})

display(physical_coverage)


### Plot diagnostic chart

produces the diagnostic plot for the preceding transformation.


In [ ]:
chart_data = physical_coverage.sort_values("coverage_pct", ascending=True)

ax = chart_data.plot(
    kind="barh",
    x="column",
    y="coverage_pct",
    legend=False,
    figsize=(10, max(4, len(chart_data) * 0.35)),
)
ax.set_title("Physical input coverage")
ax.set_xlabel("Coverage percent")
ax.set_ylabel("")
ax.grid(axis="x", alpha=0.25)
plt.tight_layout()
plt.show()


### Audit physical input quality

checks whether the derived physical variables are internally consistent.


In [ ]:
physical_qa = pd.DataFrame({
    "check": [
        "raw harvest ratio plausible",
        "raw production identity within 1%",
        "area/yield/production identity within 1%",
        "national-like production/yield/implied acres",
    ],
    "pass_count": [
        modeling_panel["harvest_ratio_raw_plausible"].sum(),
        modeling_panel["production_bu_raw_pass"].sum(),
        modeling_panel["production_area_yield_pass"].sum(),
        modeling_panel["physical_national_like"].sum(),
    ],
    "pass_share_pct": [
        modeling_panel["harvest_ratio_raw_plausible"].mean() * 100,
        modeling_panel["production_bu_raw_pass"].mean() * 100,
        modeling_panel["production_area_yield_pass"].mean() * 100,
        modeling_panel["physical_national_like"].mean() * 100,
    ],
})

display(physical_qa)

quality_counts = (
    modeling_panel["physical_crop_quality"]
    .value_counts(dropna=False)
    .rename_axis("physical_crop_quality")
    .reset_index(name="rows")
)

display(quality_counts)


### Aggregate physical variables by crop year

produces crop year level physical aggregates used by later modules.


In [ ]:
physical_by_year = (
    modeling_panel
    .groupby("marketing_year")
    .agg(
        rows=("asof_date", "count"),
        raw_harvest_ratio_pass_share=("harvest_ratio_raw_plausible", "mean"),
        raw_production_identity_pass_share=("production_bu_raw_pass", "mean"),
        national_like_share=("physical_national_like", "mean"),
        planted_acres_model_median=("planted_acres_model", "median"),
        harvested_acres_model_median=("harvested_acres_model", "median"),
        yield_model_median=("yield_bu_per_acre_model", "median"),
        production_bu_model_median=("production_bu_model", "median"),
    )
    .reset_index()
)

share_cols = [
    "raw_harvest_ratio_pass_share",
    "raw_production_identity_pass_share",
    "national_like_share",
]

for col in share_cols:
    physical_by_year[col] = physical_by_year[col] * 100

display(physical_by_year)


### Aggregate physical variables by crop year

produces crop year level physical aggregates used by later modules.


In [ ]:
physical_plot_columns = [
    "planted_acres_model",
    "harvested_acres_model",
    "yield_model",
    "production_model_bil_bu",
]

plot_time_series(
    physical_by_year,
    date_col="crop_year",
    value_cols=physical_plot_columns,
    title="Physical variables by crop year",
    ylabel="Value",
)


### Review output

Preview the current table before continuing.


In [ ]:
bad_physical_rows = modeling_panel.loc[
    ~modeling_panel["physical_national_like"],
    [
        "asof_date",
        "marketing_year",
        "seasonal_regime",
        "planted_acres",
        "harvested_acres",
        "harvest_ratio_raw",
        "yield_bu_per_acre",
        "production_bu",
        "harvested_acres_implied_mil",
        "production_bu_raw_error",
        "physical_crop_quality",
    ],
].head(20)

display(bad_physical_rows)


### Review output

Preview the current table before continuing.


In [ ]:
physical_model_cols = [
    "planted_acres_model",
    "harvested_acres_model",
    "harvest_ratio_model",
    "yield_bu_per_acre_model",
    "production_bu_model",
    "physical_national_like",
    "physical_crop_quality",
]

display(physical_model_cols)

print("Use the *_model columns in forecast models.")
print("Do not use raw planted_acres, harvested_acres, yield_bu_per_acre, or production_bu directly.")


## Acreage forecast module

Build acreage priors, test adjustment models, and select the acreage forecast used downstream.


### Transform data

Prepare the next modeling table used by the workflow.


In [ ]:
#acerage model

#acerage
#planted_acres = USDA_prior + model_adjustment (what is model adjustment in this state?)
# The model should not trying to forecast planted acres from zero. It is trying to answer:
# Given what USDA has already told the market, is acreage likely too high, too low, or fairly priced?


#so acerage model should be doing :
# Before March Prospective Plantings:
    #USDA_prior = previous crop year final acres / trend expectation
    #adjustment = economics + stocks + positioning + weather setup

#After March Prospective Plantings:
    #USDA_prior = March planting intentions
    #adjustment = corn/soy economics + planting delay + weather/prevent plant risk

#After June Acreage:
    #USDA_prior = June planted acres
    #adjustment = late revisions / prevented planting / final NASS revision risk


### Transform data

Prepare the next modeling table used by the workflow.


In [ ]:
acreage_df = modeling_panel.copy()

if "asof_date" in acreage_df.index.names:
    acreage_df = acreage_df.reset_index(drop=True)

acreage_df["asof_date"] = pd.to_datetime(acreage_df["asof_date"], errors="coerce")
acreage_df["asof_timestamp"] = pd.to_datetime(acreage_df["asof_timestamp"], errors="coerce")

acreage_df = acreage_df.sort_values("asof_date").reset_index(drop=True)
acreage_df["acreage_crop_year"] = acreage_df["asof_date"].dt.year

acreage_df = acreage_df[
    (acreage_df["asof_date"] >= "2013-01-01")
    & (acreage_df["asof_date"].dt.month <= 8)
].copy()

print(f"Rows: {acreage_df.shape[0]:,}")
print(f"Date range: {acreage_df['asof_date'].min().date()} to {acreage_df['asof_date'].max().date()}")
print(f"Crop years: {acreage_df['acreage_crop_year'].min()} to {acreage_df['acreage_crop_year'].max()}")


### Review output

Preview the current table before continuing.


In [ ]:
final_planted_acres = {
    2013: 95.365,
    2014: 90.597,
    2015: 88.019,
    2016: 94.004,
    2017: 90.167,
    2018: 88.871,
    2019: 89.745,
    2020: 90.652,
    2021: 93.357,
    2022: 88.579,
    2023: 94.641,
    2024: 91.475,
    2025: 95.200,
    2026: np.nan,
}

march_intentions = {
    2025: 95.300,
    2026: 95.300,
}

june_acreage = {
    2025: 95.200,
}

target_table = (
    pd.Series(final_planted_acres, name="final_planted_acres_mil")
    .rename_axis("acreage_crop_year")
    .reset_index()
)

acreage_df = acreage_df.merge(target_table, on="acreage_crop_year", how="left")

display(target_table)


### Prepare acreage features

prepares the acreage training table and stage specific priors.


In [ ]:
acreage_df["prospective_plantings_date"] = pd.to_datetime(
    acreage_df["acreage_crop_year"].astype(str) + "-03-31"
)

acreage_df["acreage_report_date"] = pd.to_datetime(
    acreage_df["acreage_crop_year"].astype(str) + "-06-30"
)

acreage_df["acreage_stage"] = np.select(
    [
        acreage_df["asof_date"] < acreage_df["prospective_plantings_date"],
        acreage_df["asof_date"].between(
            acreage_df["prospective_plantings_date"],
            acreage_df["acreage_report_date"],
            inclusive="left",
        ),
        acreage_df["asof_date"] >= acreage_df["acreage_report_date"],
    ],
    [
        "pre_march_intentions",
        "post_march_pre_june",
        "post_june_acreage",
    ],
    default="unknown",
)

stage_counts = (
    acreage_df
    .groupby(["acreage_crop_year", "acreage_stage"])
    .size()
    .reset_index(name="rows")
    .pivot(index="acreage_crop_year", columns="acreage_stage", values="rows")
    .fillna(0)
)

display(stage_counts)


### Define median_or_nan

defines `median_or_nan` so later cells can reuse the same transformation without repeating code.


In [ ]:
def median_or_nan(series):
    series = pd.Series(series).dropna()
    return np.nan if series.empty else float(series.median())

def sum_or_nan(series):
    series = pd.Series(series).dropna()
    return np.nan if series.empty else float(series.sum())

def clipped_z(value, center, scale, lower=-3, upper=3):
    if pd.isna(value) or scale == 0:
        return 0.0

    z = (value - center) / scale
    return float(np.clip(z, lower, upper))


### Build annual acreage signals

builds the annual acreage signals used to test adjustment models.


In [ ]:
annual_rows = []

for crop_year in sorted(acreage_df["acreage_crop_year"].dropna().unique()):
    crop_year = int(crop_year)

    rows = acreage_df[acreage_df["acreage_crop_year"] == crop_year]
    jan_feb = rows[rows["asof_date"].dt.month.isin([1, 2])]
    apr_may = rows[rows["asof_date"].dt.month.isin([4, 5])]

    lag_values = [
        final_planted_acres.get(crop_year - lag)
        for lag in [1, 2, 3]
    ]

    lag_values = [x for x in lag_values if not pd.isna(x)]

    annual_rows.append({
        "acreage_crop_year": crop_year,
        "final_planted_acres_mil": final_planted_acres.get(crop_year, np.nan),
        "lag1_final_planted_acres_mil": final_planted_acres.get(crop_year - 1, np.nan),
        "lag3_avg_final_planted_acres_mil": np.nan if not lag_values else float(np.mean(lag_values)),

        "jan_feb_front_settle": median_or_nan(jan_feb["front_settle"]),
        "jan_feb_stocks_to_use": median_or_nan(jan_feb["stocks_to_use"]),
        "jan_feb_managed_money_z": median_or_nan(jan_feb["managed_money_net_z_1y_clean"]),

        "apr_may_prcp": sum_or_nan(apr_may["prcp"]),
        "apr_may_crop_progress": median_or_nan(apr_may["crop_progress"]),
        "apr_may_drought_d1": median_or_nan(apr_may["drought_d1_or_worse_pct_area"]),
    })

annual_signals = pd.DataFrame(annual_rows)
annual_signals = annual_signals.sort_values("acreage_crop_year").reset_index(drop=True)

display(annual_signals)


### Build annual acreage signals

builds the annual acreage signals used to test adjustment models.


In [ ]:
annual_signals["price_pressure"] = annual_signals["jan_feb_front_settle"].apply(
    lambda x: clipped_z(x, center=450.0, scale=125.0)
)

annual_signals["stocks_tightness_pressure"] = annual_signals["jan_feb_stocks_to_use"].apply(
    lambda x: clipped_z(0.125 - x, center=0.0, scale=0.040)
)

annual_signals["positioning_pressure"] = (
    annual_signals["jan_feb_managed_money_z"]
    .fillna(0.0)
    .clip(-3, 3)
)

annual_signals["spring_wetness_pressure"] = annual_signals["apr_may_prcp"].apply(
    lambda x: clipped_z(x, center=2.5, scale=1.5)
)

annual_signals["planting_progress_pressure"] = annual_signals["apr_may_crop_progress"].apply(
    lambda x: clipped_z(x, center=45.0, scale=20.0)
)

annual_signals["drought_pressure"] = annual_signals["apr_may_drought_d1"].apply(
    lambda x: clipped_z(x, center=25.0, scale=15.0)
)

pressure_cols = [
    "acreage_crop_year",
    "price_pressure",
    "stocks_tightness_pressure",
    "positioning_pressure",
    "spring_wetness_pressure",
    "planting_progress_pressure",
    "drought_pressure",
]

display(annual_signals[pressure_cols])


### Build annual acreage signals

builds the annual acreage signals used to test adjustment models.


In [ ]:
annual_signal_columns = [
    "price_pressure",
    "stocks_tightness_pressure",
    "positioning_pressure",
    "spring_wetness_pressure",
    "planting_progress_pressure",
    "drought_pressure",
]

plot_time_series(
    annual_signals,
    date_col="acreage_crop_year",
    value_cols=annual_signal_columns,
    title="Annual acreage signal inputs",
    ylabel="Signal value",
)


### Build annual acreage signals

builds the annual acreage signals used to test adjustment models.


In [ ]:
rng = np.random.default_rng(456)

annual_signals["pre_march_base_prior_mil"] = (
    0.65 * annual_signals["lag1_final_planted_acres_mil"]
    + 0.35 * annual_signals["lag3_avg_final_planted_acres_mil"]
)

annual_signals["acreage_econ_adjustment_mil"] = (
    0.75 * annual_signals["price_pressure"]
    + 0.65 * annual_signals["stocks_tightness_pressure"]
    + 0.20 * annual_signals["positioning_pressure"]
)

annual_signals["planting_disruption_adjustment_mil"] = (
    -0.45 * annual_signals["spring_wetness_pressure"]
    + 0.45 * annual_signals["planting_progress_pressure"]
    - 0.20 * annual_signals["drought_pressure"]
)

annual_signals["pre_march_prior_mil"] = (
    annual_signals["pre_march_base_prior_mil"]
    + annual_signals["acreage_econ_adjustment_mil"]
)

annual_signals["simulated_march_prior_mil"] = (
    annual_signals["pre_march_prior_mil"]
    + 0.25 * annual_signals["acreage_econ_adjustment_mil"]
    + rng.normal(0.0, 0.45, len(annual_signals))
)

annual_signals["simulated_june_prior_mil"] = (
    annual_signals["simulated_march_prior_mil"]
    + annual_signals["planting_disruption_adjustment_mil"]
    + rng.normal(0.0, 0.25, len(annual_signals))
)

for col in ["pre_march_prior_mil", "simulated_march_prior_mil", "simulated_june_prior_mil"]:
    annual_signals[col] = annual_signals[col].clip(80.0, 105.0)


### Build annual acreage signals

builds the annual acreage signals used to test adjustment models.


In [ ]:
display(
    annual_signals[
        [
            "acreage_crop_year",
            "final_planted_acres_mil",
            "pre_march_prior_mil",
            "simulated_march_prior_mil",
            "simulated_june_prior_mil",
        ]
    ]
)


### Build annual acreage signals

builds the annual acreage signals used to test adjustment models.


In [ ]:
annual_signals["official_march_intentions_mil"] = annual_signals["acreage_crop_year"].map(
    march_intentions
)

annual_signals["official_june_acreage_mil"] = annual_signals["acreage_crop_year"].map(
    june_acreage
)

annual_signals["march_prior_mil"] = annual_signals["official_march_intentions_mil"].combine_first(
    annual_signals["simulated_march_prior_mil"]
)

annual_signals["june_prior_mil"] = annual_signals["official_june_acreage_mil"].combine_first(
    annual_signals["simulated_june_prior_mil"]
)

display(
    annual_signals[
        [
            "acreage_crop_year",
            "pre_march_prior_mil",
            "march_prior_mil",
            "june_prior_mil",
            "official_march_intentions_mil",
            "official_june_acreage_mil",
        ]
    ]
)


### Prepare acreage features

prepares the acreage training table and stage specific priors.


In [ ]:
annual_prior_cols = [
    "acreage_crop_year",
    "lag1_final_planted_acres_mil",
    "lag3_avg_final_planted_acres_mil",
    "price_pressure",
    "stocks_tightness_pressure",
    "positioning_pressure",
    "spring_wetness_pressure",
    "planting_progress_pressure",
    "drought_pressure",
    "pre_march_prior_mil",
    "march_prior_mil",
    "june_prior_mil",
    "official_march_intentions_mil",
    "official_june_acreage_mil",
]

acreage_df = acreage_df.merge(
    annual_signals[annual_prior_cols],
    on="acreage_crop_year",
    how="left",
)

acreage_df["acreage_prior_mil"] = np.select(
    [
        acreage_df["acreage_stage"] == "pre_march_intentions",
        acreage_df["acreage_stage"] == "post_march_pre_june",
        acreage_df["acreage_stage"] == "post_june_acreage",
    ],
    [
        acreage_df["pre_march_prior_mil"],
        acreage_df["march_prior_mil"],
        acreage_df["june_prior_mil"],
    ],
    default=np.nan,
)


### Prepare acreage features

prepares the acreage training table and stage specific priors.


In [ ]:
acreage_df["acreage_prior_source"] = np.select(
    [
        acreage_df["acreage_stage"] == "pre_march_intentions",
        (acreage_df["acreage_stage"] == "post_march_pre_june")
        & acreage_df["official_march_intentions_mil"].notna(),
        (acreage_df["acreage_stage"] == "post_march_pre_june")
        & acreage_df["official_march_intentions_mil"].isna(),
        (acreage_df["acreage_stage"] == "post_june_acreage")
        & acreage_df["official_june_acreage_mil"].notna(),
        (acreage_df["acreage_stage"] == "post_june_acreage")
        & acreage_df["official_june_acreage_mil"].isna(),
    ],
    [
        "simulated_pre_march",
        "official_march_intentions",
        "simulated_march_intentions",
        "official_june_acreage",
        "simulated_june_acreage",
    ],
    default="unknown",
)

acreage_df["prior_uses_current_year_final"] = False

display(
    acreage_df[
        [
            "asof_date",
            "acreage_crop_year",
            "acreage_stage",
            "acreage_prior_mil",
            "acreage_prior_source",
        ]
    ].head()
)


### Review output

Preview the current table before continuing.


In [ ]:
acreage_df["acreage_adjustment_target_mil"] = (
    acreage_df["final_planted_acres_mil"]
    - acreage_df["acreage_prior_mil"]
)

acreage_df["has_acreage_target"] = (
    acreage_df["final_planted_acres_mil"].notna()
    & acreage_df["acreage_prior_mil"].notna()
)

target_summary = (
    acreage_df
    .loc[acreage_df["has_acreage_target"], "acreage_adjustment_target_mil"]
    .describe()
    .to_frame()
    .T
)

display(target_summary)


### Summarize grouped data

aggregates the panel into a compact summary used by the following modelling step.


In [ ]:
acreage_df["month"] = acreage_df["asof_date"].dt.month
acreage_df["dayofyear"] = acreage_df["asof_date"].dt.dayofyear

acreage_df["prior_vs_lag1_mil"] = (
    acreage_df["acreage_prior_mil"]
    - acreage_df["lag1_final_planted_acres_mil"]
)

acreage_df["prior_vs_lag3_mil"] = (
    acreage_df["acreage_prior_mil"]
    - acreage_df["lag3_avg_final_planted_acres_mil"]
)

acreage_df["prcp_20d_sum"] = (
    acreage_df
    .sort_values("asof_date")
    .groupby("acreage_crop_year")["prcp"]
    .transform(lambda s: s.rolling(20, min_periods=5).sum())
)

acreage_df["crop_progress_change_4w"] = (
    acreage_df
    .sort_values("asof_date")
    .groupby("acreage_crop_year")["crop_progress"]
    .transform(lambda s: s.diff(20))
)


### Prepare acreage features

prepares the acreage training table and stage specific priors.


In [ ]:
acreage_feature_cols = [
    "month",
    "dayofyear",
    "acreage_stage",
    "seasonal_regime",

    "acreage_prior_mil",
    "prior_vs_lag1_mil",
    "prior_vs_lag3_mil",

    "price_pressure",
    "stocks_tightness_pressure",
    "positioning_pressure",
    "spring_wetness_pressure",
    "planting_progress_pressure",
    "drought_pressure",

    "front_settle",
    "front_return_5d",
    "front_realized_vol_20d",
    "stocks_to_use",

    "prcp",
    "tmax",
    "tmin",
    "gdd_corn_f",
    "crop_progress",
    "prcp_20d_sum",
    "crop_progress_change_4w",

    "drought_d0_or_worse_pct_area",
    "drought_d1_or_worse_pct_area",
    "drought_d2_or_worse_pct_area",
    "drought_d3_or_worse_pct_area",

    "managed_money_net_pct_oi",
    "managed_money_net_z_1y_clean",
    "managed_money_net_change_clean",
    "producer_merchant_net",
]

acreage_feature_cols = [
    col for col in acreage_feature_cols
    if col in acreage_df.columns
]

acreage_target_col = "acreage_adjustment_target_mil"


### Review output

Preview the current table before continuing.


In [ ]:
display(acreage_feature_cols)


### Prepare acreage features

prepares the acreage training table and stage specific priors.


In [ ]:
acreage_train_daily = acreage_df[acreage_df["has_acreage_target"]].copy()

acreage_train_events = (
    acreage_train_daily
    .sort_values("asof_date")
    .groupby(["acreage_crop_year", "acreage_stage"], as_index=False)
    .tail(1)
    .sort_values(["acreage_crop_year", "asof_date"])
    .reset_index(drop=True)
)

acreage_live = acreage_df[
    acreage_df["final_planted_acres_mil"].isna()
    & acreage_df["acreage_prior_mil"].notna()
].copy()

print(f"Daily training rows: {acreage_train_daily.shape[0]:,}")
print(f"Event training rows: {acreage_train_events.shape[0]:,}")
print(f"Live rows: {acreage_live.shape[0]:,}")


### Review output

Preview the current table before continuing.


In [ ]:
display(
    acreage_train_events[
        [
            "asof_date",
            "acreage_crop_year",
            "acreage_stage",
            "final_planted_acres_mil",
            "acreage_prior_mil",
            "acreage_adjustment_target_mil",
            "acreage_prior_source",
            "prior_uses_current_year_final",
        ]
    ]
)


### Review output

Preview the current table before continuing.


In [ ]:
prior_source_counts = (
    acreage_df["acreage_prior_source"]
    .value_counts(dropna=False)
    .rename_axis("acreage_prior_source")
    .reset_index(name="rows")
)

display(prior_source_counts)


### Review output

Preview the current table before continuing.


In [ ]:
acreage_model_features = [
    "acreage_stage",
    "acreage_prior_mil",
    "prior_vs_lag1_mil",
    "prior_vs_lag3_mil",
    "price_pressure",
    "stocks_tightness_pressure",
    "positioning_pressure",
    "spring_wetness_pressure",
    "planting_progress_pressure",
    "drought_pressure",
]

acreage_model_features = [
    col for col in acreage_model_features
    if col in acreage_train_events.columns
]

acreage_model_target = "acreage_adjustment_target_mil"

display(acreage_model_features)


### Review output

Preview the current table before continuing.


In [ ]:
base_cols = [
    "asof_date",
    "acreage_crop_year",
    "acreage_stage",
    "acreage_prior_mil",
    "final_planted_acres_mil",
]

model_cols = base_cols + acreage_model_features + [acreage_model_target]

model_cols = list(dict.fromkeys(model_cols))

acreage_model_data = acreage_train_events[model_cols].copy()

acreage_model_data = acreage_model_data.loc[:, ~acreage_model_data.columns.duplicated()]

acreage_model_data = acreage_model_data.dropna(
    subset=[acreage_model_target, "acreage_prior_mil"]
)

print(f"Rows: {acreage_model_data.shape[0]:,}")
print(f"Columns: {acreage_model_data.shape[1]:,}")
print(f"Duplicate columns: {acreage_model_data.columns.duplicated().sum()}")
print(f"Crop years: {acreage_model_data['acreage_crop_year'].min()} to {acreage_model_data['acreage_crop_year'].max()}")

display(acreage_model_data.head())


### Review output

Preview the current table before continuing.


In [ ]:
nass_acreage_history = pd.DataFrame([
    {"acreage_crop_year": 2014, "march_intentions_mil": 91.691, "june_acreage_mil": 91.641, "final_planted_acres_mil": 90.597, "target_status": "final_nass_track_record"},
    {"acreage_crop_year": 2015, "march_intentions_mil": 89.199, "june_acreage_mil": 88.897, "final_planted_acres_mil": 88.019, "target_status": "final_nass_track_record"},
    {"acreage_crop_year": 2016, "march_intentions_mil": 93.601, "june_acreage_mil": 94.148, "final_planted_acres_mil": 94.004, "target_status": "final_nass_track_record"},
    {"acreage_crop_year": 2017, "march_intentions_mil": 89.996, "june_acreage_mil": 90.886, "final_planted_acres_mil": 90.167, "target_status": "final_nass_track_record"},
    {"acreage_crop_year": 2018, "march_intentions_mil": 88.026, "june_acreage_mil": 89.128, "final_planted_acres_mil": 88.871, "target_status": "latest_nass_track_record_revision"},
    {"acreage_crop_year": 2019, "march_intentions_mil": 92.792, "june_acreage_mil": 91.700, "final_planted_acres_mil": 89.745, "target_status": "latest_nass_track_record_revision"},
    {"acreage_crop_year": 2020, "march_intentions_mil": 96.990, "june_acreage_mil": 92.006, "final_planted_acres_mil": 90.652, "target_status": "latest_nass_track_record_revision"},
    {"acreage_crop_year": 2021, "march_intentions_mil": 91.144, "june_acreage_mil": 92.692, "final_planted_acres_mil": 93.252, "target_status": "latest_nass_track_record_revision"},
    {"acreage_crop_year": 2022, "march_intentions_mil": 89.490, "june_acreage_mil": 89.921, "final_planted_acres_mil": 88.589, "target_status": "latest_nass_track_record_revision"},
    {"acreage_crop_year": 2023, "march_intentions_mil": 91.996, "june_acreage_mil": 94.096, "final_planted_acres_mil": 94.641, "target_status": "latest_nass_track_record_revision"},
    {"acreage_crop_year": 2024, "march_intentions_mil": 90.036, "june_acreage_mil": 91.475, "final_planted_acres_mil": 90.909, "target_status": "latest_nass_crop_production_summary"},
    {"acreage_crop_year": 2025, "march_intentions_mil": 95.326, "june_acreage_mil": 95.203, "final_planted_acres_mil": 98.788, "target_status": "latest_nass_crop_production_summary"},
    {"acreage_crop_year": 2026, "march_intentions_mil": 95.300, "june_acreage_mil": np.nan, "final_planted_acres_mil": np.nan, "target_status": "live_crop_year"},
])

nass_acreage_history["march_error_mil"] = (
    nass_acreage_history["final_planted_acres_mil"]
    - nass_acreage_history["march_intentions_mil"]
)

nass_acreage_history["june_error_mil"] = (
    nass_acreage_history["final_planted_acres_mil"]
    - nass_acreage_history["june_acreage_mil"]
)

display(nass_acreage_history)


### Prepare acreage features

prepares the acreage training table and stage specific priors.


In [ ]:
cols_to_drop = [
    "final_planted_acres_mil",
    "march_intentions_mil",
    "june_acreage_mil",
    "acreage_prior_mil",
    "acreage_prior_source",
    "acreage_adjustment_target_mil",
    "has_acreage_target",
    "target_status",
]

acreage_df = acreage_df.drop(columns=cols_to_drop, errors="ignore")

acreage_df = acreage_df.merge(
    nass_acreage_history[
        [
            "acreage_crop_year",
            "final_planted_acres_mil",
            "march_intentions_mil",
            "june_acreage_mil",
            "target_status",
        ]
    ],
    on="acreage_crop_year",
    how="left",
)

acreage_df["acreage_prior_mil"] = np.select(
    [
        acreage_df["acreage_stage"] == "pre_march_intentions",
        acreage_df["acreage_stage"] == "post_march_pre_june",
        acreage_df["acreage_stage"] == "post_june_acreage",
    ],
    [
        acreage_df["pre_march_prior_mil"],
        acreage_df["march_intentions_mil"],
        acreage_df["june_acreage_mil"],
    ],
    default=np.nan,
)


### Prepare acreage features

prepares the acreage training table and stage specific priors.


In [ ]:
acreage_df["acreage_prior_source"] = np.select(
    [
        acreage_df["acreage_stage"] == "pre_march_intentions",
        acreage_df["acreage_stage"] == "post_march_pre_june",
        acreage_df["acreage_stage"] == "post_june_acreage",
    ],
    [
        "model_pre_march_prior",
        "nass_march_prospective_plantings",
        "nass_june_acreage",
    ],
    default="unknown",
)

acreage_df["acreage_adjustment_target_mil"] = (
    acreage_df["final_planted_acres_mil"]
    - acreage_df["acreage_prior_mil"]
)

acreage_df["has_acreage_target"] = (
    acreage_df["final_planted_acres_mil"].notna()
    & acreage_df["acreage_prior_mil"].notna()
)


### Prepare acreage features

prepares the acreage training table and stage specific priors.


In [ ]:
acreage_train_daily = acreage_df[acreage_df["has_acreage_target"]].copy()

acreage_train_events = (
    acreage_train_daily
    .sort_values("asof_date")
    .groupby(["acreage_crop_year", "acreage_stage"], as_index=False)
    .tail(1)
    .sort_values(["acreage_crop_year", "asof_date"])
    .reset_index(drop=True)
)

acreage_live = acreage_df[
    acreage_df["final_planted_acres_mil"].isna()
    & acreage_df["acreage_prior_mil"].notna()
].copy()

print(f"Daily training rows: {acreage_train_daily.shape[0]:,}")
print(f"Event training rows: {acreage_train_events.shape[0]:,}")
print(f"Live rows: {acreage_live.shape[0]:,}")

display(
    acreage_train_events[
        [
            "asof_date",
            "acreage_crop_year",
            "acreage_stage",
            "final_planted_acres_mil",
            "acreage_prior_mil",
            "acreage_adjustment_target_mil",
            "acreage_prior_source",
            "target_status",
        ]
    ]
)


### Review output

Preview the current table before continuing.


In [ ]:
acreage_model_features = [
    "acreage_stage",
    "acreage_prior_mil",
    "prior_vs_lag1_mil",
    "prior_vs_lag3_mil",
    "price_pressure",
    "stocks_tightness_pressure",
    "positioning_pressure",
    "spring_wetness_pressure",
    "planting_progress_pressure",
    "drought_pressure",
]

acreage_model_features = [
    col for col in acreage_model_features
    if col in acreage_train_events.columns
]

acreage_model_target = "acreage_adjustment_target_mil"

base_cols = [
    "asof_date",
    "acreage_crop_year",
    "acreage_stage",
    "acreage_prior_mil",
    "final_planted_acres_mil",
]

model_cols = list(dict.fromkeys(base_cols + acreage_model_features + [acreage_model_target]))

acreage_model_data = acreage_train_events[model_cols].copy()
acreage_model_data = acreage_model_data.loc[:, ~acreage_model_data.columns.duplicated()]
acreage_model_data = acreage_model_data.dropna(subset=[acreage_model_target, "acreage_prior_mil"])

print(f"Rows: {acreage_model_data.shape[0]:,}")
print(f"Crop years: {acreage_model_data['acreage_crop_year'].min()} to {acreage_model_data['acreage_crop_year'].max()}")

display(acreage_model_data)


### Define predict_zero_adjustment

defines `predict_zero_adjustment` so later cells can reuse the same transformation without repeating code.


In [ ]:
def predict_zero_adjustment(train_data, test_data):
    return np.zeros(len(test_data))


def predict_stage_median_adjustment(train_data, test_data, shrink=0.25):
    predictions = []

    for _, row in test_data.iterrows():
        stage = row["acreage_stage"]

        stage_history = train_data.loc[
            train_data["acreage_stage"] == stage,
            acreage_model_target,
        ]

        if len(stage_history) >= 2:
            adjustment = stage_history.median()
        else:
            adjustment = train_data[acreage_model_target].median()

        predictions.append(shrink * adjustment)

    return np.array(predictions)


### Define predict_stage_hybrid

defines `predict_stage_hybrid` so later cells can reuse the same transformation without repeating code.


In [ ]:
def predict_stage_hybrid(train_data, test_data):
    predictions = []

    for _, row in test_data.iterrows():
        stage = row["acreage_stage"]

        stage_history = train_data.loc[
            train_data["acreage_stage"] == stage,
            acreage_model_target,
        ]

        if len(stage_history) >= 2:
            adjustment = stage_history.median()
        else:
            adjustment = train_data[acreage_model_target].median()

        if stage == "pre_march_intentions":
            shrink = 0.50
        elif stage == "post_march_pre_june":
            shrink = 0.00
        elif stage == "post_june_acreage":
            shrink = 0.50
        else:
            shrink = 0.00

        predictions.append(shrink * adjustment)

    return np.array(predictions)


### Transform data

Prepare the next modeling table used by the workflow.


In [ ]:
candidate_models = {
    "no_adjustment_baseline": lambda train, test: predict_zero_adjustment(train, test),
    "stage_median_shrink_10": lambda train, test: predict_stage_median_adjustment(train, test, shrink=0.10),
    "stage_median_shrink_25": lambda train, test: predict_stage_median_adjustment(train, test, shrink=0.25),
    "stage_median_shrink_50": lambda train, test: predict_stage_median_adjustment(train, test, shrink=0.50),
    "stage_specific_hybrid": lambda train, test: predict_stage_hybrid(train, test),
}

min_train_years = 4
rows = []

crop_years = sorted(acreage_model_data["acreage_crop_year"].unique())

for test_year in crop_years:
    train_years = [year for year in crop_years if year < test_year]

    if len(train_years) < min_train_years:
        continue

    train_data = acreage_model_data[
        acreage_model_data["acreage_crop_year"].isin(train_years)
    ].copy()

    test_data = acreage_model_data[
        acreage_model_data["acreage_crop_year"] == test_year
    ].copy()

    for model_name, model_func in candidate_models.items():
        candidate_frame = test_data.copy()
        candidate_frame["model_name"] = model_name
        candidate_frame["predicted_adjustment_mil"] = model_func(train_data, test_data)
        candidate_frame["forecast_acres_mil"] = candidate_frame["acreage_prior_mil"] + candidate_frame["predicted_adjustment_mil"]
        rows.append(candidate_frame)

acreage_oos = pd.concat(rows, ignore_index=True)


### Review output

Preview the current table before continuing.


In [ ]:
display(
    acreage_oos[
        [
            "model_name",
            "asof_date",
            "acreage_crop_year",
            "acreage_stage",
            "final_planted_acres_mil",
            "acreage_prior_mil",
            "predicted_adjustment_mil",
            "forecast_acres_mil",
        ]
    ].head(20)
)


### Score acreage models

evaluates acreage models with expanding result_series of sample tests.


In [ ]:
from sklearn.metrics import mean_absolute_error, mean_squared_error

scores = []

for model_name, group in acreage_oos.groupby("model_name"):
    mae = mean_absolute_error(
        group["final_planted_acres_mil"],
        group["forecast_acres_mil"],
    )

    rmse = np.sqrt(
        mean_squared_error(
            group["final_planted_acres_mil"],
            group["forecast_acres_mil"],
        )
    )

    scores.append({
        "model_name": model_name,
        "rows": len(group),
        "mae_mil_acres": mae,
        "rmse_mil_acres": rmse,
    })

acreage_scores = pd.DataFrame(scores)

baseline_mae = acreage_scores.loc[
    acreage_scores["model_name"] == "no_adjustment_baseline",
    "mae_mil_acres",
].iloc[0]

baseline_rmse = acreage_scores.loc[
    acreage_scores["model_name"] == "no_adjustment_baseline",
    "rmse_mil_acres",
].iloc[0]

acreage_scores["mae_improvement_vs_baseline"] = (
    baseline_mae - acreage_scores["mae_mil_acres"]
)

acreage_scores["rmse_improvement_vs_baseline"] = (
    baseline_rmse - acreage_scores["rmse_mil_acres"]
)

acreage_scores = acreage_scores.sort_values("mae_mil_acres")

display(acreage_scores)


### Review acreage hybrid errors

compares the hybrid acreage model against the prior baseline by event.


In [ ]:
acreage_hybrid_oos = acreage_oos[
    acreage_oos["model_name"] == "stage_specific_hybrid"
].copy()

hybrid_error_review = acreage_hybrid_oos.copy()
hybrid_error_review["baseline_forecast_acres_mil"] = hybrid_error_review["acreage_prior_mil"]

hybrid_error_review["baseline_error_mil"] = (
    hybrid_error_review["baseline_forecast_acres_mil"]
    - hybrid_error_review["final_planted_acres_mil"]
)

hybrid_error_review["hybrid_error_mil"] = (
    hybrid_error_review["forecast_acres_mil"]
    - hybrid_error_review["final_planted_acres_mil"]
)

hybrid_error_review["baseline_abs_error_mil"] = hybrid_error_review["baseline_error_mil"].abs()
hybrid_error_review["hybrid_abs_error_mil"] = hybrid_error_review["hybrid_error_mil"].abs()

hybrid_error_review["hybrid_abs_error_improvement_mil"] = (
    hybrid_error_review["baseline_abs_error_mil"]
    - hybrid_error_review["hybrid_abs_error_mil"]
)

display(
    hybrid_error_review[
        [
            "asof_date",
            "acreage_crop_year",
            "acreage_stage",
            "final_planted_acres_mil",
            "acreage_prior_mil",
            "forecast_acres_mil",
            "baseline_error_mil",
            "hybrid_error_mil",
            "hybrid_abs_error_improvement_mil",
        ]
    ].sort_values("hybrid_abs_error_improvement_mil")
)


### Review acreage hybrid errors

compares the hybrid acreage model against the prior baseline by event.


In [ ]:
hybrid_year_review = (
    hybrid_error_review
    .groupby("acreage_crop_year")
    .agg(
        rows=("acreage_stage", "count"),
        baseline_mae=("baseline_abs_error_mil", "mean"),
        hybrid_mae=("hybrid_abs_error_mil", "mean"),
        net_improvement=("hybrid_abs_error_improvement_mil", "sum"),
    )
    .reset_index()
)

hybrid_year_review["mae_improvement"] = (
    hybrid_year_review["baseline_mae"]
    - hybrid_year_review["hybrid_mae"]
)

display(hybrid_year_review.sort_values("net_improvement"))


### Review output

Preview the current table before continuing.


In [ ]:
analog_features = [
    "acreage_prior_mil",
    "prior_vs_lag1_mil",
    "prior_vs_lag3_mil",
    "price_pressure",
    "stocks_tightness_pressure",
    "positioning_pressure",
    "spring_wetness_pressure",
    "planting_progress_pressure",
    "drought_pressure",
]

analog_features = [
    column for column in analog_features
    if column in acreage_model_data.columns
]

display(analog_features)


### Build acreage analog model

adds the analog acreage adjustment model from the second notebook.


In [ ]:
def analog_adjustment(train_data, test_data, k=3, shrink=0.50):
    predictions = []

    for _, test_row in test_data.iterrows():
        stage = test_row["acreage_stage"]
        stage_train = train_data[train_data["acreage_stage"] == stage].copy()

        if len(stage_train) < 2:
            predictions.append(0.0)
            continue

        feature_median = stage_train[analog_features].median()
        feature_iqr = (
            stage_train[analog_features].quantile(0.75)
            - stage_train[analog_features].quantile(0.25)
        )

        feature_iqr = feature_iqr.replace(0, 1.0).fillna(1.0)

        train_x = (stage_train[analog_features] - feature_median) / feature_iqr
        test_x = (test_row[analog_features] - feature_median) / feature_iqr

        train_x = train_x.fillna(0.0)
        test_x = test_x.fillna(0.0)

        distances = ((train_x - test_x) ** 2).sum(axis=1) ** 0.5
        neighbors = stage_train.loc[distances.nsmallest(k).index].copy()
        neighbor_distances = distances.loc[neighbors.index]

        weights = 1 / (neighbor_distances + 0.25)
        raw_adjustment = np.average(
            neighbors[acreage_model_target],
            weights=weights,
        )

        predictions.append(shrink * raw_adjustment)

    return np.array(predictions)


### Build acreage analog model

adds the analog acreage adjustment model from the second notebook.


In [ ]:
analog_rows = []
crop_years = sorted(acreage_model_data["acreage_crop_year"].unique())

analog_settings = [
    {"model_name": "analog_k2_shrink25", "k": 2, "shrink": 0.25},
    {"model_name": "analog_k2_shrink50", "k": 2, "shrink": 0.50},
    {"model_name": "analog_k3_shrink25", "k": 3, "shrink": 0.25},
    {"model_name": "analog_k3_shrink50", "k": 3, "shrink": 0.50},
    {"model_name": "analog_k4_shrink25", "k": 4, "shrink": 0.25},
    {"model_name": "analog_k4_shrink50", "k": 4, "shrink": 0.50},
]

for test_year in crop_years:
    train_years = [year for year in crop_years if year < test_year]

    if len(train_years) < 4:
        continue

    train_data = acreage_model_data[
        acreage_model_data["acreage_crop_year"].isin(train_years)
    ].copy()

    test_data = acreage_model_data[
        acreage_model_data["acreage_crop_year"] == test_year
    ].copy()

    for setting in analog_settings:
        candidate_frame = test_data.copy()

        candidate_frame["model_name"] = setting["model_name"]
        candidate_frame["predicted_adjustment_mil"] = analog_adjustment(
            train_data,
            test_data,
            k=setting["k"],
            shrink=setting["shrink"],
        )

        candidate_frame["forecast_acres_mil"] = (
            candidate_frame["acreage_prior_mil"]
            + candidate_frame["predicted_adjustment_mil"]
        )

        analog_rows.append(candidate_frame)

acreage_analog_oos = pd.concat(analog_rows, ignore_index=True)


### Review output

Preview the current table before continuing.


In [ ]:
display(
    acreage_analog_oos[
        [
            "model_name",
            "asof_date",
            "acreage_crop_year",
            "acreage_stage",
            "final_planted_acres_mil",
            "acreage_prior_mil",
            "predicted_adjustment_mil",
            "forecast_acres_mil",
        ]
    ].head(20)
)


### Score acreage models

evaluates acreage models with expanding result_series of sample tests.


In [ ]:
analog_scores = []

for model_name, rows in acreage_analog_oos.groupby("model_name"):
    analog_scores.append({
        "model_name": model_name,
        "rows": len(rows),
        "mae_mil_acres": mean_absolute_error(
            rows["final_planted_acres_mil"],
            rows["forecast_acres_mil"],
        ),
        "rmse_mil_acres": np.sqrt(
            mean_squared_error(
                rows["final_planted_acres_mil"],
                rows["forecast_acres_mil"],
            )
        ),
    })

analog_scores = pd.DataFrame(analog_scores)

all_acreage_scores = pd.concat(
    [
        acreage_scores,
        analog_scores,
    ],
    ignore_index=True,
)

baseline_mae = all_acreage_scores.loc[
    all_acreage_scores["model_name"] == "no_adjustment_baseline",
    "mae_mil_acres",
].iloc[0]

baseline_rmse = all_acreage_scores.loc[
    all_acreage_scores["model_name"] == "no_adjustment_baseline",
    "rmse_mil_acres",
].iloc[0]

all_acreage_scores["mae_improvement_vs_baseline"] = (
    baseline_mae - all_acreage_scores["mae_mil_acres"]
)

all_acreage_scores["rmse_improvement_vs_baseline"] = (
    baseline_rmse - all_acreage_scores["rmse_mil_acres"]
)

all_acreage_scores = all_acreage_scores.sort_values("mae_mil_acres")


### Score acreage models

evaluates acreage models with expanding result_series of sample tests.


In [ ]:
display(all_acreage_scores)


### Score acreage models

evaluates acreage models with expanding result_series of sample tests.


In [ ]:
all_model_scores = all_acreage_scores.copy()

all_model_scores = all_model_scores.dropna(
    subset=["model_name", "mae_mil_acres", "rmse_mil_acres"]
).copy()

max_rows = all_model_scores["rows"].max()
all_model_scores = all_model_scores[all_model_scores["rows"] == max_rows].copy()

baseline_row = all_model_scores.loc[
    all_model_scores["model_name"] == "no_adjustment_baseline"
].iloc[0]

baseline_mae = baseline_row["mae_mil_acres"]
baseline_rmse = baseline_row["rmse_mil_acres"]

all_model_scores["mae_improvement_vs_baseline"] = (
    baseline_mae - all_model_scores["mae_mil_acres"]
)

all_model_scores["rmse_improvement_vs_baseline"] = (
    baseline_rmse - all_model_scores["rmse_mil_acres"]
)

all_model_scores = all_model_scores.sort_values(
    ["mae_mil_acres", "rmse_mil_acres"]
).reset_index(drop=True)

best_acreage_model_row = all_model_scores.iloc[0]
best_acreage_model_name = best_acreage_model_row["model_name"]

material_mae_threshold = 0.10
display(all_model_scores)


### Select the acreage model

chooses the acreage model used downstream in the state panel.


In [ ]:
if pd.notna(best_acreage_model_row["mae_improvement_vs_baseline"]):
    best_acreage_model_is_material = (
        best_acreage_model_row["mae_improvement_vs_baseline"] >= material_mae_threshold
    )
else:
    best_acreage_model_is_material = False

if best_acreage_model_name == "no_adjustment_baseline":
    selected_acreage_model_name = "no_adjustment_baseline"
    selected_acreage_model_reason = "Baseline had the lowest MAE."
elif best_acreage_model_is_material:
    selected_acreage_model_name = best_acreage_model_name
    selected_acreage_model_reason = "Model beat baseline by the required MAE threshold."
else:
    selected_acreage_model_name = best_acreage_model_name
    selected_acreage_model_reason = (
        "Model had the lowest MAE, but improvement versus baseline is small. "
        "Use cautiously and keep the baseline as a benchmark."
    )

print(f"Best acreage model by MAE: {best_acreage_model_name}")
print(f"Selected acreage model: {selected_acreage_model_name}")
print(f"Reason: {selected_acreage_model_reason}")

display(all_model_scores)

best_acreage_model_summary = pd.DataFrame([{
    "best_model_by_mae": best_acreage_model_name,
    "selected_model": selected_acreage_model_name,
    "mae_mil_acres": best_acreage_model_row["mae_mil_acres"],
    "rmse_mil_acres": best_acreage_model_row["rmse_mil_acres"],
    "mae_improvement_vs_baseline": best_acreage_model_row["mae_improvement_vs_baseline"],
    "rmse_improvement_vs_baseline": best_acreage_model_row["rmse_improvement_vs_baseline"],
    "material_mae_threshold": material_mae_threshold,
    "material_improvement": best_acreage_model_is_material,
}])

display(best_acreage_model_summary)


### Plot acreage model scores

visualizes acreage model error metrics.


In [ ]:
plot_model_scores(
    all_model_scores,
    model_col="model_name",
    metric_cols=["mae_mil_acres", "rmse_mil_acres"],
    title="Acreage model scorecard",
)


## Harvested ratio forecast module

Estimate harvested acreage from planted acreage and harvested ratio history.


### Build harvested ratio history

builds the harvested ratio history from planted and harvested acres.


In [ ]:
nass_harvest_history = pd.DataFrame([
    {"acreage_crop_year": 2014, "planted_acres_mil": 90.597, "harvested_for_grain_mil": 83.146, "harvest_source": "nass_track_records_apr_2025"},
    {"acreage_crop_year": 2015, "planted_acres_mil": 88.019, "harvested_for_grain_mil": 80.753, "harvest_source": "nass_track_records_apr_2025"},
    {"acreage_crop_year": 2016, "planted_acres_mil": 94.004, "harvested_for_grain_mil": 86.748, "harvest_source": "nass_track_records_apr_2025"},
    {"acreage_crop_year": 2017, "planted_acres_mil": 90.167, "harvested_for_grain_mil": 82.733, "harvest_source": "nass_track_records_apr_2025"},
    {"acreage_crop_year": 2018, "planted_acres_mil": 88.821, "harvested_for_grain_mil": 81.169, "harvest_source": "nass_track_records_apr_2025"},
    {"acreage_crop_year": 2019, "planted_acres_mil": 89.370, "harvested_for_grain_mil": 80.991, "harvest_source": "nass_track_records_apr_2025"},
    {"acreage_crop_year": 2020, "planted_acres_mil": 90.432, "harvested_for_grain_mil": 82.168, "harvest_source": "nass_track_records_apr_2025"},
    {"acreage_crop_year": 2021, "planted_acres_mil": 92.901, "harvested_for_grain_mil": 84.988, "harvest_source": "nass_track_records_apr_2025"},
    {"acreage_crop_year": 2022, "planted_acres_mil": 88.162, "harvested_for_grain_mil": 78.705, "harvest_source": "nass_track_records_apr_2025"},
    {"acreage_crop_year": 2023, "planted_acres_mil": 94.641, "harvested_for_grain_mil": 86.506, "harvest_source": "nass_track_records_apr_2025_latest_revision"},
    {"acreage_crop_year": 2024, "planted_acres_mil": 90.909, "harvested_for_grain_mil": 83.046, "harvest_source": "nass_crop_production_2025_summary"},
    {"acreage_crop_year": 2025, "planted_acres_mil": 98.788, "harvested_for_grain_mil": 91.258, "harvest_source": "nass_crop_production_2025_summary"},
    {"acreage_crop_year": 2026, "planted_acres_mil": np.nan, "harvested_for_grain_mil": np.nan, "harvest_source": "live_crop_year_no_harvested_acres_yet"},
])

nass_harvest_history["harvest_ratio"] = (
    nass_harvest_history["harvested_for_grain_mil"]
    / nass_harvest_history["planted_acres_mil"]
)

nass_harvest_history["harvest_ratio_valid"] = (
    nass_harvest_history["harvest_ratio"].between(0.75, 1.00)
)

display(nass_harvest_history)


### Build harvested ratio history

builds the harvested ratio history from planted and harvested acres.


In [ ]:
annual_harvest_features = (
    acreage_df
    .groupby("acreage_crop_year", as_index=False)
    .agg(
        price_pressure=("price_pressure", "first"),
        stocks_tightness_pressure=("stocks_tightness_pressure", "first"),
        positioning_pressure=("positioning_pressure", "first"),
        spring_wetness_pressure=("spring_wetness_pressure", "first"),
        planting_progress_pressure=("planting_progress_pressure", "first"),
        drought_pressure=("drought_pressure", "first"),
    )
)

harvest_model_data = nass_harvest_history.merge(
    annual_harvest_features,
    on="acreage_crop_year",
    how="left",
)

harvest_model_data = harvest_model_data[
    harvest_model_data["harvest_ratio_valid"]
].copy()

harvest_model_data = harvest_model_data.dropna(
    subset=["harvest_ratio", "planted_acres_mil"]
)

display(harvest_model_data)


### Review output

Preview the current table before continuing.


In [ ]:
harvest_feature_cols = [
    "planted_acres_mil",
    "price_pressure",
    "stocks_tightness_pressure",
    "positioning_pressure",
    "spring_wetness_pressure",
    "planting_progress_pressure",
    "drought_pressure",
]

harvest_feature_cols = [
    col for col in harvest_feature_cols
    if col in harvest_model_data.columns
]

harvest_target_col = "harvest_ratio"

display(harvest_feature_cols)


### Transform data

Prepare the next modeling table used by the workflow.


In [ ]:
from sklearn.linear_model import Ridge
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.impute import SimpleImputer
from sklearn.metrics import mean_absolute_error, mean_squared_error

def rmse(actual, predicted):
    return np.sqrt(mean_squared_error(actual, predicted))

harvest_oos_rows = []

crop_years = sorted(harvest_model_data["acreage_crop_year"].unique())

for test_year in crop_years:
    train_years = [year for year in crop_years if year < test_year]

    if len(train_years) < 5:
        continue

    train_data = harvest_model_data[
        harvest_model_data["acreage_crop_year"].isin(train_years)
    ].copy()

    test_data = harvest_model_data[
        harvest_model_data["acreage_crop_year"] == test_year
    ].copy()

    baseline_ratio = train_data["harvest_ratio"].median()

    model = Pipeline([
        ("imputer", SimpleImputer(strategy="median")),
        ("scaler", StandardScaler()),
        ("model", Ridge(alpha=10.0)),
    ])

    model.fit(
        train_data[harvest_feature_cols],
        train_data[harvest_target_col],
    )

    test_data["baseline_harvest_ratio"] = baseline_ratio
    test_data["ridge_harvest_ratio"] = model.predict(
        test_data[harvest_feature_cols]
    )

    test_data["ridge_harvest_ratio"] = test_data["ridge_harvest_ratio"].clip(0.75, 1.00)

    harvest_oos_rows.append(test_data)


### Review output

Preview the current table before continuing.


In [ ]:
harvest_oos = pd.concat(harvest_oos_rows, ignore_index=True)

display(
    harvest_oos[
        [
            "acreage_crop_year",
            "planted_acres_mil",
            "harvested_for_grain_mil",
            "harvest_ratio",
            "baseline_harvest_ratio",
            "ridge_harvest_ratio",
        ]
    ]
)


### Score harvested ratio models

evaluates harvested ratio model candidates.


In [ ]:
harvest_scores = []

for model_name, pred_col in {
    "historical_median_baseline": "baseline_harvest_ratio",
    "ridge_weather_adjustment": "ridge_harvest_ratio",
}.items():
    mae = mean_absolute_error(
        harvest_oos["harvest_ratio"],
        harvest_oos[pred_col],
    )

    score_rmse = rmse(
        harvest_oos["harvest_ratio"],
        harvest_oos[pred_col],
    )

    avg_planted_acres = harvest_oos["planted_acres_mil"].mean()

    harvest_scores.append({
        "model_name": model_name,
        "rows": len(harvest_oos),
        "mae_ratio_points": mae,
        "rmse_ratio_points": score_rmse,
        "mae_mil_acres_equivalent": mae * avg_planted_acres,
        "rmse_mil_acres_equivalent": score_rmse * avg_planted_acres,
    })

harvest_scores = pd.DataFrame(harvest_scores)

baseline_mae = harvest_scores.loc[
    harvest_scores["model_name"] == "historical_median_baseline",
    "mae_ratio_points",
].iloc[0]

baseline_rmse = harvest_scores.loc[
    harvest_scores["model_name"] == "historical_median_baseline",
    "rmse_ratio_points",
].iloc[0]

harvest_scores["mae_improvement_vs_baseline"] = (
    baseline_mae - harvest_scores["mae_ratio_points"]
)

harvest_scores["rmse_improvement_vs_baseline"] = (
    baseline_rmse - harvest_scores["rmse_ratio_points"]
)

harvest_scores = harvest_scores.sort_values("mae_ratio_points")


### Score harvested ratio models

evaluates harvested ratio model candidates.


In [ ]:
display(harvest_scores)


### Score harvested ratio models

evaluates harvested ratio model candidates.


In [ ]:
plot_model_scores(
    harvest_scores,
    model_col="model_name",
    metric_cols=["mae_ratio", "rmse_ratio"],
    title="Harvested ratio model scorecard",
)


### Score harvested ratio models

evaluates harvested ratio model candidates.


In [ ]:
best_harvest_row = harvest_scores.iloc[0].copy()

selected_harvest_model = best_harvest_row["model_name"]

harvest_model_selection = pd.DataFrame([{
    "selected_harvest_model": selected_harvest_model,
    "mae_ratio_points": best_harvest_row["mae_ratio_points"],
    "rmse_ratio_points": best_harvest_row["rmse_ratio_points"],
    "mae_mil_acres_equivalent": best_harvest_row["mae_mil_acres_equivalent"],
    "rmse_mil_acres_equivalent": best_harvest_row["rmse_mil_acres_equivalent"],
    "mae_improvement_vs_baseline": best_harvest_row["mae_improvement_vs_baseline"],
    "rmse_improvement_vs_baseline": best_harvest_row["rmse_improvement_vs_baseline"],
    "oos_rows": int(best_harvest_row["rows"]),
    "confidence": "medium_low_small_sample",
}])

display(harvest_model_selection)


### Transform data

Prepare the next modeling table used by the workflow.


In [ ]:
final_harvest_model = Pipeline([
    ("imputer", SimpleImputer(strategy="median")),
    ("scaler", StandardScaler()),
    ("model", Ridge(alpha=10.0)),
])

final_harvest_model.fit(
    harvest_model_data[harvest_feature_cols],
    harvest_model_data[harvest_target_col],
)

print("Final harvested-ratio model fitted")


### Forecast harvested acreage

combines the selected acreage forecast with the harvested ratio forecast.


In [ ]:
latest_live_acreage = acreage_live.sort_values("asof_date").tail(1)
latest_acreage_forecast_mil = latest_live_acreage["acreage_prior_mil"].iloc[0]
latest_crop_year = int(acreage_live["acreage_crop_year"].max())

latest_harvest_features = annual_harvest_features[
    annual_harvest_features["acreage_crop_year"] == latest_crop_year
].copy()

latest_harvest_features["planted_acres_mil"] = latest_acreage_forecast_mil

if selected_harvest_model == "ridge_weather_adjustment":
    latest_harvest_ratio = final_harvest_model.predict(
        latest_harvest_features[harvest_feature_cols]
    )[0]
else:
    latest_harvest_ratio = harvest_model_data["harvest_ratio"].median()

latest_harvest_ratio = float(np.clip(latest_harvest_ratio, 0.80, 0.95))

latest_harvest_forecast = pd.DataFrame([{
    "crop_year": latest_crop_year,
    "planted_acres_forecast_mil": latest_acreage_forecast_mil,
    "harvest_ratio_forecast": latest_harvest_ratio,
    "harvested_acres_forecast_mil": latest_acreage_forecast_mil * latest_harvest_ratio,
    "selected_harvest_model": selected_harvest_model,
}])

display(latest_harvest_forecast)


## Yield forecast module

Build annual and daily yield models while preserving the expanding result_series of sample structure.


### Build yield history

builds the yield training history.


In [ ]:
nass_yield_history = pd.DataFrame([
    {"crop_year": 2014, "final_yield_bu_acre": 171.0},
    {"crop_year": 2015, "final_yield_bu_acre": 168.4},
    {"crop_year": 2016, "final_yield_bu_acre": 174.6},
    {"crop_year": 2017, "final_yield_bu_acre": 176.6},
    {"crop_year": 2018, "final_yield_bu_acre": 176.4},
    {"crop_year": 2019, "final_yield_bu_acre": 167.5},
    {"crop_year": 2020, "final_yield_bu_acre": 171.4},
    {"crop_year": 2021, "final_yield_bu_acre": 176.7},
    {"crop_year": 2022, "final_yield_bu_acre": 173.4},
    {"crop_year": 2023, "final_yield_bu_acre": 177.3},
    {"crop_year": 2024, "final_yield_bu_acre": 179.3},
    {"crop_year": 2025, "final_yield_bu_acre": 186.5},
    {"crop_year": 2026, "final_yield_bu_acre": np.nan},
])

display(nass_yield_history)


### Build yield history

builds the yield training history.


In [ ]:
yield_df = modeling_panel.copy()

yield_df["asof_date"] = pd.to_datetime(yield_df["asof_date"], errors="coerce")
yield_df["crop_year"] = yield_df["asof_date"].dt.year

yield_df = yield_df[
    (yield_df["asof_date"] >= "2014-01-01")
    & (yield_df["asof_date"].dt.month <= 8)
].copy()

yield_df = yield_df.merge(
    nass_yield_history,
    on="crop_year",
    how="left",
)

print(f"Rows: {yield_df.shape[0]:,}")
print(f"Crop years: {yield_df['crop_year'].min()} to {yield_df['crop_year'].max()}")


### Define mean_or_nan

defines `mean_or_nan` so later cells can reuse the same transformation without repeating code.


In [ ]:
def mean_or_nan(series):
    series = pd.Series(series).dropna()
    return np.nan if series.empty else float(series.mean())

def max_or_nan(series):
    series = pd.Series(series).dropna()
    return np.nan if series.empty else float(series.max())

def last_or_nan(series):
    series = pd.Series(series).dropna()
    return np.nan if series.empty else float(series.iloc[-1])

annual_yield_rows = []


### Transform data

Prepare the next modeling table used by the workflow.


In [ ]:
for crop_year in sorted(yield_df["crop_year"].dropna().unique()):
    crop_year = int(crop_year)

    rows = yield_df[yield_df["crop_year"] == crop_year].sort_values("asof_date")

    may_jun = rows[rows["asof_date"].dt.month.isin([5, 6])]
    jul_aug = rows[rows["asof_date"].dt.month.isin([7, 8])]
    jun_aug = rows[rows["asof_date"].dt.month.isin([6, 7, 8])]

    annual_yield_rows.append({
        "crop_year": crop_year,
        "final_yield_bu_acre": rows["final_yield_bu_acre"].dropna().iloc[0]
            if rows["final_yield_bu_acre"].notna().any()
            else np.nan,

        "crop_condition_jul_aug_mean": mean_or_nan(jul_aug["crop_condition"]),
        "crop_condition_latest": last_or_nan(rows["crop_condition"]),

        "crop_progress_may_jun_mean": mean_or_nan(may_jun["crop_progress"]),
        "crop_progress_latest": last_or_nan(rows["crop_progress"]),

        "gdd_jun_aug_mean": mean_or_nan(jun_aug["gdd_corn_f"]),
        "gdd_latest": last_or_nan(rows["gdd_corn_f"]),

        "prcp_jun_aug_sum": jun_aug["prcp"].sum(skipna=True),
        "prcp_jul_aug_sum": jul_aug["prcp"].sum(skipna=True),

        "tmax_jul_aug_mean": mean_or_nan(jul_aug["tmax"]),
        "tmin_jul_aug_mean": mean_or_nan(jul_aug["tmin"]),

        "drought_d1_jul_aug_mean": mean_or_nan(jul_aug["drought_d1_or_worse_pct_area"]),
        "drought_d2_jul_aug_mean": mean_or_nan(jul_aug["drought_d2_or_worse_pct_area"]),
        "drought_d3_jul_aug_mean": mean_or_nan(jul_aug["drought_d3_or_worse_pct_area"]),
        "drought_d1_jul_aug_max": max_or_nan(jul_aug["drought_d1_or_worse_pct_area"]),
        "drought_d2_jul_aug_max": max_or_nan(jul_aug["drought_d2_or_worse_pct_area"]),
    })

yield_model_data = pd.DataFrame(annual_yield_rows)

yield_model_data = yield_model_data.sort_values("crop_year").reset_index(drop=True)

yield_model_data["trend_year"] = (
    yield_model_data["crop_year"]
    - yield_model_data["crop_year"].min()
)

yield_model_data["lag1_yield"] = yield_model_data["final_yield_bu_acre"].shift(1)


### Review output

Preview the current table before continuing.


In [ ]:
yield_model_data["lag3_yield_avg"] = (
    yield_model_data["final_yield_bu_acre"]
    .shift(1)
    .rolling(3, min_periods=1)
    .mean()
)

display(yield_model_data)


### Review output

Preview the current table before continuing.


In [ ]:
yield_feature_cols = [
    "trend_year",
    "lag1_yield",
    "lag3_yield_avg",

    "crop_condition_jul_aug_mean",
    "crop_condition_latest",

    "crop_progress_may_jun_mean",
    "crop_progress_latest",

    "gdd_jun_aug_mean",
    "gdd_latest",

    "prcp_jun_aug_sum",
    "prcp_jul_aug_sum",

    "tmax_jul_aug_mean",
    "tmin_jul_aug_mean",

    "drought_d1_jul_aug_mean",
    "drought_d2_jul_aug_mean",
    "drought_d3_jul_aug_mean",
    "drought_d1_jul_aug_max",
    "drought_d2_jul_aug_max",
]

yield_feature_cols = [
    col for col in yield_feature_cols
    if col in yield_model_data.columns
]

yield_target_col = "final_yield_bu_acre"

display(yield_feature_cols)


### Review output

Preview the current table before continuing.


In [ ]:
yield_train_data = yield_model_data.dropna(
    subset=[yield_target_col]
).copy()

yield_live_data = yield_model_data[
    yield_model_data[yield_target_col].isna()
].copy()

print(f"Training crop years: {yield_train_data['crop_year'].min()} to {yield_train_data['crop_year'].max()}")
print(f"Training rows: {yield_train_data.shape[0]:,}")
print(f"Live rows: {yield_live_data.shape[0]:,}")

display(yield_train_data)


### Run daily yield backtest

runs the daily expanding yield backtest.


In [ ]:
from sklearn.linear_model import Ridge, LinearRegression
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.impute import SimpleImputer
from sklearn.metrics import mean_absolute_error, mean_squared_error

def rmse(actual, predicted):
    return np.sqrt(mean_squared_error(actual, predicted))

yield_oos_rows = []

crop_years = sorted(yield_train_data["crop_year"].unique())


### Run daily yield backtest

runs the daily expanding yield backtest.


In [ ]:
for test_year in crop_years:
    train_years = [year for year in crop_years if year < test_year]

    if len(train_years) < 5:
        continue

    train_data = yield_train_data[
        yield_train_data["crop_year"].isin(train_years)
    ].copy()

    test_data = yield_train_data[
        yield_train_data["crop_year"] == test_year
    ].copy()

    trend_model = Pipeline([
        ("imputer", SimpleImputer(strategy="median")),
        ("model", LinearRegression()),
    ])

    ridge_model = Pipeline([
        ("imputer", SimpleImputer(strategy="median")),
        ("scaler", StandardScaler()),
        ("model", Ridge(alpha=10.0)),
    ])

    trend_features = ["trend_year", "lag3_yield_avg"]

    trend_model.fit(
        train_data[trend_features],
        train_data[yield_target_col],
    )

    ridge_model.fit(
        train_data[yield_feature_cols],
        train_data[yield_target_col],
    )

    test_data["trend_yield_forecast"] = trend_model.predict(
        test_data[trend_features]
    )

    test_data["ridge_yield_forecast"] = ridge_model.predict(
        test_data[yield_feature_cols]
    )

    test_data["lag3_baseline_yield_forecast"] = train_data["final_yield_bu_acre"].tail(3).mean()

    yield_oos_rows.append(test_data)


### Run daily yield backtest

runs the daily expanding yield backtest.


In [ ]:
yield_oos = pd.concat(yield_oos_rows, ignore_index=True)

display(
    yield_oos[
        [
            "crop_year",
            "final_yield_bu_acre",
            "lag3_baseline_yield_forecast",
            "trend_yield_forecast",
            "ridge_yield_forecast",
        ]
    ]
)


### Score annual yield models

evaluates annual yield forecasting candidates.


In [ ]:
yield_scores = []

forecast_cols = {
    "lag3_baseline": "lag3_baseline_yield_forecast",
    "trend_model": "trend_yield_forecast",
    "ridge_weather_model": "ridge_yield_forecast",
}

for model_name, pred_col in forecast_cols.items():
    mae = mean_absolute_error(
        yield_oos[yield_target_col],
        yield_oos[pred_col],
    )

    model_rmse = rmse(
        yield_oos[yield_target_col],
        yield_oos[pred_col],
    )

    yield_scores.append({
        "model_name": model_name,
        "rows": len(yield_oos),
        "mae_bu_acre": mae,
        "rmse_bu_acre": model_rmse,
    })

yield_scores = pd.DataFrame(yield_scores)

baseline_mae = yield_scores.loc[
    yield_scores["model_name"] == "lag3_baseline",
    "mae_bu_acre",
].iloc[0]

baseline_rmse = yield_scores.loc[
    yield_scores["model_name"] == "lag3_baseline",
    "rmse_bu_acre",
].iloc[0]

yield_scores["mae_improvement_vs_baseline"] = (
    baseline_mae - yield_scores["mae_bu_acre"]
)

yield_scores["rmse_improvement_vs_baseline"] = (
    baseline_rmse - yield_scores["rmse_bu_acre"]
)

yield_scores = yield_scores.sort_values("mae_bu_acre")


### Score annual yield models

evaluates annual yield forecasting candidates.


In [ ]:
display(yield_scores)


### Score annual yield models

evaluates annual yield forecasting candidates.


In [ ]:
plot_model_scores(
    yield_scores,
    model_col="model_name",
    metric_cols=["mae_bu_acre", "rmse_bu_acre"],
    title="Annual yield model scorecard",
)


### Build yield history

builds the yield training history.


In [ ]:
yield_prior_df = modeling_panel.copy()

yield_prior_df["asof_date"] = pd.to_datetime(yield_prior_df["asof_date"], errors="coerce")
yield_prior_df["crop_year"] = yield_prior_df["asof_date"].dt.year

yield_prior_df = yield_prior_df[
    (yield_prior_df["asof_date"] >= "2014-01-01")
    & (yield_prior_df["asof_date"].dt.month <= 8)
].copy()

yield_prior_df = yield_prior_df.merge(
    nass_yield_history,
    on="crop_year",
    how="left",
)

yield_prior_candidates = [
    "yield_bu_per_acre",
    "yield_bu_per_acre_model",
    "yield",
]

yield_prior_candidates = [
    col for col in yield_prior_candidates
    if col in yield_prior_df.columns
]

display(yield_prior_candidates)


### Review output

Preview the current table before continuing.


In [ ]:
candidate_rows = []

for col in yield_prior_candidates:
    values = yield_prior_df[col]

    plausible = values.between(100, 230)

    coverage = plausible.mean() * 100

    exact_final_match = (
        (values - yield_prior_df["final_yield_bu_acre"]).abs() <= 0.01
    )

    early_rows = yield_prior_df["asof_date"].dt.month <= 5

    early_exact_match_share = exact_final_match[early_rows].mean() * 100

    candidate_rows.append({
        "candidate_col": col,
        "plausible_coverage_pct": coverage,
        "early_exact_final_match_pct": early_exact_match_share,
        "median_value": values[plausible].median(),
        "min_value": values[plausible].min(),
        "max_value": values[plausible].max(),
    })

yield_prior_audit = pd.DataFrame(candidate_rows)

display(yield_prior_audit)


### Create rule based fields

creates rule based fields used by the forecasting workflow.


In [ ]:
usable_yield_prior_cols = yield_prior_audit[
    (yield_prior_audit["plausible_coverage_pct"] > 50)
    & (yield_prior_audit["early_exact_final_match_pct"] < 40)
]["candidate_col"].tolist()

if usable_yield_prior_cols:
    yield_prior_col = usable_yield_prior_cols[0]
else:
    yield_prior_col = None

print(f"Selected yield prior column: {yield_prior_col}")

yield_prior_df["yield_stage"] = np.select(
    [
        yield_prior_df["asof_date"].dt.month <= 4,
        yield_prior_df["asof_date"].dt.month.isin([5, 6]),
        yield_prior_df["asof_date"].dt.month == 7,
        yield_prior_df["asof_date"].dt.month == 8,
    ],
    [
        "pre_weather",
        "planting_early_weather",
        "pollination_weather",
        "late_season",
    ],
    default="unknown",
)

if yield_prior_col is not None:
    yield_prior_df["yield_prior_bu_acre"] = yield_prior_df[yield_prior_col]
    yield_prior_df["yield_prior_source"] = yield_prior_col
else:
    yield_prior_df["yield_prior_bu_acre"] = np.nan
    yield_prior_df["yield_prior_source"] = "none_usable"

yield_prior_df["yield_adjustment_target"] = (
    yield_prior_df["final_yield_bu_acre"]
    - yield_prior_df["yield_prior_bu_acre"]
)


### Review output

Preview the current table before continuing.


In [ ]:
display(
    yield_prior_df[
        [
            "asof_date",
            "crop_year",
            "yield_stage",
            "final_yield_bu_acre",
            "yield_prior_bu_acre",
            "yield_adjustment_target",
            "yield_prior_source",
        ]
    ].head(20)
)


### Review output

Preview the current table before continuing.


In [ ]:
yield_prior_train_daily = yield_prior_df[
    yield_prior_df["final_yield_bu_acre"].notna()
    & yield_prior_df["yield_prior_bu_acre"].notna()
].copy()

yield_prior_train_events = (
    yield_prior_train_daily
    .sort_values("asof_date")
    .groupby(["crop_year", "yield_stage"], as_index=False)
    .tail(1)
    .sort_values(["crop_year", "asof_date"])
    .reset_index(drop=True)
)

yield_prior_live = yield_prior_df[
    yield_prior_df["final_yield_bu_acre"].isna()
    & yield_prior_df["yield_prior_bu_acre"].notna()
].copy()

print(f"Yield-prior daily training rows: {yield_prior_train_daily.shape[0]:,}")
print(f"Yield-prior event training rows: {yield_prior_train_events.shape[0]:,}")
print(f"Yield-prior live rows: {yield_prior_live.shape[0]:,}")

display(
    yield_prior_train_events[
        [
            "asof_date",
            "crop_year",
            "yield_stage",
            "final_yield_bu_acre",
            "yield_prior_bu_acre",
            "yield_adjustment_target",
            "yield_prior_source",
        ]
    ]
)


### Define zero_yield_adjustment

defines `zero_yield_adjustment` so later cells can reuse the same transformation without repeating code.


In [ ]:
def zero_yield_adjustment(train_data, test_data):
    return np.zeros(len(test_data))


def stage_median_yield_adjustment(train_data, test_data, shrink=0.25):
    predictions = []

    for _, row in test_data.iterrows():
        stage = row["yield_stage"]

        stage_history = train_data.loc[
            train_data["yield_stage"] == stage,
            "yield_adjustment_target",
        ]

        if len(stage_history) >= 2:
            adjustment = stage_history.median()
        else:
            adjustment = train_data["yield_adjustment_target"].median()

        predictions.append(shrink * adjustment)

    return np.array(predictions)


def last_year_yield_adjustment(train_data, test_data, shrink=0.25):
    predictions = []

    for _, row in test_data.iterrows():
        last_year = row["crop_year"] - 1
        stage = row["yield_stage"]

        match = train_data[
            (train_data["crop_year"] == last_year)
            & (train_data["yield_stage"] == stage)
        ]

        if len(match) == 1:
            adjustment = match["yield_adjustment_target"].iloc[0]
        else:
            adjustment = 0.0

        predictions.append(shrink * adjustment)

    return np.array(predictions)


### Review output

Preview the current table before continuing.


In [ ]:
if len(yield_prior_train_events) == 0:
    print("No usable yield-prior training events. Keep lag3_baseline for yield.")
    yield_prior_oos = pd.DataFrame()
else:
    yield_candidate_models = {
        "prior_no_adjustment": lambda train, test: zero_yield_adjustment(train, test),
        "stage_median_shrink_10": lambda train, test: stage_median_yield_adjustment(train, test, shrink=0.10),
        "stage_median_shrink_25": lambda train, test: stage_median_yield_adjustment(train, test, shrink=0.25),
        "stage_median_shrink_50": lambda train, test: stage_median_yield_adjustment(train, test, shrink=0.50),
        "last_year_same_stage_shrink_25": lambda train, test: last_year_yield_adjustment(train, test, shrink=0.25),
    }

    rows = []

    crop_years = sorted(yield_prior_train_events["crop_year"].unique())

    for test_year in crop_years:
        train_years = [year for year in crop_years if year < test_year]

        if len(train_years) < 5:
            continue

        train_data = yield_prior_train_events[
            yield_prior_train_events["crop_year"].isin(train_years)
        ].copy()

        test_data = yield_prior_train_events[
            yield_prior_train_events["crop_year"] == test_year
        ].copy()

        for model_name, model_func in yield_candidate_models.items():
            candidate_frame = test_data.copy()

            candidate_frame["model_name"] = model_name
            candidate_frame["predicted_yield_adjustment"] = model_func(train_data, test_data)
            candidate_frame["yield_forecast_bu_acre"] = (
                candidate_frame["yield_prior_bu_acre"]
                + candidate_frame["predicted_yield_adjustment"]
            )

            rows.append(candidate_frame)

    yield_prior_oos = pd.concat(rows, ignore_index=True)

    display(
        yield_prior_oos[
            [
                "model_name",
                "asof_date",
                "crop_year",
                "yield_stage",
                "final_yield_bu_acre",
                "yield_prior_bu_acre",
                "predicted_yield_adjustment",
                "yield_forecast_bu_acre",
            ]
        ].head(20)
    )


### Review output

Preview the current table before continuing.


In [ ]:
if len(yield_prior_oos) == 0:
    yield_prior_scores = pd.DataFrame()
    print("No yield-prior OOS score available.")
else:
    yield_prior_scores = []

    for model_name, rows in yield_prior_oos.groupby("model_name"):
        mae = mean_absolute_error(
            rows["final_yield_bu_acre"],
            rows["yield_forecast_bu_acre"],
        )

        model_rmse = rmse(
            rows["final_yield_bu_acre"],
            rows["yield_forecast_bu_acre"],
        )

        yield_prior_scores.append({
            "model_name": model_name,
            "rows": len(rows),
            "mae_bu_acre": mae,
            "rmse_bu_acre": model_rmse,
        })

    yield_prior_scores = pd.DataFrame(yield_prior_scores)

    baseline_mae = yield_prior_scores.loc[
        yield_prior_scores["model_name"] == "prior_no_adjustment",
        "mae_bu_acre",
    ].iloc[0]

    baseline_rmse = yield_prior_scores.loc[
        yield_prior_scores["model_name"] == "prior_no_adjustment",
        "rmse_bu_acre",
    ].iloc[0]

    yield_prior_scores["mae_improvement_vs_prior"] = (
        baseline_mae - yield_prior_scores["mae_bu_acre"]
    )

    yield_prior_scores["rmse_improvement_vs_prior"] = (
        baseline_rmse - yield_prior_scores["rmse_bu_acre"]
    )

    yield_prior_scores = yield_prior_scores.sort_values("mae_bu_acre")

    display(yield_prior_scores)


### Build yield history

builds the yield training history.


In [ ]:
yield_daily = modeling_panel.copy()

yield_daily["asof_date"] = pd.to_datetime(yield_daily["asof_date"], errors="coerce")
yield_daily["crop_year"] = yield_daily["asof_date"].dt.year

yield_daily = yield_daily[
    (yield_daily["asof_date"] >= "2014-01-01")
    & (yield_daily["asof_date"].dt.month <= 8)
].copy()

yield_daily = yield_daily.merge(
    nass_yield_history,
    on="crop_year",
    how="left",
)

yield_daily["yield_stage"] = np.select(
    [
        yield_daily["asof_date"].dt.month <= 4,
        yield_daily["asof_date"].dt.month.isin([5, 6]),
        yield_daily["asof_date"].dt.month == 7,
        yield_daily["asof_date"].dt.month == 8,
    ],
    [
        "pre_weather",
        "planting_early_weather",
        "pollination_weather",
        "late_season",
    ],
    default="unknown",
)

yield_daily["month"] = yield_daily["asof_date"].dt.month
yield_daily["dayofyear"] = yield_daily["asof_date"].dt.dayofyear

print(f"Rows: {yield_daily.shape[0]:,}")
print(f"Crop years: {yield_daily['crop_year'].min()} to {yield_daily['crop_year'].max()}")


### Review output

Preview the current table before continuing.


In [ ]:
display(
    yield_daily[
        [
            "asof_date",
            "crop_year",
            "yield_stage",
            "final_yield_bu_acre",
            "crop_condition",
            "crop_progress",
            "gdd_corn_f",
            "prcp",
            "tmax",
            "tmin",
        ]
    ].head()
)


### Summarize grouped data

aggregates the panel into a compact summary used by the following modelling step.


In [ ]:
yield_daily = yield_daily.sort_values(["crop_year", "asof_date"]).copy()

def crop_year_expanding_mean(series):
    return series.expanding(min_periods=3).mean()

def crop_year_expanding_sum(series):
    return series.expanding(min_periods=3).sum()

def crop_year_rolling_mean(series, window):
    return series.rolling(window, min_periods=max(3, window // 3)).mean()

def crop_year_rolling_sum(series, window):
    return series.rolling(window, min_periods=max(3, window // 3)).sum()

feature_source_cols = [
    "crop_condition",
    "crop_progress",
    "gdd_corn_f",
    "prcp",
    "tmax",
    "tmin",
    "drought_d0_or_worse_pct_area",
    "drought_d1_or_worse_pct_area",
    "drought_d2_or_worse_pct_area",
    "drought_d3_or_worse_pct_area",
]

for col in feature_source_cols:
    if col not in yield_daily.columns:
        continue

    yield_daily[f"{col}_exp_mean"] = (
        yield_daily
        .groupby("crop_year")[col]
        .transform(crop_year_expanding_mean)
    )

    yield_daily[f"{col}_20d_mean"] = (
        yield_daily
        .groupby("crop_year")[col]
        .transform(lambda s: crop_year_rolling_mean(s, 20))
    )


### Summarize grouped data

aggregates the panel into a compact summary used by the following modelling step.


In [ ]:
if "prcp" in yield_daily.columns:
    yield_daily["prcp_exp_sum"] = (
        yield_daily
        .groupby("crop_year")["prcp"]
        .transform(crop_year_expanding_sum)
    )

    yield_daily["prcp_20d_sum"] = (
        yield_daily
        .groupby("crop_year")["prcp"]
        .transform(lambda s: crop_year_rolling_sum(s, 20))
    )

if "tmax" in yield_daily.columns:
    yield_daily["heat_stress_days_20d"] = (
        yield_daily
        .assign(heat_stress_day=(yield_daily["tmax"] >= 90).astype(float))
        .groupby("crop_year")["heat_stress_day"]
        .transform(lambda s: crop_year_rolling_sum(s, 20))
    )

print("Daily yield features created")


### Transform data

Prepare the next modeling table used by the workflow.


In [ ]:
yield_daily_features = [
    "month",
    "dayofyear",
    "yield_stage",
    "seasonal_regime",

    "crop_condition",
    "crop_condition_exp_mean",
    "crop_condition_20d_mean",

    "crop_progress",
    "crop_progress_exp_mean",
    "crop_progress_20d_mean",

    "gdd_corn_f",
    "gdd_corn_f_exp_mean",
    "gdd_corn_f_20d_mean",

    "prcp",
    "prcp_exp_mean",
    "prcp_20d_mean",
    "prcp_exp_sum",
    "prcp_20d_sum",

    "tmax",
    "tmax_exp_mean",
    "tmax_20d_mean",
    "tmin",
    "tmin_exp_mean",
    "tmin_20d_mean",

    "heat_stress_days_20d",

    "drought_d0_or_worse_pct_area",
    "drought_d0_or_worse_pct_area_exp_mean",
    "drought_d0_or_worse_pct_area_20d_mean",

    "drought_d1_or_worse_pct_area",
    "drought_d1_or_worse_pct_area_exp_mean",
    "drought_d1_or_worse_pct_area_20d_mean",

    "drought_d2_or_worse_pct_area",
    "drought_d2_or_worse_pct_area_exp_mean",
    "drought_d2_or_worse_pct_area_20d_mean",

    "drought_d3_or_worse_pct_area",
    "drought_d3_or_worse_pct_area_exp_mean",
    "drought_d3_or_worse_pct_area_20d_mean",
]


### Review output

Preview the current table before continuing.


In [ ]:
yield_daily_features = [
    col for col in yield_daily_features
    if col in yield_daily.columns
]

display(yield_daily_features)


### Transform data

Prepare the next modeling table used by the workflow.


In [ ]:
from sklearn.linear_model import LinearRegression

def fit_trend_from_prior_years(history):
    history = history.dropna(subset=["crop_year", "final_yield_bu_acre"]).copy()

    if len(history) < 4:
        return None

    x = history[["crop_year"]]
    y = history["final_yield_bu_acre"]

    model = LinearRegression()
    model.fit(x, y)

    return model


def predict_trend_yield(trend_model, crop_year):
    if trend_model is None:
        return np.nan

    return float(
        trend_model.predict(
            pd.DataFrame({"crop_year": [crop_year]})
        )[0]
    )


### Transform data

Prepare the next modeling table used by the workflow.


In [ ]:
from sklearn.compose import ColumnTransformer
from sklearn.ensemble import RandomForestRegressor, HistGradientBoostingRegressor
from sklearn.impute import SimpleImputer
from sklearn.linear_model import HuberRegressor, Ridge
from sklearn.metrics import mean_absolute_error, mean_squared_error
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler

def rmse(actual, predicted):
    return np.sqrt(mean_squared_error(actual, predicted))

one_hot = OneHotEncoder(handle_unknown="ignore", sparse_output=False)

categorical_yield_features = [
    column for column in yield_daily_features
    if yield_daily[column].dtype == "object"
]

numeric_yield_features = [
    column for column in yield_daily_features
    if column not in categorical_yield_features
]

numeric_preprocess = Pipeline([
    ("imputer", SimpleImputer(strategy="median")),
    ("scaler", StandardScaler()),
])

categorical_preprocess = Pipeline([
    ("imputer", SimpleImputer(strategy="most_frequent")),
    ("one_hot", one_hot),
])

yield_preprocessor = ColumnTransformer([
    ("numeric", numeric_preprocess, numeric_yield_features),
    ("categorical", categorical_preprocess, categorical_yield_features),
])


### Run daily yield backtest

runs the daily expanding yield backtest.


In [ ]:
yield_models = {
    "ridge_anomaly": Pipeline([
        ("preprocessor", yield_preprocessor),
        ("model", Ridge(alpha=20.0)),
    ]),
    "huber_anomaly": Pipeline([
        ("preprocessor", yield_preprocessor),
        ("model", HuberRegressor(alpha=0.01, max_iter=1000)),
    ]),
    "random_forest_anomaly": Pipeline([
        ("preprocessor", yield_preprocessor),
        ("model", RandomForestRegressor(
            n_estimators=300,
            max_depth=3,
            min_samples_leaf=20,
            random_state=42,
        )),
    ]),
}

yield_train_daily = yield_daily[
    yield_daily["final_yield_bu_acre"].notna()
].copy()

yield_live_daily = yield_daily[
    yield_daily["final_yield_bu_acre"].isna()
].copy()

yield_oos_rows = []

crop_years = sorted(yield_train_daily["crop_year"].unique())


### Build yield history

builds the yield training history.


In [ ]:
def build_train_year_trends(train_years):
    trend_values = {}

    for year in train_years:
        prior_years_for_trend = [y for y in train_years if y < year]

        trend_for_year = fit_trend_from_prior_years(
            nass_yield_history[
                nass_yield_history["crop_year"].isin(prior_years_for_trend)
            ]
        )

        trend_value = predict_trend_yield(trend_for_year, year)

        if pd.isna(trend_value):
            trend_value = nass_yield_history[
                nass_yield_history["crop_year"].isin(prior_years_for_trend)
            ]["final_yield_bu_acre"].tail(3).mean()

        trend_values[year] = trend_value

    return trend_values


def get_yield_event_rows(test_data_all):
    return (
        test_data_all
        .sort_values("asof_date")
        .groupby("yield_stage", as_index=False)
        .tail(1)
        .sort_values("asof_date")
        .reset_index(drop=True)
    )


### Build yield history

builds the yield training history.


In [ ]:
def build_yield_oos_rows_for_year(test_year, train_years):
    train_data = yield_train_daily[
        yield_train_daily["crop_year"].isin(train_years)
    ].copy()

    test_data_all = yield_train_daily[
        yield_train_daily["crop_year"] == test_year
    ].copy()

    trend_model = fit_trend_from_prior_years(
        nass_yield_history[
            nass_yield_history["crop_year"].isin(train_years)
        ]
    )

    test_trend_yield = predict_trend_yield(trend_model, test_year)

    train_data = train_data.merge(
        nass_yield_history[["crop_year", "final_yield_bu_acre"]],
        on="crop_year",
        how="left",
        suffixes=("", "_check"),
    )

    train_data["trend_yield_asof"] = train_data["crop_year"].map(
        build_train_year_trends(train_years)
    )

    train_data["yield_anomaly_target"] = (
        train_data["final_yield_bu_acre"]
        - train_data["trend_yield_asof"]
    )

    train_data = train_data.dropna(subset=["yield_anomaly_target"])

    event_test_rows = get_yield_event_rows(test_data_all)

    lag3_baseline = nass_yield_history[
        nass_yield_history["crop_year"].isin(train_years)
    ]["final_yield_bu_acre"].tail(3).mean()

    rows = []

    for model_name, model in yield_models.items():
        model.fit(train_data[yield_daily_features], train_data["yield_anomaly_target"])

        candidate_frame = event_test_rows.copy()
        candidate_frame["model_name"] = model_name
        candidate_frame["trend_yield_forecast"] = test_trend_yield
        candidate_frame["predicted_yield_anomaly"] = model.predict(candidate_frame[yield_daily_features])
        candidate_frame["yield_forecast_bu_acre"] = candidate_frame["trend_yield_forecast"] + candidate_frame["predicted_yield_anomaly"]
        candidate_frame["lag3_baseline_yield_forecast"] = lag3_baseline

        rows.append(candidate_frame)

    baseline_temp = event_test_rows.copy()
    baseline_temp["model_name"] = "lag3_baseline"
    baseline_temp["trend_yield_forecast"] = test_trend_yield
    baseline_temp["predicted_yield_anomaly"] = 0.0
    baseline_temp["yield_forecast_bu_acre"] = lag3_baseline
    baseline_temp["lag3_baseline_yield_forecast"] = lag3_baseline

    rows.append(baseline_temp)

    return rows


### Run daily yield backtest

runs the daily expanding yield backtest.


In [ ]:
yield_oos_rows = []

for test_year in crop_years:
    train_years = [year for year in crop_years if year < test_year]

    if len(train_years) < 5:
        continue

    yield_oos_rows.extend(
        build_yield_oos_rows_for_year(test_year, train_years)
    )

print("Yield result_series of sample row groups:", len(yield_oos_rows))


### Run daily yield backtest

runs the daily expanding yield backtest.


In [ ]:
yield_daily_oos = pd.concat(yield_oos_rows, ignore_index=True)

display(
    yield_daily_oos[
        [
            "model_name",
            "asof_date",
            "crop_year",
            "yield_stage",
            "final_yield_bu_acre",
            "trend_yield_forecast",
            "predicted_yield_anomaly",
            "yield_forecast_bu_acre",
            "lag3_baseline_yield_forecast",
        ]
    ].head(40)
)


### Review output

Preview the current table before continuing.


In [ ]:
yield_daily_scores = []

for model_name, rows in yield_daily_oos.groupby("model_name"):
    mae = mean_absolute_error(
        rows["final_yield_bu_acre"],
        rows["yield_forecast_bu_acre"],
    )

    score_rmse = rmse(
        rows["final_yield_bu_acre"],
        rows["yield_forecast_bu_acre"],
    )

    yield_daily_scores.append({
        "model_name": model_name,
        "rows": len(rows),
        "mae_bu_acre": mae,
        "rmse_bu_acre": score_rmse,
    })

yield_daily_scores = pd.DataFrame(yield_daily_scores)

baseline_mae = yield_daily_scores.loc[
    yield_daily_scores["model_name"] == "lag3_baseline",
    "mae_bu_acre",
].iloc[0]

baseline_rmse = yield_daily_scores.loc[
    yield_daily_scores["model_name"] == "lag3_baseline",
    "rmse_bu_acre",
].iloc[0]

yield_daily_scores["mae_improvement_vs_baseline"] = (
    baseline_mae - yield_daily_scores["mae_bu_acre"]
)

yield_daily_scores["rmse_improvement_vs_baseline"] = (
    baseline_rmse - yield_daily_scores["rmse_bu_acre"]
)

yield_daily_scores = yield_daily_scores.sort_values("mae_bu_acre")

display(yield_daily_scores)


### Plot daily yield model scores

visualizes the daily yield model scorecard.


In [ ]:
plot_model_scores(
    yield_daily_scores,
    model_col="model_name",
    metric_cols=["mae_bu_acre", "rmse_bu_acre"],
    title="Daily yield model scorecard",
)


### Review output

Preview the current table before continuing.


In [ ]:
yield_stage_scores = []

for (model_name, stage), rows in yield_daily_oos.groupby(["model_name", "yield_stage"]):
    yield_stage_scores.append({
        "model_name": model_name,
        "yield_stage": stage,
        "rows": len(rows),
        "mae_bu_acre": mean_absolute_error(
            rows["final_yield_bu_acre"],
            rows["yield_forecast_bu_acre"],
        ),
        "rmse_bu_acre": rmse(
            rows["final_yield_bu_acre"],
            rows["yield_forecast_bu_acre"],
        ),
    })

yield_stage_scores = pd.DataFrame(yield_stage_scores)

display(
    yield_stage_scores.sort_values(
        ["yield_stage", "mae_bu_acre"]
    )
)


### Review output

Preview the current table before continuing.


In [ ]:
best_yield_row = yield_daily_scores.iloc[0].copy()

selected_yield_model = best_yield_row["model_name"]

yield_model_selection = pd.DataFrame([{
    "selected_yield_model": selected_yield_model,
    "mae_bu_acre": best_yield_row["mae_bu_acre"],
    "rmse_bu_acre": best_yield_row["rmse_bu_acre"],
    "mae_improvement_vs_baseline": best_yield_row["mae_improvement_vs_baseline"],
    "rmse_improvement_vs_baseline": best_yield_row["rmse_improvement_vs_baseline"],
    "oos_rows": int(best_yield_row["rows"]),
}])

display(yield_model_selection)


### Review output

Preview the current table before continuing.


In [ ]:
if selected_yield_model == "lag3_baseline":
    latest_yield_forecast = yield_train_data["final_yield_bu_acre"].tail(3).mean()

elif selected_yield_model == "trend_model":
    latest_yield_features = yield_model_data.sort_values("crop_year").tail(1).copy()

    latest_yield_features["lag1_yield"] = yield_train_data["final_yield_bu_acre"].iloc[-1]
    latest_yield_features["lag3_yield_avg"] = yield_train_data["final_yield_bu_acre"].tail(3).mean()

    latest_yield_forecast = final_trend_yield_model.predict(
        latest_yield_features[trend_features]
    )[0]

elif selected_yield_model == "ridge_weather_model":
    latest_yield_features = yield_model_data.sort_values("crop_year").tail(1).copy()

    latest_yield_features["lag1_yield"] = yield_train_data["final_yield_bu_acre"].iloc[-1]
    latest_yield_features["lag3_yield_avg"] = yield_train_data["final_yield_bu_acre"].tail(3).mean()

    latest_yield_forecast = final_ridge_yield_model.predict(
        latest_yield_features[yield_feature_cols]
    )[0]

else:
    latest_yield_forecast = yield_train_data["final_yield_bu_acre"].tail(3).mean()

latest_yield_forecast = float(np.clip(latest_yield_forecast, 120.0, 220.0))

latest_crop_year = int(yield_model_data["crop_year"].max())

latest_yield_forecast_df = pd.DataFrame([{
    "crop_year": latest_crop_year,
    "yield_forecast_bu_acre": latest_yield_forecast,
    "selected_yield_model": selected_yield_model,
}])

display(latest_yield_forecast_df)


### Forecast harvested acreage

combines the selected acreage forecast with the harvested ratio forecast.


In [ ]:
latest_harvested_acres_mil = latest_harvest_forecast[
    "harvested_acres_forecast_mil"
].iloc[-1]

production_forecast_bu = (
    latest_harvested_acres_mil
    * 1_000_000
    * latest_yield_forecast
)

latest_production_forecast = pd.DataFrame([{
    "crop_year": latest_crop_year,
    "harvested_acres_forecast_mil": latest_harvested_acres_mil,
    "yield_forecast_bu_acre": latest_yield_forecast,
    "production_forecast_bu": production_forecast_bu,
    "production_forecast_bil_bu": production_forecast_bu / 1_000_000_000,
}])

display(latest_production_forecast)


## Export and demand modules

Prepare export and demand information while isolating low quality fields from the final state.


### Transform data

Prepare the next modeling table used by the workflow.


In [ ]:
export_df = modeling_panel.copy()

export_df["asof_date"] = pd.to_datetime(export_df["asof_date"], errors="coerce")
export_df["asof_timestamp"] = pd.to_datetime(export_df["asof_timestamp"], errors="coerce")

export_df = export_df.sort_values("asof_date").reset_index(drop=True)

required_cols = ["marketing_year", "marketing_year_day", "exports"]

export_df = export_df[
    export_df["asof_date"] >= "2013-01-01"
].copy()

export_df["export_prior"] = export_df["exports"]

print(f"Rows: {export_df.shape[0]:,}")
print(f"Marketing years: {export_df['marketing_year'].min()} to {export_df['marketing_year'].max()}")


### Review output

Preview the current table before continuing.


In [ ]:
export_year_status = (
    export_df
    .groupby("marketing_year", as_index=False)
    .agg(
        max_marketing_year_day=("marketing_year_day", "max"),
        last_asof_date=("asof_date", "max"),
        final_exports=("export_prior", "last"),
        rows=("asof_date", "count"),
    )
)

export_year_status["marketing_year_complete"] = (
    export_year_status["max_marketing_year_day"] >= 330
)

display(export_year_status)


### Review output

Preview the current table before continuing.


In [ ]:
export_df = export_df.merge(
    export_year_status[
        [
            "marketing_year",
            "final_exports",
            "marketing_year_complete",
            "max_marketing_year_day",
        ]
    ],
    on="marketing_year",
    how="left",
)

export_df["export_adjustment_target"] = (
    export_df["final_exports"]
    - export_df["export_prior"]
)

export_df["has_export_target"] = (
    export_df["marketing_year_complete"]
    & export_df["final_exports"].notna()
    & export_df["export_prior"].notna()
)

display(
    export_df[
        [
            "asof_date",
            "marketing_year",
            "marketing_year_day",
            "export_prior",
            "final_exports",
            "export_adjustment_target",
            "has_export_target",
        ]
    ].head(20)
)


### Review output

Preview the current table before continuing.


In [ ]:
export_df["export_stage"] = np.select(
    [
        export_df["marketing_year_day"] <= 90,
        export_df["marketing_year_day"].between(91, 180),
        export_df["marketing_year_day"].between(181, 270),
        export_df["marketing_year_day"] > 270,
    ],
    [
        "early_marketing_year",
        "mid_marketing_year",
        "late_marketing_year",
        "final_quarter",
    ],
    default="unknown",
)

stage_counts = (
    export_df
    .groupby(["marketing_year", "export_stage"])
    .size()
    .reset_index(name="rows")
    .pivot(index="marketing_year", columns="export_stage", values="rows")
    .fillna(0)
)

display(stage_counts)


### Add strategy input price risk features

adds lagged price and realized volatility features.


In [ ]:
export_df["month"] = export_df["asof_date"].dt.month
export_df["dayofyear"] = export_df["asof_date"].dt.dayofyear

if "weekly_exports" in export_df.columns:
    export_df["weekly_exports_4w_avg"] = (
        export_df
        .sort_values("asof_date")
        .groupby("marketing_year")["weekly_exports"]
        .transform(lambda s: s.rolling(4, min_periods=1).mean())
    )

if "accumulated_exports" in export_df.columns:
    export_df["accumulated_exports_change_4w"] = (
        export_df
        .sort_values("asof_date")
        .groupby("marketing_year")["accumulated_exports"]
        .transform(lambda s: s.diff(20))
    )

candidate_export_features = [
    "marketing_year_day",
    "month",
    "dayofyear",
    "export_stage",
    "seasonal_regime",

    "export_prior",
    "ty_exports",
    "ty_imports",

    "weekly_exports",
    "weekly_exports_4w_avg",
    "accumulated_exports",
    "accumulated_exports_1",
    "accumulated_exports_2",
    "accumulated_exports_change_4w",

    "front_settle",
    "front_return_5d",
    "front_realized_vol_20d",
    "stocks_to_use",

    "managed_money_net_pct_oi",
    "managed_money_net_z_1y_clean",
    "managed_money_net_change_clean",
]


### Review output

Preview the current table before continuing.


In [ ]:
export_feature_cols = [
    col for col in candidate_export_features
    if col in export_df.columns
]

export_target_col = "export_adjustment_target"

display(export_feature_cols)


### Review output

Preview the current table before continuing.


In [ ]:
export_train_daily = export_df[
    export_df["has_export_target"]
].copy()

export_train_events = (
    export_train_daily
    .sort_values("asof_date")
    .groupby(["marketing_year", "export_stage"], as_index=False)
    .tail(1)
    .sort_values(["marketing_year", "asof_date"])
    .reset_index(drop=True)
)

export_live = export_df[
    ~export_df["marketing_year_complete"]
    & export_df["export_prior"].notna()
].copy()

print(f"Daily export training rows: {export_train_daily.shape[0]:,}")
print(f"Event export training rows: {export_train_events.shape[0]:,}")
print(f"Live export rows: {export_live.shape[0]:,}")

display(
    export_train_events[
        [
            "asof_date",
            "marketing_year",
            "marketing_year_day",
            "export_stage",
            "export_prior",
            "final_exports",
            "export_adjustment_target",
        ]
    ]
)


### Build export model data

prepares export model features and target variables.


In [ ]:
base_export_cols = [
    "asof_date",
    "marketing_year",
    "marketing_year_day",
    "export_stage",
    "export_prior",
    "final_exports",
]

export_model_cols = list(dict.fromkeys(
    base_export_cols + export_feature_cols + [export_target_col]
))

export_model_data = export_train_events[export_model_cols].copy()
export_model_data = export_model_data.loc[:, ~export_model_data.columns.duplicated()]
export_model_data = export_model_data.dropna(
    subset=["export_prior", "final_exports", export_target_col]
)

print(f"Rows: {export_model_data.shape[0]:,}")
print(f"Marketing years: {export_model_data['marketing_year'].min()} to {export_model_data['marketing_year'].max()}")

display(export_model_data)


### Define zero_export_adjustment

defines `zero_export_adjustment` so later cells can reuse the same transformation without repeating code.


In [ ]:
def zero_export_adjustment(train_data, test_data):
    return np.zeros(len(test_data))


def stage_median_export_adjustment(train_data, test_data, shrink=0.25):
    predictions = []

    for _, row in test_data.iterrows():
        stage = row["export_stage"]

        stage_history = train_data.loc[
            train_data["export_stage"] == stage,
            export_target_col,
        ]

        if len(stage_history) >= 2:
            adjustment = stage_history.median()
        else:
            adjustment = train_data[export_target_col].median()

        predictions.append(shrink * adjustment)

    return np.array(predictions)


### Build export model data

prepares export model features and target variables.


In [ ]:
from sklearn.compose import ColumnTransformer
from sklearn.impute import SimpleImputer
from sklearn.linear_model import Ridge
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler

export_feature_cols = list(dict.fromkeys(export_feature_cols))

categorical_export_features = [
    column for column in export_feature_cols
    if export_model_data[column].dtype == "object"
]

numeric_export_features = [
    column for column in export_feature_cols
    if column not in categorical_export_features
]

one_hot = OneHotEncoder(handle_unknown="ignore", sparse_output=False)

numeric_export_pipeline = Pipeline([
    ("imputer", SimpleImputer(strategy="median")),
    ("scaler", StandardScaler()),
])

categorical_export_pipeline = Pipeline([
    ("imputer", SimpleImputer(strategy="most_frequent")),
    ("one_hot", one_hot),
])

export_preprocessor = ColumnTransformer([
    ("numeric", numeric_export_pipeline, numeric_export_features),
    ("categorical", categorical_export_pipeline, categorical_export_features),
])

ridge_export_model = Pipeline([
    ("preprocessor", export_preprocessor),
    ("model", Ridge(alpha=10.0)),
])

print("Numeric export features:")
display(numeric_export_features)

print("Categorical export features:")
display(categorical_export_features)


### Build export model data

prepares export model features and target variables.


In [ ]:
from sklearn.metrics import mean_absolute_error, mean_squared_error

def rmse(actual, predicted):
    return np.sqrt(mean_squared_error(actual, predicted))

export_oos_rows = []

marketing_years = sorted(export_model_data["marketing_year"].unique())


### Build export model data

prepares export model features and target variables.


In [ ]:
for test_year in marketing_years:
    train_years = [year for year in marketing_years if year < test_year]

    if len(train_years) < 5:
        continue

    train_data = export_model_data[
        export_model_data["marketing_year"].isin(train_years)
    ].copy()

    test_data = export_model_data[
        export_model_data["marketing_year"] == test_year
    ].copy()

    model_specs = {
        "no_adjustment_baseline": lambda train, test: zero_export_adjustment(train, test),
        "stage_median_shrink_10": lambda train, test: stage_median_export_adjustment(train, test, shrink=0.10),
        "stage_median_shrink_25": lambda train, test: stage_median_export_adjustment(train, test, shrink=0.25),
        "stage_median_shrink_50": lambda train, test: stage_median_export_adjustment(train, test, shrink=0.50),
    }

    for model_name, model_func in model_specs.items():
        candidate_frame = test_data.copy()
        candidate_frame["model_name"] = model_name
        candidate_frame["predicted_export_adjustment"] = model_func(train_data, test_data)
        candidate_frame["export_forecast"] = candidate_frame["export_prior"] + candidate_frame["predicted_export_adjustment"]
        export_oos_rows.append(candidate_frame)

    ridge_export_model.fit(
        train_data[export_feature_cols],
        train_data[export_target_col],
    )

    candidate_frame = test_data.copy()
    candidate_frame["model_name"] = "ridge_export_adjustment"
    candidate_frame["predicted_export_adjustment"] = ridge_export_model.predict(
        test_data[export_feature_cols]
    )
    candidate_frame["export_forecast"] = candidate_frame["export_prior"] + candidate_frame["predicted_export_adjustment"]
    export_oos_rows.append(candidate_frame)

export_oos = pd.concat(export_oos_rows, ignore_index=True)


### Review output

Preview the current table before continuing.


In [ ]:
display(
    export_oos[
        [
            "model_name",
            "asof_date",
            "marketing_year",
            "export_stage",
            "export_prior",
            "final_exports",
            "predicted_export_adjustment",
            "export_forecast",
        ]
    ].head(40)
)


### Score export models

evaluates export model candidates.


In [ ]:
export_scores = []

for model_name, rows in export_oos.groupby("model_name"):
    mae = mean_absolute_error(
        rows["final_exports"],
        rows["export_forecast"],
    )

    model_rmse = rmse(
        rows["final_exports"],
        rows["export_forecast"],
    )

    export_scores.append({
        "model_name": model_name,
        "rows": len(rows),
        "mae_export_units": mae,
        "rmse_export_units": model_rmse,
    })

export_scores = pd.DataFrame(export_scores)

baseline_mae = export_scores.loc[
    export_scores["model_name"] == "no_adjustment_baseline",
    "mae_export_units",
].iloc[0]

baseline_rmse = export_scores.loc[
    export_scores["model_name"] == "no_adjustment_baseline",
    "rmse_export_units",
].iloc[0]

export_scores["mae_improvement_vs_baseline"] = (
    baseline_mae - export_scores["mae_export_units"]
)

export_scores["rmse_improvement_vs_baseline"] = (
    baseline_rmse - export_scores["rmse_export_units"]
)

export_scores = export_scores.sort_values("mae_export_units")

display(export_scores)


### Score export models

evaluates export model candidates.


In [ ]:
plot_model_scores(
    export_scores,
    model_col="model_name",
    metric_cols=["mae_exports_mil_bu", "rmse_exports_mil_bu"],
    title="Export model scorecard",
)


### Score export models

evaluates export model candidates.


In [ ]:
best_export_row = export_scores.iloc[0].copy()

selected_export_model = best_export_row["model_name"]

export_model_selection = pd.DataFrame([{
    "selected_export_model": selected_export_model,
    "mae_export_units": best_export_row["mae_export_units"],
    "rmse_export_units": best_export_row["rmse_export_units"],
    "mae_improvement_vs_baseline": best_export_row["mae_improvement_vs_baseline"],
    "rmse_improvement_vs_baseline": best_export_row["rmse_improvement_vs_baseline"],
    "oos_rows": int(best_export_row["rows"]),
}])

display(export_model_selection)


### Build export model data

prepares export model features and target variables.


In [ ]:
latest_export_row = export_live.sort_values("asof_date").tail(1).copy()

if len(latest_export_row) == 0:
    latest_export_row = export_df.sort_values("asof_date").tail(1).copy()

if selected_export_model == "no_adjustment_baseline":
    latest_export_row["predicted_export_adjustment"] = 0.0

elif selected_export_model.startswith("stage_median"):
    shrink_map = {
        "stage_median_shrink_10": 0.10,
        "stage_median_shrink_25": 0.25,
        "stage_median_shrink_50": 0.50,
    }

    shrink = shrink_map[selected_export_model]
    stage = latest_export_row["export_stage"].iloc[0]

    stage_history = export_model_data.loc[
        export_model_data["export_stage"] == stage,
        export_target_col,
    ]

    if len(stage_history) >= 2:
        adjustment = stage_history.median()
    else:
        adjustment = export_model_data[export_target_col].median()

    latest_export_row["predicted_export_adjustment"] = shrink * adjustment

else:
    final_ridge_export_model = ridge_export_model

    final_ridge_export_model.fit(
        export_model_data[export_feature_cols],
        export_model_data[export_target_col],
    )

    latest_export_row["predicted_export_adjustment"] = final_ridge_export_model.predict(
        latest_export_row[export_feature_cols]
    )

latest_export_row["export_forecast"] = (
    latest_export_row["export_prior"]
    + latest_export_row["predicted_export_adjustment"]
)


### Review output

Preview the current table before continuing.


In [ ]:
latest_export_forecast = latest_export_row[
    [
        "asof_date",
        "marketing_year",
        "marketing_year_day",
        "export_stage",
        "export_prior",
        "predicted_export_adjustment",
        "export_forecast",
        "selected_export_model",
    ]
].copy() if "selected_export_model" in latest_export_row.columns else latest_export_row[
    [
        "asof_date",
        "marketing_year",
        "marketing_year_day",
        "export_stage",
        "export_prior",
        "predicted_export_adjustment",
        "export_forecast",
    ]
].copy()

latest_export_forecast["selected_export_model"] = selected_export_model

display(latest_export_forecast)


### Review output

Preview the current table before continuing.


In [ ]:
export_unit_audit = (
    export_df
    .groupby("marketing_year", as_index=False)
    .agg(
        first_asof_date=("asof_date", "min"),
        last_asof_date=("asof_date", "max"),
        max_marketing_year_day=("marketing_year_day", "max"),
        first_exports=("exports", "first"),
        median_exports=("exports", "median"),
        last_exports=("exports", "last"),
        min_exports=("exports", "min"),
        max_exports=("exports", "max"),
        rows=("asof_date", "count"),
    )
)

display(export_unit_audit)


### Review output

Preview the current table before continuing.


In [ ]:
export_cols_to_check = [
    "asof_date",
    "marketing_year",
    "marketing_year_day",
    "exports",
    "ty_exports",
    "weekly_exports",
    "accumulated_exports",
    "accumulated_exports_1",
    "accumulated_exports_2",
]

export_cols_to_check = [
    col for col in export_cols_to_check
    if col in export_df.columns
]

display(
    export_df[export_cols_to_check]
    .sort_values("asof_date")
    .tail(20)
)


### Review output

Preview the current table before continuing.


In [ ]:
selected_export_model = "no_adjustment_baseline"

latest_export_forecast = latest_export_forecast.copy()
latest_export_forecast["selected_export_model"] = selected_export_model

display(latest_export_forecast)


### Review output

Preview the current table before continuing.


In [ ]:
export_module_status = pd.DataFrame([{
    "module": "exports",
    "status": "quarantined",
    "reason": "exports, ty_exports, weekly_exports, and accumulated_exports show inconsistent units or mismerged values",
    "temporary_rule": "use balance-sheet prior total_use and ending_stocks; do not use raw exports for reconciliation",
}])

display(export_module_status)


### Review output

Preview the current table before continuing.


In [ ]:
balance_df = modeling_panel.copy()

balance_df["asof_date"] = pd.to_datetime(balance_df["asof_date"], errors="coerce")
balance_df = balance_df.sort_values("asof_date").reset_index(drop=True)

demand_cols = [
    "beginning_stocks",
    "production_bu",
    "imports",
    "feed_and_residual",
    "feed_dom_consumption",
    "fsi_consumption",
    "ethanol",
    "exports",
    "total_use",
    "ending_stocks",
    "stocks_to_use",
]

demand_cols = [
    col for col in demand_cols
    if col in balance_df.columns
]

demand_audit = []

for col in demand_cols:
    series = balance_df[col]

    demand_audit.append({
        "column": col,
        "non_null": series.notna().sum(),
        "coverage_pct": series.notna().mean() * 100,
        "median": series.median(),
        "min": series.min(),
        "max": series.max(),
        "latest": series.dropna().iloc[-1] if series.notna().any() else np.nan,
    })

demand_audit = pd.DataFrame(demand_audit)

display(demand_audit)


### Review output

Preview the current table before continuing.


In [ ]:
latest_balance_row = balance_df.tail(1).copy()

latest_balance_snapshot = latest_balance_row[
    [
        col for col in [
            "asof_date",
            "marketing_year",
            "marketing_year_day",
            "beginning_stocks",
            "production_bu",
            "imports",
            "feed_and_residual",
            "feed_dom_consumption",
            "fsi_consumption",
            "ethanol",
            "exports",
            "total_use",
            "ending_stocks",
            "stocks_to_use",
        ]
        if col in latest_balance_row.columns
    ]
].T

display(latest_balance_snapshot)


## Balance sheet reconciliation

Reconcile acreage, yield, demand, ending stocks, and stocks to use into a state forecast state forecast.


### Review output

Preview the current table before continuing.


In [ ]:
model_production_bil_bu = latest_production_forecast["production_forecast_bil_bu"].iloc[-1]
model_production_mil_bu = model_production_bil_bu * 1_000

prior_production_raw = latest_balance_row["production_bu"].iloc[0]

if prior_production_raw > 1_000_000:
    prior_production_mil_bu = prior_production_raw / 1_000_000
    prior_production_unit = "bushels_converted_to_million_bushels"
else:
    prior_production_mil_bu = prior_production_raw
    prior_production_unit = "already_million_bushels"

production_delta_mil_bu = model_production_mil_bu - prior_production_mil_bu

production_unit_check = pd.DataFrame([{
    "model_production_bil_bu": model_production_bil_bu,
    "model_production_mil_bu": model_production_mil_bu,
    "prior_production_raw": prior_production_raw,
    "prior_production_unit": prior_production_unit,
    "prior_production_mil_bu": prior_production_mil_bu,
    "production_delta_mil_bu": production_delta_mil_bu,
}])

display(production_unit_check)


### Review output

Preview the current table before continuing.


In [ ]:
latest_demand_forecast = pd.DataFrame([{
    "asof_date": latest_balance_row["asof_date"].iloc[0],
    "marketing_year": latest_balance_row["marketing_year"].iloc[0],
    "demand_model_version": "prior_baseline_v1",

    "imports_forecast_mil_bu": latest_balance_row["imports"].iloc[0]
        if "imports" in latest_balance_row.columns else np.nan,

    "feed_residual_forecast_mil_bu": latest_balance_row["feed_and_residual"].iloc[0]
        if "feed_and_residual" in latest_balance_row.columns else np.nan,

    "fsi_forecast_mil_bu": latest_balance_row["fsi_consumption"].iloc[0]
        if "fsi_consumption" in latest_balance_row.columns else np.nan,

    "ethanol_forecast_mil_bu": latest_balance_row["ethanol"].iloc[0]
        if "ethanol" in latest_balance_row.columns else np.nan,

    "exports_forecast_mil_bu": np.nan,

    "total_use_forecast_mil_bu": latest_balance_row["total_use"].iloc[0],
    "demand_forecast_note": "total_use held at latest balance-sheet prior; exports component quarantined",
}])

display(latest_demand_forecast)


### Reconcile balance sheet

reconciles production, demand, ending stocks, and stocks to use.


In [ ]:
prior_ending_stocks_mil_bu = latest_balance_row["ending_stocks"].iloc[0]
prior_total_use_mil_bu = latest_balance_row["total_use"].iloc[0]
prior_stocks_to_use = latest_balance_row["stocks_to_use"].iloc[0]

model_ending_stocks_mil_bu = prior_ending_stocks_mil_bu + production_delta_mil_bu
model_total_use_mil_bu = prior_total_use_mil_bu
model_stocks_to_use = model_ending_stocks_mil_bu / model_total_use_mil_bu

stocks_to_use_change = model_stocks_to_use - prior_stocks_to_use

balance_signal = np.select(
    [
        stocks_to_use_change <= -0.01,
        stocks_to_use_change >= 0.01,
    ],
    [
        "tighter_than_prior",
        "looser_than_prior",
    ],
    default="near_prior",
)

latest_balance_reconciliation = pd.DataFrame([{
    "asof_date": latest_balance_row["asof_date"].iloc[0],
    "marketing_year": latest_balance_row["marketing_year"].iloc[0],
    "prior_production_mil_bu": prior_production_mil_bu,
    "model_production_mil_bu": model_production_mil_bu,
    "production_delta_mil_bu": production_delta_mil_bu,
    "prior_ending_stocks_mil_bu": prior_ending_stocks_mil_bu,
    "model_ending_stocks_mil_bu": model_ending_stocks_mil_bu,
    "prior_total_use_mil_bu": prior_total_use_mil_bu,
    "model_total_use_mil_bu": model_total_use_mil_bu,
    "prior_stocks_to_use": prior_stocks_to_use,
    "model_stocks_to_use": model_stocks_to_use,
    "stocks_to_use_change": stocks_to_use_change,
    "balance_sheet_signal": balance_signal,
}])

display(latest_balance_reconciliation)


### Select the acreage model

chooses the acreage model used downstream in the state panel.


In [ ]:
state_forecast_module_status = pd.DataFrame([
    {
        "module": "acreage",
        "status": "complete_v1",
        "selected_model": selected_acreage_model_name,
        "note": "uses NASS priors plus tested adjustment model",
    },
    {
        "module": "harvested_ratio",
        "status": "complete_v1",
        "selected_model": selected_harvest_model,
        "note": "uses official NASS harvested acre history",
    },
    {
        "module": "yield",
        "status": "complete_v1",
        "selected_model": selected_yield_model,
        "note": "weather models failed; lag3 baseline selected",
    },
    {
        "module": "production",
        "status": "complete_v1",
        "selected_model": "acreage_x_harvest_ratio_x_yield",
        "note": "computed from supply submodules",
    },
    {
        "module": "exports",
        "status": "quarantined",
        "selected_model": "not_used",
        "note": "raw export columns show inconsistent units or mismerged values",
    },
    {
        "module": "ethanol_fsi_feed_total_use",
        "status": "prior_baseline_v1",
        "selected_model": "latest_balance_sheet_prior",
        "note": "component demand models deferred until cleaner WASDE EIA Export Sales data are added",
    },
    {
        "module": "ending_stocks_stocks_to_use",
        "status": "complete_v1",
        "selected_model": "prior_ending_stocks_plus_production_delta",
        "note": "demand held constant and production delta flows into ending stocks",
    },
])

display(state_forecast_module_status)


### Summarize state forecast modules

records the selected status and model choice for each state forecast module.


In [ ]:
status_counts = state_forecast_module_status["status"].value_counts()

ax = status_counts.plot(kind="bar", figsize=(8, 4))
ax.set_title("state forecast module status counts")
ax.set_xlabel("Status")
ax.set_ylabel("Modules")
ax.grid(axis="y", alpha=0.25)
plt.xticks(rotation=45, ha="right")
plt.tight_layout()
plt.show()


### Forecast harvested acreage

combines the selected acreage forecast with the harvested ratio forecast.


In [ ]:
latest_acreage_value = acreage_live.sort_values("asof_date")["acreage_prior_mil"].iloc[-1]
latest_harvest_ratio = latest_harvest_forecast["harvest_ratio_forecast"].iloc[-1]
latest_harvested_acres_mil = latest_harvest_forecast["harvested_acres_forecast_mil"].iloc[-1]
latest_yield_bu_acre = latest_yield_forecast_df["yield_forecast_bu_acre"].iloc[-1]

state_forecast_state_forecast = pd.DataFrame([{
    "date": latest_balance_row["asof_date"].iloc[0],
    "crop_year": latest_production_forecast["crop_year"].iloc[-1],
    "marketing_year": latest_balance_row["marketing_year"].iloc[0],

    "acreage_forecast_mil": latest_acreage_value,
    "harvest_ratio_forecast": latest_harvest_ratio,
    "harvested_acres_forecast_mil": latest_harvested_acres_mil,
    "yield_forecast_bu_acre": latest_yield_bu_acre,
    "production_forecast_bil_bu": model_production_bil_bu,

    "prior_production_mil_bu": prior_production_mil_bu,
    "model_production_mil_bu": model_production_mil_bu,
    "production_delta_mil_bu": production_delta_mil_bu,

    "prior_ending_stocks_mil_bu": prior_ending_stocks_mil_bu,
    "model_ending_stocks_mil_bu": model_ending_stocks_mil_bu,

    "prior_total_use_mil_bu": prior_total_use_mil_bu,
    "model_total_use_mil_bu": model_total_use_mil_bu,

    "prior_stocks_to_use": prior_stocks_to_use,
    "model_stocks_to_use": model_stocks_to_use,
    "stocks_to_use_change": stocks_to_use_change,

    "state_signal": balance_signal,
}])

state_forecast_state_forecast["qc_target_direction"] = np.select(
    [
        state_forecast_state_forecast["stocks_to_use_change"] < -0.005,
        state_forecast_state_forecast["stocks_to_use_change"] > 0.005,
    ],
    [
        "bullish_tighter_balance",
        "bearish_looser_balance",
    ],
    default="neutral_small_change",
)

display(state_forecast_state_forecast)


### Reconcile balance sheet

reconciles production, demand, ending stocks, and stocks to use.


In [ ]:
snapshot_cols = [
    "acreage_forecast_mil",
    "harvested_acres_forecast_mil",
    "yield_forecast_bu_acre",
    "production_forecast_bil_bu",
    "model_ending_stocks_mil_bu",
    "model_stocks_to_use",
    "stocks_to_use_change",
]

chart_data = state_forecast_state_forecast[snapshot_cols].T.reset_index()
chart_data.columns = ["metric", "value"]

ax = chart_data.plot(kind="bar", x="metric", y="value", legend=False, figsize=(10, 4))
ax.set_title("Final state forecast state snapshot")
ax.set_xlabel("")
ax.set_ylabel("Value")
ax.grid(axis="y", alpha=0.25)
plt.xticks(rotation=45, ha="right")
plt.tight_layout()
plt.show()


### Create final state forecast snapshot

creates the single latest state forecast state forecast snapshot.


In [ ]:
state_forecast_completion_check = pd.DataFrame([
    {"item": "acreage_forecast", "available": state_forecast_state_forecast["acreage_forecast_mil"].notna().all()},
    {"item": "harvest_ratio_forecast", "available": state_forecast_state_forecast["harvest_ratio_forecast"].notna().all()},
    {"item": "harvested_acres_forecast", "available": state_forecast_state_forecast["harvested_acres_forecast_mil"].notna().all()},
    {"item": "yield_forecast", "available": state_forecast_state_forecast["yield_forecast_bu_acre"].notna().all()},
    {"item": "production_forecast", "available": state_forecast_state_forecast["production_forecast_bil_bu"].notna().all()},
    {"item": "ending_stocks_forecast", "available": state_forecast_state_forecast["model_ending_stocks_mil_bu"].notna().all()},
    {"item": "stocks_to_use_forecast", "available": state_forecast_state_forecast["model_stocks_to_use"].notna().all()},
    {"item": "qc_target_direction", "available": state_forecast_state_forecast["qc_target_direction"].notna().all()},
])

display(state_forecast_completion_check)

if state_forecast_completion_check["available"].all():
    print("state forecast v1 is complete.")
else:
    print("state forecast v1 still has missing fields.")


### Build the enriched state forecast panel

builds the daily state forecast panel used by the factor layer.


In [ ]:
state_forecast_panel = modeling_panel.copy()

if "asof_date" in state_forecast_panel.index.names:
    state_forecast_panel = state_forecast_panel.reset_index(drop=True)

state_forecast_panel["asof_date"] = pd.to_datetime(state_forecast_panel["asof_date"], errors="coerce")
state_forecast_panel = state_forecast_panel.sort_values("asof_date").reset_index(drop=True)

state_forecast_panel["state_forecast_crop_year"] = state_forecast_panel["asof_date"].dt.year

print(f"Rows: {state_forecast_panel.shape[0]:,}")
print(f"Columns before state forecast additions: {state_forecast_panel.shape[1]:,}")
print(f"Date range: {state_forecast_panel['asof_date'].min().date()} to {state_forecast_panel['asof_date'].max().date()}")


### Prepare acreage features

prepares the acreage training table and stage specific priors.


In [ ]:
acreage_history = acreage_df.copy()

acreage_history["asof_date"] = pd.to_datetime(acreage_history["asof_date"], errors="coerce")
acreage_history = acreage_history.sort_values("asof_date").reset_index(drop=True)

acreage_history = acreage_history.rename(columns={
    "acreage_information_stage": "acreage_stage",
    "acreage_usda_prior_mil": "acreage_prior_mil",
})

acreage_target_history = acreage_history[
    acreage_history["acreage_adjustment_target_mil"].notna()
].copy()


### Select the acreage model

chooses the acreage model used downstream in the state panel.


In [ ]:
def acreage_adjustment_for_row(row, history):
    crop_year = row["acreage_crop_year"]
    stage = row["acreage_stage"]

    train = history[history["acreage_crop_year"] < crop_year].copy()

    if len(train) < 4:
        return 0.0

    selected = selected_acreage_model_name

    if selected == "no_adjustment_baseline":
        return 0.0

    stage_history = train.loc[
        train["acreage_stage"] == stage,
        "acreage_adjustment_target_mil",
    ]

    if len(stage_history) >= 2:
        stage_median = stage_history.median()
    else:
        stage_median = train["acreage_adjustment_target_mil"].median()

    if selected == "stage_specific_hybrid":
        shrink_by_stage = {
            "pre_march_intentions": 0.50,
            "post_march_pre_june": 0.00,
            "post_june_acreage": 0.50,
        }
        shrink = shrink_by_stage.get(stage, 0.00)
    elif "shrink_10" in selected:
        shrink = 0.10
    elif "shrink_25" in selected:
        shrink = 0.25
    elif "shrink_50" in selected:
        shrink = 0.50
    else:
        shrink = 0.00

    return float(shrink * stage_median)

state_forecast_panel["acreage_adjustment_forecast_mil"] = state_forecast_panel.apply(
    acreage_adjustment_for_row,
    axis=1,
    history=acreage_target_history,
)


### Build the enriched state forecast panel

builds the daily state forecast panel used by the factor layer.


In [ ]:
state_forecast_panel["acreage_forecast_mil"] = (
    state_forecast_panel["acreage_prior_mil"]
    + state_forecast_panel["acreage_adjustment_forecast_mil"]
)

display(
    state_forecast_panel[
        [
            "asof_date",
            "acreage_crop_year",
            "acreage_stage",
            "acreage_prior_mil",
            "acreage_adjustment_forecast_mil",
            "acreage_forecast_mil",
        ]
    ].tail(20)
)


### Build the enriched state forecast panel

builds the daily state forecast panel used by the factor layer.


In [ ]:
acreage_export_cols = [
    "asof_date",
    "acreage_crop_year",
    "acreage_stage",
    "acreage_prior_mil",
    "acreage_model_adjustment_mil",
    "acreage_forecast_mil",
    "acreage_prior_source",
]

acreage_export_cols = [
    col for col in acreage_export_cols
    if col in acreage_history.columns
]

state_forecast_panel = state_forecast_panel.merge(
    acreage_history[acreage_export_cols],
    on="asof_date",
    how="left",
)

state_forecast_panel["acreage_crop_year"] = state_forecast_panel["acreage_crop_year"].fillna(
    state_forecast_panel["state_forecast_crop_year"]
)

state_forecast_panel["acreage_forecast_mil"] = (
    state_forecast_panel
    .sort_values("asof_date")
    .groupby("acreage_crop_year")["acreage_forecast_mil"]
    .ffill()
    .bfill()
)

display(
    state_forecast_panel[
        [
            "asof_date",
            "acreage_crop_year",
            "acreage_stage",
            "acreage_prior_mil",
            "acreage_model_adjustment_mil",
            "acreage_forecast_mil",
        ]
    ].tail(20)
)


### Build harvested ratio history

builds the harvested ratio history from planted and harvested acres.


In [ ]:
harvest_ratio_history = (
    nass_harvest_history
    .dropna(subset=["acreage_crop_year", "harvest_ratio"])
    .sort_values("acreage_crop_year")
    .copy()
)

harvest_ratio_map = harvest_ratio_history.set_index("acreage_crop_year")["harvest_ratio"].to_dict()

def trailing_harvest_ratio(crop_year, lookback=5):
    values = [
        harvest_ratio_map[year]
        for year in sorted(harvest_ratio_map)
        if year < crop_year
    ]

    if len(values) == 0:
        return np.nan

    return float(np.median(values[-lookback:]))

state_forecast_panel["harvest_ratio_forecast"] = state_forecast_panel["acreage_crop_year"].apply(
    trailing_harvest_ratio
)

state_forecast_panel["harvested_acres_forecast_mil"] = (
    state_forecast_panel["acreage_forecast_mil"]
    * state_forecast_panel["harvest_ratio_forecast"]
)

display(
    state_forecast_panel[
        [
            "asof_date",
            "acreage_crop_year",
            "acreage_forecast_mil",
            "harvest_ratio_forecast",
            "harvested_acres_forecast_mil",
        ]
    ].tail(20)
)


### Build yield history

builds the yield training history.


In [ ]:
yield_history = (
    nass_yield_history
    .dropna(subset=["crop_year", "final_yield_bu_acre"])
    .sort_values("crop_year")
    .copy()
)

yield_map = yield_history.set_index("crop_year")["final_yield_bu_acre"].to_dict()

def trailing_lag3_yield(crop_year):
    values = [
        yield_map[year]
        for year in sorted(yield_map)
        if year < crop_year
    ]

    if len(values) == 0:
        return np.nan

    return float(np.mean(values[-3:]))

state_forecast_panel["yield_forecast_bu_acre"] = state_forecast_panel["acreage_crop_year"].apply(
    trailing_lag3_yield
)

display(
    state_forecast_panel[
        [
            "asof_date",
            "acreage_crop_year",
            "yield_forecast_bu_acre",
        ]
    ].tail(20)
)


### Build the enriched state forecast panel

builds the daily state forecast panel used by the factor layer.


In [ ]:
state_forecast_panel["production_forecast_mil_bu"] = (
    state_forecast_panel["harvested_acres_forecast_mil"]
    * state_forecast_panel["yield_forecast_bu_acre"]
)

state_forecast_panel["production_forecast_bil_bu"] = (
    state_forecast_panel["production_forecast_mil_bu"]
    / 1_000
)

display(
    state_forecast_panel[
        [
            "asof_date",
            "acreage_crop_year",
            "acreage_forecast_mil",
            "harvest_ratio_forecast",
            "harvested_acres_forecast_mil",
            "yield_forecast_bu_acre",
            "production_forecast_mil_bu",
            "production_forecast_bil_bu",
        ]
    ].tail(20)
)


### Build the enriched state forecast panel

builds the daily state forecast panel used by the factor layer.


In [ ]:
plot_time_series(
    state_forecast_panel,
    date_col="asof_date",
    value_cols=[
        "production_forecast_bil_bu",
        "yield_forecast_bu_acre",
        "acreage_forecast_mil",
        "harvested_acres_forecast_mil",
    ],
    title="state forecast production forecast and drivers",
    ylabel="Value",
    tail=500,
)


### Reconcile balance sheet

reconciles production, demand, ending stocks, and stocks to use.


In [ ]:
state_forecast_panel["prior_production_raw"] = state_forecast_panel["production_bu"]

state_forecast_panel["prior_production_mil_bu"] = np.where(
    state_forecast_panel["prior_production_raw"] > 1_000_000,
    state_forecast_panel["prior_production_raw"] / 1_000_000,
    state_forecast_panel["prior_production_raw"],
)

state_forecast_panel["prior_ending_stocks_mil_bu"] = state_forecast_panel["ending_stocks"]
state_forecast_panel["prior_stocks_to_use"] = state_forecast_panel["stocks_to_use"]

state_forecast_panel["prior_total_use_mil_bu"] = state_forecast_panel["total_use"]

state_forecast_panel["prior_total_use_mil_bu"] = state_forecast_panel["prior_total_use_mil_bu"].fillna(
    state_forecast_panel["prior_ending_stocks_mil_bu"] / state_forecast_panel["prior_stocks_to_use"]
)

state_forecast_panel["production_delta_mil_bu"] = (
    state_forecast_panel["production_forecast_mil_bu"]
    - state_forecast_panel["prior_production_mil_bu"]
)

state_forecast_panel["model_ending_stocks_mil_bu"] = (
    state_forecast_panel["prior_ending_stocks_mil_bu"]
    + state_forecast_panel["production_delta_mil_bu"]
)

state_forecast_panel["model_total_use_mil_bu"] = state_forecast_panel["prior_total_use_mil_bu"]

state_forecast_panel["model_stocks_to_use"] = (
    state_forecast_panel["model_ending_stocks_mil_bu"]
    / state_forecast_panel["model_total_use_mil_bu"]
)

state_forecast_panel["stocks_to_use_change"] = (
    state_forecast_panel["model_stocks_to_use"]
    - state_forecast_panel["prior_stocks_to_use"]
)


### Build the enriched state forecast panel

builds the daily state forecast panel used by the factor layer.


In [ ]:
display(
    state_forecast_panel[
        [
            "asof_date",
            "prior_production_mil_bu",
            "production_forecast_mil_bu",
            "production_delta_mil_bu",
            "prior_stocks_to_use",
            "model_stocks_to_use",
            "stocks_to_use_change",
        ]
    ].tail(20)
)


## Factor construction

Convert the reconciled physical state into supply, balance, demand, positioning, seasonality, and risk factors.


### Define state forecast factor helpers

defines the reusable factor construction helpers.


In [ ]:
import numpy as np
import pandas as pd

PHASE3_FACTOR_MODEL_VERSION = "state_forecast_factor_panel_v3_anchor_for_interpretation_weights_fitted_in_qc"

def _as_float_series(series):
    return pd.to_numeric(series, errors="coerce").astype(float)

def _empty_series(index):
    return pd.Series(np.nan, index=index, dtype=float)

def first_available_numeric(raw_corn_panel, candidates, default=np.nan):
    for col in candidates:
        if col in raw_corn_panel.columns:
            return _as_float_series(raw_corn_panel[col])
    return pd.Series(default, index=raw_corn_panel.index, dtype=float)

def lagged_rolling_mean(series, window=756, min_periods=126):
    x = _as_float_series(series)
    roll = x.rolling(window=window, min_periods=min_periods).mean().shift(1)
    exp = x.expanding(min_periods=min_periods).mean().shift(1)
    return roll.fillna(exp)

def lagged_rolling_zscore(series, window=756, min_periods=126, clip=3.0):
    """
    No lookahead rolling z score.

    The rolling mean and standard deviation are shifted by one row so the value
    at t is scaled with information available before t.
    """
    x = _as_float_series(series)

    roll_mean = x.rolling(window=window, min_periods=min_periods).mean().shift(1)
    roll_std = x.rolling(window=window, min_periods=min_periods).std(ddof=0).shift(1)

    exp_mean = x.expanding(min_periods=min_periods).mean().shift(1)
    exp_std = x.expanding(min_periods=min_periods).std(ddof=0).shift(1)

    mean = roll_mean.fillna(exp_mean)
    std = roll_std.fillna(exp_std).replace(0, np.nan)

    z = (x - mean) / std

    if clip is not None:
        z = z.clip(-clip, clip)

    return z


### Define bounded_value

defines `bounded_value` so later cells can reuse the same transformation without repeating code.


In [ ]:
def bounded_value(series, lower=-3.0, upper=3.0):
    return _as_float_series(series).clip(lower, upper)


### Build the enriched state forecast panel

builds the daily state forecast panel used by the factor layer.


In [ ]:
def row_weighted_mean(data, columns, weights=None, min_non_null=1):
    """
    Weighted row wise mean that ignores missing components.
    """
    existing = [column for column in columns if column in data.columns]

    if weights is None:
        weights = {column: 1.0 for column in existing}
    else:
        weights = {column: float(weights.get(column, 0.0)) for column in existing}

    numerator = pd.Series(0.0, index=data.index)
    denominator = pd.Series(0.0, index=data.index)
    non_null_count = pd.Series(0, index=data.index)

    for column in existing:
        values = _as_float_series(data[column])
        mask = values.notna()
        weight = weights.get(column, 0.0)

        numerator = numerator + values.fillna(0.0) * weight
        denominator = denominator + mask.astype(float) * abs(weight)
        non_null_count = non_null_count + mask.astype(int)

    result_series = numerator / denominator.replace(0, np.nan)
    result_series = result_series.where(non_null_count >= min_non_null)

    return result_series.clip(-3, 3)

state_forecast_panel = state_forecast_panel.copy()
state_forecast_panel["asof_date"] = pd.to_datetime(state_forecast_panel["asof_date"], errors="coerce")
state_forecast_panel = state_forecast_panel.sort_values("asof_date").reset_index(drop=True)

print("state forecast factor helpers ready")
print("Rows:", len(state_forecast_panel))
print("Date range:", state_forecast_panel["asof_date"].min(), "to", state_forecast_panel["asof_date"].max())


### Build the enriched state forecast panel

builds the daily state forecast panel used by the factor layer.


In [ ]:
front_settle_series = first_available_numeric(state_forecast_panel, ["front_settle"])

state_forecast_panel["front_return_1d_calc"] = front_settle_series.pct_change().shift(1)
state_forecast_panel["front_return_5d_calc"] = front_settle_series.pct_change(5).shift(1)

state_forecast_panel["front_realized_vol_20d_calc"] = (
    state_forecast_panel["front_return_1d_calc"]
    .rolling(20, min_periods=15)
    .std()
    * np.sqrt(252)
)

state_forecast_panel["front_realized_vol_60d_calc"] = (
    state_forecast_panel["front_return_1d_calc"]
    .rolling(60, min_periods=40)
    .std()
    * np.sqrt(252)
)

print("Lagged price risk inputs created.")


### Build the enriched state forecast panel

builds the daily state forecast panel used by the factor layer.


In [ ]:
production_delta = first_available_numeric(
    state_forecast_panel,
    ["production_delta_mil_bu"]
)

if production_delta.notna().any():
    state_forecast_panel["production_tightness_raw"] = -production_delta / 1000.0
else:
    production_forecast = first_available_numeric(state_forecast_panel, ["production_forecast_mil_bu"])
    production_prior = first_available_numeric(state_forecast_panel, ["prior_production_mil_bu"])
    production_baseline = production_prior.fillna(lagged_rolling_mean(production_forecast))
    state_forecast_panel["production_tightness_raw"] = -(production_forecast - production_baseline) / 1000.0

acreage_forecast = first_available_numeric(state_forecast_panel, ["acreage_forecast_mil"])
acreage_prior = first_available_numeric(
    state_forecast_panel,
    ["acreage_prior_mil", "planted_acres_prior_mil", "prior_planted_acres_mil"]
)
acreage_baseline = acreage_prior.fillna(lagged_rolling_mean(acreage_forecast))
state_forecast_panel["acreage_surprise_mil"] = acreage_forecast - acreage_baseline
state_forecast_panel["acreage_tightness_raw"] = -state_forecast_panel["acreage_surprise_mil"]

harvested_forecast = first_available_numeric(state_forecast_panel, ["harvested_acres_forecast_mil"])
harvested_prior = first_available_numeric(
    state_forecast_panel,
    ["harvested_acres_prior_mil", "prior_harvested_acres_mil"]
)
harvested_baseline = harvested_prior.fillna(lagged_rolling_mean(harvested_forecast))
state_forecast_panel["harvested_acres_surprise_mil"] = harvested_forecast - harvested_baseline
state_forecast_panel["harvested_acres_tightness_raw"] = -state_forecast_panel["harvested_acres_surprise_mil"]

yield_forecast = first_available_numeric(state_forecast_panel, ["yield_forecast_bu_acre"])
yield_prior = first_available_numeric(
    state_forecast_panel,
    ["yield_prior_bu_acre", "prior_yield_bu_acre", "trend_yield_bu_acre"]
)
yield_baseline = yield_prior.fillna(lagged_rolling_mean(yield_forecast))
state_forecast_panel["yield_surprise_bu_acre"] = yield_forecast - yield_baseline
state_forecast_panel["yield_tightness_raw"] = -state_forecast_panel["yield_surprise_bu_acre"]


### Build the enriched state forecast panel

builds the daily state forecast panel used by the factor layer.


In [ ]:
for raw_col, z_col in [
    ("production_tightness_raw", "production_tightness_z"),
    ("acreage_tightness_raw", "acreage_tightness_z"),
    ("harvested_acres_tightness_raw", "harvested_acres_tightness_z"),
    ("yield_tightness_raw", "yield_tightness_z"),
]:
    state_forecast_panel[z_col] = lagged_rolling_zscore(
        state_forecast_panel[raw_col],
        window=756,
        min_periods=126,
    )

state_forecast_panel["supply_factor_z"] = row_weighted_mean(
    state_forecast_panel,
    [
        "production_tightness_z",
        "acreage_tightness_z",
        "harvested_acres_tightness_z",
        "yield_tightness_z",
    ],
    weights={
        "production_tightness_z": 0.40,
        "acreage_tightness_z": 0.20,
        "harvested_acres_tightness_z": 0.15,
        "yield_tightness_z": 0.25,
    },
    min_non_null=2,
)

print("Supply factor created.")


### Reconcile balance sheet

reconciles production, demand, ending stocks, and stocks to use.


In [ ]:
state_forecast_panel["stocks_to_use_tightness_raw"] = -first_available_numeric(
    state_forecast_panel,
    ["model_stocks_to_use"]
)

state_forecast_panel["stocks_to_use_revision_tightness_raw"] = -first_available_numeric(
    state_forecast_panel,
    ["stocks_to_use_change"]
)

state_forecast_panel["ending_stocks_tightness_raw"] = -first_available_numeric(
    state_forecast_panel,
    ["model_ending_stocks_mil_bu"]
)

for raw_col, z_col in [
    ("stocks_to_use_tightness_raw", "stocks_to_use_tightness_z"),
    ("stocks_to_use_revision_tightness_raw", "stocks_to_use_revision_tightness_z"),
    ("ending_stocks_tightness_raw", "ending_stocks_tightness_z"),
]:
    state_forecast_panel[z_col] = lagged_rolling_zscore(
        state_forecast_panel[raw_col],
        window=756,
        min_periods=126,
    )

state_forecast_panel["balance_factor_z"] = row_weighted_mean(
    state_forecast_panel,
    [
        "stocks_to_use_tightness_z",
        "stocks_to_use_revision_tightness_z",
        "ending_stocks_tightness_z",
    ],
    weights={
        "stocks_to_use_tightness_z": 0.50,
        "stocks_to_use_revision_tightness_z": 0.35,
        "ending_stocks_tightness_z": 0.15,
    },
    min_non_null=2,
)

print("Balance sheet factor created.")


### Build the enriched state forecast panel

builds the daily state forecast panel used by the factor layer.


In [ ]:
demand_components = []

total_use_model = first_available_numeric(state_forecast_panel, ["model_total_use_mil_bu"])
total_use_prior = first_available_numeric(state_forecast_panel, ["prior_total_use_mil_bu"])
state_forecast_panel["total_use_revision_raw"] = total_use_model - total_use_prior
if state_forecast_panel["total_use_revision_raw"].notna().any():
    state_forecast_panel["total_use_revision_z"] = lagged_rolling_zscore(
        state_forecast_panel["total_use_revision_raw"],
        window=756,
        min_periods=126,
    )
    demand_components.append("total_use_revision_z")

export_signal = first_available_numeric(
    state_forecast_panel,
    [
        "export_forecast_mil_bu",
        "exports_forecast_mil_bu",
        "accumulated_exports",
        "accumulated_exports_1",
        "accumulated_exports_2",
    ]
)
state_forecast_panel["export_strength_raw"] = export_signal
if state_forecast_panel["export_strength_raw"].notna().any():
    state_forecast_panel["export_strength_z"] = lagged_rolling_zscore(
        state_forecast_panel["export_strength_raw"],
        window=756,
        min_periods=126,
    )
    demand_components.append("export_strength_z")

ethanol_signal = first_available_numeric(
    state_forecast_panel,
    [
        "ethanol_use_mil_bu",
        "ethanol",
        "ethanol_production",
        "weekly_ethanol_output",
    ]
)
state_forecast_panel["ethanol_strength_raw"] = ethanol_signal


### Build the enriched state forecast panel

builds the daily state forecast panel used by the factor layer.


In [ ]:
if state_forecast_panel["ethanol_strength_raw"].notna().any():
    state_forecast_panel["ethanol_strength_z"] = lagged_rolling_zscore(
        state_forecast_panel["ethanol_strength_raw"],
        window=756,
        min_periods=126,
    )
    demand_components.append("ethanol_strength_z")

feed_signal = first_available_numeric(
    state_forecast_panel,
    [
        "feed_residual_mil_bu",
        "feed_and_residual",
        "feed_dom_consumption",
    ]
)
state_forecast_panel["feed_strength_raw"] = feed_signal
if state_forecast_panel["feed_strength_raw"].notna().any():
    state_forecast_panel["feed_strength_z"] = lagged_rolling_zscore(
        state_forecast_panel["feed_strength_raw"],
        window=756,
        min_periods=126,
    )
    demand_components.append("feed_strength_z")

demand_weights = {
    "export_strength_z": 0.35,
    "ethanol_strength_z": 0.30,
    "feed_strength_z": 0.20,
    "total_use_revision_z": 0.15,
}


### Build the enriched state forecast panel

builds the daily state forecast panel used by the factor layer.


In [ ]:
state_forecast_panel["demand_factor_z"] = row_weighted_mean(
    state_forecast_panel,
    demand_components,
    weights=demand_weights,
    min_non_null=1,
)

state_forecast_panel["demand_factor_available_count"] = (
    state_forecast_panel[demand_components].notna().sum(axis=1)
    if demand_components else 0
)

state_forecast_panel["demand_factor_quality"] = np.select(
    [
        state_forecast_panel["demand_factor_available_count"].ge(3),
        state_forecast_panel["demand_factor_available_count"].eq(2),
        state_forecast_panel["demand_factor_available_count"].eq(1),
    ],
    [
        "good_component_coverage",
        "medium_component_coverage",
        "provisional_single_component",
    ],
    default="missing",
)

print("Demand factor created.")
print("Demand components:", demand_components)


### Build the enriched state forecast panel

builds the daily state forecast panel used by the factor layer.


In [ ]:
managed_money_level = first_available_numeric(
    state_forecast_panel,
    ["managed_money_net_z_1y_clean", "managed_money_z_clean"]
)

state_forecast_panel["managed_money_level_factor_z"] = managed_money_level.clip(-3, 3)

managed_money_change = first_available_numeric(
    state_forecast_panel,
    ["managed_money_net_change_clean", "managed_money_change_clean"]
)

state_forecast_panel["managed_money_change_factor_z"] = lagged_rolling_zscore(
    managed_money_change,
    window=756,
    min_periods=126,
)

state_forecast_panel["positioning_factor_z"] = row_weighted_mean(
    state_forecast_panel,
    [
        "managed_money_level_factor_z",
        "managed_money_change_factor_z",
    ],
    weights={
        "managed_money_level_factor_z": 0.70,
        "managed_money_change_factor_z": 0.30,
    },
    min_non_null=1,
)

state_forecast_panel["positioning_crowding_risk_z"] = (
    state_forecast_panel["managed_money_level_factor_z"]
    .abs()
    .clip(0, 3)
)

print("Positioning factor created.")


### Build the enriched state forecast panel

builds the daily state forecast panel used by the factor layer.


In [ ]:
myday = first_available_numeric(state_forecast_panel, ["marketing_year_day"])

state_forecast_panel["myday_sin"] = np.sin(2.0 * np.pi * myday / 365.25)
state_forecast_panel["myday_cos"] = np.cos(2.0 * np.pi * myday / 365.25)

state_forecast_panel["acreage_window_factor"] = myday.between(175, 225).astype(float)
state_forecast_panel["planting_window_factor"] = myday.between(210, 275).astype(float)
state_forecast_panel["growing_window_factor"] = myday.between(275, 365).astype(float)
state_forecast_panel["harvest_window_factor"] = myday.between(0, 75).astype(float)
state_forecast_panel["postharvest_demand_window_factor"] = myday.between(75, 175).astype(float)

state_forecast_panel["seasonality_raw"] = row_weighted_mean(
    state_forecast_panel,
    [
        "acreage_window_factor",
        "planting_window_factor",
        "growing_window_factor",
        "harvest_window_factor",
        "postharvest_demand_window_factor",
    ],
    weights={
        "acreage_window_factor": 0.15,
        "planting_window_factor": 0.20,
        "growing_window_factor": 0.40,
        "harvest_window_factor": 0.15,
        "postharvest_demand_window_factor": 0.10,
    },
    min_non_null=1,
)

state_forecast_panel["seasonality_factor_z"] = lagged_rolling_zscore(
    state_forecast_panel["seasonality_raw"],
    window=756,
    min_periods=126,
)

print("Seasonality factor created.")


### Build the enriched state forecast panel

builds the daily state forecast panel used by the factor layer.


In [ ]:
state_forecast_panel["price_momentum_20d_raw"] = (
    front_settle_series
    .pct_change(20)
    .shift(1)
)

state_forecast_panel["momentum_factor_z"] = lagged_rolling_zscore(
    state_forecast_panel["price_momentum_20d_raw"],
    window=756,
    min_periods=126,
)

state_forecast_panel["volatility_factor_z"] = lagged_rolling_zscore(
    state_forecast_panel["front_realized_vol_20d_calc"],
    window=756,
    min_periods=126,
)

state_forecast_panel["return_shock_z"] = lagged_rolling_zscore(
    state_forecast_panel["front_return_1d_calc"],
    window=756,
    min_periods=126,
)

state_forecast_panel["risk_factor_z"] = row_weighted_mean(
    state_forecast_panel,
    [
        "momentum_factor_z",
        "volatility_factor_z",
        "return_shock_z",
    ],
    weights={
        "momentum_factor_z": 0.45,
        "volatility_factor_z": 0.40,
        "return_shock_z": 0.15,
    },
    min_non_null=2,
)

print("Risk factor created.")


### Build the enriched state forecast panel

builds the daily state forecast panel used by the factor layer.


In [ ]:
CORE_PHASE3_FACTORS = [
    "supply_factor_z",
    "balance_factor_z",
    "demand_factor_z",
    "positioning_factor_z",
    "seasonality_factor_z",
    "risk_factor_z",
]

state_forecast_panel["fundamental_tightness_anchor_z"] = row_weighted_mean(
    state_forecast_panel,
    CORE_PHASE3_FACTORS,
    weights={
        "supply_factor_z": 0.30,
        "balance_factor_z": 0.35,
        "demand_factor_z": 0.10,
        "positioning_factor_z": 0.10,
        "seasonality_factor_z": 0.05,
        "risk_factor_z": 0.10,
    },
    min_non_null=4,
)

state_forecast_panel["fundamental_tightness_factor_z"] = state_forecast_panel[
    "fundamental_tightness_anchor_z"
]

state_forecast_panel["state_forecast_factor_available_count"] = (
    state_forecast_panel[CORE_PHASE3_FACTORS]
    .notna()
    .sum(axis=1)
)

state_forecast_panel["state_forecast_factor_model_ready"] = (
    state_forecast_panel["state_forecast_factor_available_count"].ge(4)
    & state_forecast_panel["fundamental_tightness_anchor_z"].notna()
).astype(int)

state_forecast_panel["state_forecast_factor_model_version"] = PHASE3_FACTOR_MODEL_VERSION


### Build the enriched state forecast panel

builds the daily state forecast panel used by the factor layer.


In [ ]:
factor_summary_cols = [
    "asof_date",
    "supply_factor_z",
    "balance_factor_z",
    "demand_factor_z",
    "positioning_factor_z",
    "seasonality_factor_z",
    "risk_factor_z",
    "fundamental_tightness_anchor_z",
    "state_forecast_factor_available_count",
    "state_forecast_factor_model_ready",
    "demand_factor_quality",
]

display(state_forecast_panel[[col for col in factor_summary_cols if col in state_forecast_panel.columns]].tail(10))

print("state forecast factor ready rows:", int(state_forecast_panel["state_forecast_factor_model_ready"].sum()))


### Build the enriched state forecast panel

builds the daily state forecast panel used by the factor layer.


In [ ]:
plot_time_series(
    state_forecast_panel,
    date_col="asof_date",
    value_cols=[
        "supply_factor_z",
        "balance_factor_z",
        "demand_factor_z",
        "positioning_factor_z",
        "seasonality_factor_z",
        "risk_factor_z",
        "fundamental_tightness_anchor_z",
    ],
    title="state forecast factor panel standardized factor values",
    ylabel="Lagged rolling z score or factor value",
    tail=1000,
)


### Build the enriched state forecast panel

builds the daily state forecast panel used by the factor layer.


In [ ]:
latest_factor_row = (
    state_forecast_panel
    .dropna(subset=["fundamental_tightness_anchor_z"], how="all")
    .tail(1)
)

latest_factors = latest_factor_row[
    [
        "supply_factor_z",
        "balance_factor_z",
        "demand_factor_z",
        "positioning_factor_z",
        "seasonality_factor_z",
        "risk_factor_z",
        "fundamental_tightness_anchor_z",
    ]
].T

latest_factors.columns = ["latest_value"]

ax = latest_factors["latest_value"].plot(kind="bar", figsize=(12, 5))
ax.axhline(0, linewidth=1)
ax.set_title("Latest state forecast factor snapshot")
ax.set_ylabel("Factor value")
ax.grid(axis="y", alpha=0.25)
plt.xticks(rotation=45, ha="right")
plt.tight_layout()
plt.show()

display(latest_factors)


### Build the enriched state forecast panel

builds the daily state forecast panel used by the factor layer.


In [ ]:
factor_corr_cols = [
    "supply_factor_z",
    "balance_factor_z",
    "demand_factor_z",
    "positioning_factor_z",
    "seasonality_factor_z",
    "risk_factor_z",
    "fundamental_tightness_anchor_z",
]

corr_data = state_forecast_panel[factor_corr_cols].dropna(how="all")
corr = corr_data.corr()

fig, ax = plt.subplots(figsize=(9, 7))
image = ax.imshow(corr.values, aspect="auto")
ax.set_xticks(range(len(corr.columns)))
ax.set_yticks(range(len(corr.index)))
ax.set_xticklabels(corr.columns, rotation=45, ha="right")
ax.set_yticklabels(corr.index)
ax.set_title("state forecast factor correlation matrix")

for row_index in range(len(corr.index)):
    for column_index in range(len(corr.columns)):
        ax.text(
            column_index,
            row_index,
            f"{corr.iloc[row_index, column_index]:.2f}",
            ha="center",
            va="center",
            fontsize=8,
        )

fig.colorbar(image, ax=ax, fraction=0.046, pad=0.04)
plt.tight_layout()
plt.show()

display(corr)


### Reconcile balance sheet

reconciles production, demand, ending stocks, and stocks to use.


In [ ]:
qc_model_input_cols = [
    "asof_date",
    "front_settle",
    "front_contract",
    "marketing_year",
    "marketing_year_day",
    "acreage_crop_year",
    "acreage_forecast_mil",
    "acreage_prior_mil",
    "acreage_surprise_mil",
    "harvest_ratio_forecast",
    "harvested_acres_forecast_mil",
    "harvested_acres_surprise_mil",
    "yield_forecast_bu_acre",
    "yield_surprise_bu_acre",
    "production_forecast_mil_bu",
    "production_forecast_bil_bu",
    "prior_production_mil_bu",
    "production_delta_mil_bu",
    "prior_ending_stocks_mil_bu",
    "model_ending_stocks_mil_bu",
    "prior_total_use_mil_bu",
    "model_total_use_mil_bu",
    "prior_stocks_to_use",
    "model_stocks_to_use",
    "stocks_to_use_change",
    "seasonal_regime",
    "managed_money_net_pct_oi",
    "managed_money_net_z_1y_clean",
    "managed_money_net_change_clean",
    "supply_factor_z",
    "production_tightness_z",
    "acreage_tightness_z",
    "harvested_acres_tightness_z",
    "yield_tightness_z",
    "balance_factor_z",
    "stocks_to_use_tightness_z",
    "stocks_to_use_revision_tightness_z",
    "ending_stocks_tightness_z",
    "demand_factor_z",
    "demand_factor_available_count",
    "demand_factor_quality",
    "total_use_revision_z",
    "export_strength_z",
    "ethanol_strength_z",
    "feed_strength_z",
    "positioning_factor_z",
    "managed_money_level_factor_z",
    "managed_money_change_factor_z",
    "positioning_crowding_risk_z",
    "seasonality_factor_z",
    "acreage_window_factor",
    "planting_window_factor",
    "growing_window_factor",
    "harvest_window_factor",
    "postharvest_demand_window_factor",
    "myday_sin",
    "myday_cos",
    "risk_factor_z",
    "momentum_factor_z",
    "volatility_factor_z",
    "return_shock_z",
    "fundamental_tightness_anchor_z",
    "fundamental_tightness_factor_z",
    "state_forecast_factor_available_count",
    "state_forecast_factor_model_ready",
    "state_forecast_factor_model_version",
]


### Build the enriched state forecast panel

builds the daily state forecast panel used by the factor layer.


In [ ]:
qc_model_input_cols = [
    col for col in qc_model_input_cols
    if col in state_forecast_panel.columns
]

print("QuantConnect export columns:", len(qc_model_input_cols))


### Build the enriched state forecast panel

builds the daily state forecast panel used by the factor layer.


In [ ]:
qc_state_model_input = state_forecast_panel[qc_model_input_cols].copy()

qc_state_model_input = qc_state_model_input.rename(
    columns={"asof_date": "date"}
)

required_qc_cols = [
    col for col in ["date", "production_forecast_mil_bu", "production_delta_mil_bu"]
    if col in qc_state_model_input.columns
]

qc_state_model_input = (
    qc_state_model_input
    .dropna(subset=required_qc_cols)
    .sort_values("date")
    .drop_duplicates(subset=["date"], keep="last")
    .reset_index(drop=True)
)

display(qc_state_model_input.head())
display(qc_state_model_input.tail())

if "state_forecast_factor_model_ready" in qc_state_model_input.columns:
    ready_rows = int(qc_state_model_input["state_forecast_factor_model_ready"].sum())
    print("state forecast factor ready rows:", ready_rows)

if "fundamental_tightness_anchor_z" in qc_state_model_input.columns:
    anchor = qc_state_model_input["fundamental_tightness_anchor_z"].dropna()
    latest_anchor = anchor.iloc[-1] if len(anchor) else np.nan
    print("Latest fundamental tightness anchor:", latest_anchor)

print(f"Rows: {qc_state_model_input.shape[0]:,}")

if len(qc_state_model_input):
    start_date = qc_state_model_input["date"].min().date()
    end_date = qc_state_model_input["date"].max().date()
    print(f"Date range: {start_date} to {end_date}")


### Build the enriched state forecast panel

builds the daily state forecast panel used by the factor layer.


In [ ]:
state_forecast_output_dir = state_forecast_output_directory
state_forecast_output_dir.mkdir(parents=True, exist_ok=True)

full_panel_path = state_forecast_output_dir / "corn_full_panel_enriched.csv"
qc_input_path = state_forecast_output_dir / "corn_model_input_for_quantconnect.csv"

state_forecast_panel.to_csv(full_panel_path, index=False)
qc_state_model_input.to_csv(qc_input_path, index=False)

print("Saved enriched state forecast panel:")
print(full_panel_path)

print()
print("Saved QuantConnect model input file:")
print(qc_input_path)


## QuantConnect handoff

Prepare the final state file used by the fitted futures curve pricing engine in QuantConnect.


### Build QuantConnect model input

creates the daily model input table for QuantConnect.


In [ ]:
from pathlib import Path
import pandas as pd
import numpy as np

strategy_input_state_file_name = "corn_strategy_input_state_for_quantconnect.csv"

strategy_input = qc_state_model_input.copy()
strategy_input["date"] = pd.to_datetime(strategy_input["date"], errors="coerce")

strategy_input = (
    strategy_input
    .sort_values("date")
    .drop_duplicates("date")
    .reset_index(drop=True)
)

print("Prepared strategy input state input")
print("Rows:", len(strategy_input))
print("Date range:", strategy_input["date"].min(), "to", strategy_input["date"].max())
print("Columns:", len(strategy_input.columns))

display(strategy_input.head())
display(strategy_input.tail())


### Add strategy input price risk features

adds lagged price and realized volatility features.


In [ ]:
strategy_input["front_settle"] = pd.to_numeric(strategy_input["front_settle"], errors="coerce")

strategy_input["front_return_1d_calc"] = strategy_input["front_settle"].pct_change().shift(1)
strategy_input["front_return_5d_calc"] = strategy_input["front_settle"].pct_change(5).shift(1)

strategy_input["front_realized_vol_20d_calc"] = (
    strategy_input["front_return_1d_calc"]
    .rolling(20, min_periods=15)
    .std()
    * np.sqrt(252)
)

strategy_input["front_realized_vol_60d_calc"] = (
    strategy_input["front_return_1d_calc"]
    .rolling(60, min_periods=40)
    .std()
    * np.sqrt(252)
)

print("strategy input lagged price features created.")


### Reconcile balance sheet

reconciles production, demand, ending stocks, and stocks to use.


In [ ]:
checks = {
    "q_front_settle": strategy_input["front_settle"].between(200, 1200),
    "q_acreage": strategy_input["acreage_forecast_mil"].between(75, 105),
    "q_harvest_ratio": strategy_input["harvest_ratio_forecast"].between(0.84, 0.96),
    "q_yield": strategy_input["yield_forecast_bu_acre"].between(130, 220),
    "q_production_forecast": strategy_input["production_forecast_mil_bu"].between(10000, 18000),
    "q_prior_prod": strategy_input["prior_production_mil_bu"].between(10000, 18000),
    "q_prod_delta": strategy_input["production_delta_mil_bu"].between(-2500, 2500),
    "q_prior_ending_stocks": strategy_input["prior_ending_stocks_mil_bu"].between(300, 4000),
    "q_model_ending_stocks": strategy_input["model_ending_stocks_mil_bu"].between(0, 5000),
    "q_total_use": strategy_input["prior_total_use_mil_bu"].between(10000, 18000),
    "q_prior_stu": strategy_input["prior_stocks_to_use"].between(0.02, 0.35),
    "q_model_stu": strategy_input["model_stocks_to_use"].between(0.02, 0.35),
    "q_stu_change": strategy_input["stocks_to_use_change"].between(-0.15, 0.15),
}

for name, mask in checks.items():
    strategy_input[name] = mask.fillna(False).astype(int)

strategy_input["strategy_input_state_quality_score"] = strategy_input[list(checks.keys())].sum(axis=1)

strategy_input["strategy_input_state_quality_ok"] = (
    strategy_input["strategy_input_state_quality_score"] == len(checks)
).astype(int)

print("strategy input state quality rows:", int(strategy_input["strategy_input_state_quality_ok"].sum()))


### Apply strategy input quality gates

creates physical state quality gates before export.


In [ ]:
strategy_input["prior_stocks_to_use_clean"] = strategy_input["prior_stocks_to_use"].where(
    strategy_input["strategy_input_state_quality_ok"].eq(1)
)

strategy_input["model_stocks_to_use_clean"] = strategy_input["model_stocks_to_use"].where(
    strategy_input["strategy_input_state_quality_ok"].eq(1)
)

strategy_input["stocks_to_use_change_clean"] = strategy_input["stocks_to_use_change"].where(
    strategy_input["strategy_input_state_quality_ok"].eq(1)
)

strategy_input["production_delta_bil_bu_clean"] = (
    strategy_input["production_delta_mil_bu"] / 1000.0
).where(
    strategy_input["strategy_input_state_quality_ok"].eq(1)
)

strategy_input["managed_money_z_clean"] = (
    pd.to_numeric(strategy_input.get("managed_money_net_z_1y_clean", np.nan), errors="coerce")
    .clip(-3, 3)
)

print("Clean physical state variables created.")


### Construct supply factor

turns supply surprises into a standardized factor.


In [ ]:
if "fundamental_tightness_anchor_z" not in strategy_input.columns and "fundamental_tightness_factor_z" in strategy_input.columns:
    strategy_input["fundamental_tightness_anchor_z"] = strategy_input["fundamental_tightness_factor_z"]

if "fundamental_tightness_factor_z" not in strategy_input.columns and "fundamental_tightness_anchor_z" in strategy_input.columns:
    strategy_input["fundamental_tightness_factor_z"] = strategy_input["fundamental_tightness_anchor_z"]

PHASE3_CORE_FACTOR_COLS = [
    "supply_factor_z",
    "balance_factor_z",
    "demand_factor_z",
    "positioning_factor_z",
    "seasonality_factor_z",
    "risk_factor_z",
    "fundamental_tightness_anchor_z",
    "fundamental_tightness_factor_z",
]

PHASE3_SUBFACTOR_COLS = [
    "production_tightness_z",
    "acreage_tightness_z",
    "harvested_acres_tightness_z",
    "yield_tightness_z",
    "stocks_to_use_tightness_z",
    "stocks_to_use_revision_tightness_z",
    "ending_stocks_tightness_z",
    "total_use_revision_z",
    "export_strength_z",
    "ethanol_strength_z",
    "feed_strength_z",
    "managed_money_level_factor_z",
    "managed_money_change_factor_z",
    "positioning_crowding_risk_z",
    "acreage_window_factor",
    "planting_window_factor",
    "growing_window_factor",
    "harvest_window_factor",
    "postharvest_demand_window_factor",
    "momentum_factor_z",
    "volatility_factor_z",
    "return_shock_z",
]


### Define state forecast factor helpers

defines the reusable factor construction helpers.


In [ ]:
for col in PHASE3_CORE_FACTOR_COLS + PHASE3_SUBFACTOR_COLS:
    if col not in strategy_input.columns:
        strategy_input[col] = np.nan

    strategy_input[f"{col}_clean"] = (
        pd.to_numeric(strategy_input[col], errors="coerce")
        .clip(-3, 3)
        .where(strategy_input["strategy_input_state_quality_ok"].eq(1))
    )

if "state_forecast_factor_model_ready" not in strategy_input.columns:
    strategy_input["state_forecast_factor_model_ready"] = 0

strategy_input["strategy_input_factor_ready"] = (
    strategy_input["strategy_input_state_quality_ok"].eq(1)
    & strategy_input["state_forecast_factor_model_ready"].fillna(0).astype(int).eq(1)
    & strategy_input["fundamental_tightness_anchor_z_clean"].notna()
).astype(int)

if "state_forecast_factor_model_version" not in strategy_input.columns:
    strategy_input["state_forecast_factor_model_version"] = "missing"

print("Clean state forecast factor fields created.")
print("strategy input factor ready rows:", int(strategy_input["strategy_input_factor_ready"].sum()))


### Clean physical state variables

prepares clean physical variables for the trading model.


In [ ]:
strategy_input["myday_sin"] = np.sin(
    2 * np.pi * strategy_input["marketing_year_day"] / 365.25
)

strategy_input["myday_cos"] = np.cos(
    2 * np.pi * strategy_input["marketing_year_day"] / 365.25
)

strategy_input["scarcity_score"] = -strategy_input["model_stocks_to_use_clean"]
strategy_input["looseness_score"] = strategy_input["stocks_to_use_change_clean"]

strategy_input["large_state_revision_flag"] = (
    strategy_input["stocks_to_use_change_clean"].diff().abs().gt(0.01)
    | strategy_input["model_stocks_to_use_clean"].diff().abs().gt(0.01)
    | strategy_input["production_delta_bil_bu_clean"].diff().abs().gt(0.15)
).fillna(False).astype(int)

print("Seasonal and interpretation fields created.")


### Apply strategy input quality gates

creates physical state quality gates before export.


In [ ]:
strategy_input["strategy_input_compute_location"] = "quantconnect"
strategy_input["strategy_input_market_data_required"] = "qc_futures_chain"

strategy_input["strategy_input_outright_ready"] = strategy_input["strategy_input_factor_ready"]
strategy_input["strategy_input_calendar_spread_ready"] = strategy_input["strategy_input_factor_ready"]

strategy_input["strategy_input_basis_ready"] = 0
strategy_input["strategy_input_basis_reason"] = "cash_basis_panel_missing"

strategy_input["strategy_input_model_version"] = "strategy_input_qc_side_fitted_factor_curve_pricing_state_v3"

quality_summary = pd.DataFrame(
    {
        "check": list(checks.keys()) + [
            "strategy_input_state_quality_ok",
            "strategy_input_factor_ready",
            "strategy_input_outright_ready",
            "strategy_input_calendar_spread_ready",
            "strategy_input_basis_ready",
        ],
        "pass_rows": [int(strategy_input[c].sum()) for c in checks.keys()]
        + [
            int(strategy_input["strategy_input_state_quality_ok"].sum()),
            int(strategy_input["strategy_input_factor_ready"].sum()),
            int(strategy_input["strategy_input_outright_ready"].sum()),
            int(strategy_input["strategy_input_calendar_spread_ready"].sum()),
            int(strategy_input["strategy_input_basis_ready"].sum()),
        ],
        "pass_pct": [
            round(strategy_input[c].mean() * 100, 2) for c in checks.keys()
        ]
        + [
            round(strategy_input["strategy_input_state_quality_ok"].mean() * 100, 2),
            round(strategy_input["strategy_input_factor_ready"].mean() * 100, 2),
            round(strategy_input["strategy_input_outright_ready"].mean() * 100, 2),
            round(strategy_input["strategy_input_calendar_spread_ready"].mean() * 100, 2),
            round(strategy_input["strategy_input_basis_ready"].mean() * 100, 2),
        ],
    }
)

display(quality_summary)


### Construct supply factor

turns supply surprises into a standardized factor.


In [ ]:
plot_time_series(
    strategy_input,
    date_col="date",
    value_cols=[
        "front_settle",
        "front_realized_vol_20d_calc",
        "front_realized_vol_60d_calc",
        "model_stocks_to_use_clean",
        "stocks_to_use_change_clean",
        "production_delta_bil_bu_clean",
        "fundamental_tightness_anchor_z_clean",
        "supply_factor_z_clean",
        "balance_factor_z_clean",
    ],
    title="strategy input ready price risk physical state and factors",
    ylabel="Value or standardized factor",
    tail=750,
)

plot_time_series(
    strategy_input,
    date_col="date",
    value_cols=[
        "strategy_input_state_quality_score",
        "strategy_input_state_quality_ok",
        "state_forecast_factor_model_ready",
        "strategy_input_factor_ready",
        "strategy_input_outright_ready",
        "strategy_input_calendar_spread_ready",
    ],
    title="strategy input quality factor readiness and routing flags",
    ylabel="Score or flag",
    tail=750,
)


### Add strategy input price risk features

adds lagged price and realized volatility features.


In [ ]:
strategy_input_core_cols = [
    "date",
    "front_settle",
    "front_contract",
    "marketing_year",
    "marketing_year_day",
    "acreage_crop_year",
    "seasonal_regime",
    "myday_sin",
    "myday_cos",
    "front_return_1d_calc",
    "front_return_5d_calc",
    "front_realized_vol_20d_calc",
    "front_realized_vol_60d_calc",
]


### Reconcile balance sheet

reconciles production, demand, ending stocks, and stocks to use.


In [ ]:
strategy_input_physical_cols = [
    "acreage_forecast_mil",
    "acreage_prior_mil",
    "acreage_surprise_mil",
    "harvest_ratio_forecast",
    "harvested_acres_forecast_mil",
    "harvested_acres_surprise_mil",
    "yield_forecast_bu_acre",
    "yield_surprise_bu_acre",
    "production_forecast_mil_bu",
    "production_forecast_bil_bu",
    "prior_production_mil_bu",
    "production_delta_mil_bu",
    "prior_ending_stocks_mil_bu",
    "model_ending_stocks_mil_bu",
    "prior_total_use_mil_bu",
    "model_total_use_mil_bu",
    "prior_stocks_to_use",
    "model_stocks_to_use",
    "stocks_to_use_change",
    "prior_stocks_to_use_clean",
    "model_stocks_to_use_clean",
    "stocks_to_use_change_clean",
    "production_delta_bil_bu_clean",
]


### Define state forecast factor helpers

defines the reusable factor construction helpers.


In [ ]:
strategy_input_factor_cols = [
    "managed_money_net_pct_oi",
    "managed_money_net_z_1y_clean",
    "managed_money_z_clean",
    "managed_money_net_change_clean",
    "supply_factor_z",
    "production_tightness_z",
    "acreage_tightness_z",
    "harvested_acres_tightness_z",
    "yield_tightness_z",
    "balance_factor_z",
    "stocks_to_use_tightness_z",
    "stocks_to_use_revision_tightness_z",
    "ending_stocks_tightness_z",
    "demand_factor_z",
    "demand_factor_available_count",
    "demand_factor_quality",
    "total_use_revision_z",
    "export_strength_z",
    "ethanol_strength_z",
    "feed_strength_z",
    "positioning_factor_z",
    "managed_money_level_factor_z",
    "managed_money_change_factor_z",
    "positioning_crowding_risk_z",
    "seasonality_factor_z",
    "acreage_window_factor",
    "planting_window_factor",
    "growing_window_factor",
    "harvest_window_factor",
    "postharvest_demand_window_factor",
    "risk_factor_z",
    "momentum_factor_z",
    "volatility_factor_z",
    "return_shock_z",
    "fundamental_tightness_anchor_z",
    "fundamental_tightness_factor_z",
    "state_forecast_factor_available_count",
    "state_forecast_factor_model_ready",
    "state_forecast_factor_model_version",
]


### Construct supply factor

turns supply surprises into a standardized factor.


In [ ]:
strategy_input_clean_and_quality_cols = [
    "supply_factor_z_clean",
    "balance_factor_z_clean",
    "demand_factor_z_clean",
    "positioning_factor_z_clean",
    "seasonality_factor_z_clean",
    "risk_factor_z_clean",
    "fundamental_tightness_anchor_z_clean",
    "fundamental_tightness_factor_z_clean",
    "positioning_crowding_risk_z_clean",
    "momentum_factor_z_clean",
    "volatility_factor_z_clean",
    "return_shock_z_clean",
    "scarcity_score",
    "looseness_score",
    "large_state_revision_flag",
    "strategy_input_state_quality_score",
    "strategy_input_state_quality_ok",
    "strategy_input_factor_ready",
    "strategy_input_outright_ready",
    "strategy_input_calendar_spread_ready",
    "strategy_input_basis_ready",
    "strategy_input_basis_reason",
    "strategy_input_compute_location",
    "strategy_input_market_data_required",
    "strategy_input_model_version",
]


### Clean physical state variables

prepares clean physical variables for the trading model.


In [ ]:
strategy_input_state_cols = (
    strategy_input_core_cols
    + strategy_input_physical_cols
    + strategy_input_factor_cols
    + strategy_input_clean_and_quality_cols
)

strategy_input_state_cols = [
    col for col in strategy_input_state_cols
    if col in strategy_input.columns
]

print("strategy input export columns:", len(strategy_input_state_cols))


### Apply strategy input quality gates

creates physical state quality gates before export.


In [ ]:
qc_strategy_input_state = strategy_input[strategy_input_state_cols].copy()

for column in qc_strategy_input_state.select_dtypes(include=[float]).columns:
    qc_strategy_input_state[column] = qc_strategy_input_state[column].round(8)

strategy_input_output_path = Path(state_forecast_output_dir) / strategy_input_state_file_name
qc_strategy_input_state.to_csv(strategy_input_output_path, index=False)

print(f"Exported: {strategy_input_output_path}")
print("Rows:", len(qc_strategy_input_state))
print("Date range:", qc_strategy_input_state["date"].min(), "to", qc_strategy_input_state["date"].max())
print("Duplicate dates:", qc_strategy_input_state["date"].duplicated().sum())
print("strategy input state quality rows:", int(qc_strategy_input_state["strategy_input_state_quality_ok"].sum()))
print("strategy input factor ready rows:", int(qc_strategy_input_state["strategy_input_factor_ready"].sum()))
print("Outright ready rows:", int(qc_strategy_input_state["strategy_input_outright_ready"].sum()))
print("Calendar spread ready rows:", int(qc_strategy_input_state["strategy_input_calendar_spread_ready"].sum()))
print("Basis ready rows:", int(qc_strategy_input_state["strategy_input_basis_ready"].sum()))

display(qc_strategy_input_state.head())
display(qc_strategy_input_state.tail())


### Transform data

Prepare the next modeling table used by the workflow.


In [ ]:
factor_ready_series = (
    qc_strategy_input_state["strategy_input_factor_ready"]
    if "strategy_input_factor_ready" in qc_strategy_input_state.columns
    else pd.Series(0, index=qc_strategy_input_state.index)
)


### Construct supply factor

turns supply surprises into a standardized factor.


In [ ]:
export_checks = pd.DataFrame(
    {
        "check": [
            "rows",
            "date_min",
            "date_max",
            "duplicate_dates",
            "state_quality_ok_rows",
            "state_quality_ok_pct",
            "factor_ready_rows",
            "factor_ready_pct",
            "outright_ready_rows",
            "calendar_spread_ready_rows",
            "basis_ready_rows",
            "model_stocks_to_use_clean_coverage_pct",
            "stocks_to_use_change_clean_coverage_pct",
            "production_delta_bil_bu_clean_coverage_pct",
            "fundamental_tightness_anchor_clean_coverage_pct",
            "supply_factor_clean_coverage_pct",
            "balance_factor_clean_coverage_pct",
        ],
        "value": [
            len(qc_strategy_input_state),
            qc_strategy_input_state["date"].min(),
            qc_strategy_input_state["date"].max(),
            int(qc_strategy_input_state["date"].duplicated().sum()),
            int(qc_strategy_input_state["strategy_input_state_quality_ok"].sum()),
            round(qc_strategy_input_state["strategy_input_state_quality_ok"].mean() * 100, 2),
            int(factor_ready_series.sum()),
            round(factor_ready_series.mean() * 100, 2),
            int(qc_strategy_input_state["strategy_input_outright_ready"].sum()),
            int(qc_strategy_input_state["strategy_input_calendar_spread_ready"].sum()),
            int(qc_strategy_input_state["strategy_input_basis_ready"].sum()),
            round(qc_strategy_input_state["model_stocks_to_use_clean"].notna().mean() * 100, 2),
            round(qc_strategy_input_state["stocks_to_use_change_clean"].notna().mean() * 100, 2),
            round(qc_strategy_input_state["production_delta_bil_bu_clean"].notna().mean() * 100, 2),
            round(qc_strategy_input_state["fundamental_tightness_anchor_z_clean"].notna().mean() * 100, 2)
                if "fundamental_tightness_anchor_z_clean" in qc_strategy_input_state.columns else 0,
            round(qc_strategy_input_state["supply_factor_z_clean"].notna().mean() * 100, 2)
                if "supply_factor_z_clean" in qc_strategy_input_state.columns else 0,
            round(qc_strategy_input_state["balance_factor_z_clean"].notna().mean() * 100, 2)
                if "balance_factor_z_clean" in qc_strategy_input_state.columns else 0,
        ],
    }
)

display(export_checks)


### Construct supply factor

turns supply surprises into a standardized factor.


In [ ]:
tail_cols = [
    col for col in [
        "date",
        "front_settle",
        "model_stocks_to_use_clean",
        "stocks_to_use_change_clean",
        "production_delta_bil_bu_clean",
        "fundamental_tightness_factor_z_clean",
        "supply_factor_z_clean",
        "balance_factor_z_clean",
        "positioning_factor_z_clean",
        "seasonal_regime",
        "strategy_input_state_quality_ok",
        "strategy_input_factor_ready",
        "strategy_input_calendar_spread_ready",
        "strategy_input_basis_ready",
    ]
    if col in qc_strategy_input_state.columns
]


### Review output

Preview the current table before continuing.


In [ ]:
display(qc_strategy_input_state[tail_cols].tail(30))


### Construct supply factor

turns supply surprises into a standardized factor.


In [ ]:
readiness_cols = [
    "strategy_input_state_quality_ok",
    "strategy_input_factor_ready",
    "strategy_input_outright_ready",
    "strategy_input_calendar_spread_ready",
    "strategy_input_basis_ready",
]

readiness_rates = qc_strategy_input_state[readiness_cols].mean().sort_values(ascending=False) * 100

ax = readiness_rates.plot(kind="bar", figsize=(10, 4))
ax.set_title("strategy input exported readiness rate")
ax.set_xlabel("Flag")
ax.set_ylabel("Rows ready percent")
ax.grid(axis="y", alpha=0.25)
plt.xticks(rotation=45, ha="right")
plt.tight_layout()
plt.show()

plot_time_series(
    qc_strategy_input_state,
    date_col="date",
    value_cols=[
        "front_settle",
        "model_stocks_to_use_clean",
        "stocks_to_use_change_clean",
        "production_delta_bil_bu_clean",
        "fundamental_tightness_anchor_z_clean",
        "supply_factor_z_clean",
        "balance_factor_z_clean",
    ],
    title="Exported strategy input physical and factor state variables",
    ylabel="Value or standardized factor",
    tail=750,
)
